# Repeat Purchase Prediction, part 4
# Model benchmarking and optimisation

## Notebook purpose

This notebook compares more model families with the logistic regression and histogram gradient boosting from `03_modelling_homework.ipynb`, tunes them, tests blends and weighted ensembles, and picks the model for the final submission. It covers the modelling and evaluation phases of the project.

## Approach

- **Same validation as notebook 3:** four expanding yearly folds (2020–2023) with a 90-day gap, and the holdout from January 2024. Every model is scored on the same rows.
- **Same inputs as notebook 3:** the notebook 2 configuration, including the time-stable rates and the purchase-process probability fitted inside each training part.
- **Defaults first:** each new model runs with its default settings and is compared with the untuned logistic regression and gradient boosting.
- **Then tuning:** each model is tuned on the development folds and compared with its own default, then with logistic regression.
- **One decision rule:** the notebook 2 rule. Log-loss must be clearly lower and F1 not clearly worse, judged with customer-bootstrap intervals and fold consistency. ROC-AUC and balanced accuracy are reported alongside.
- **Holdout only records:** choices are made on development results; the holdout scores the finalists and changes nothing.

## Structure

1. Setup and data
2. Reference models
3. New models with default settings
4. Tuning
5. Scoreboard, blends, weighted ensembles and model choice
6. Threshold and calibration
7. Holdout assessment
8. Final fit and submission
9. Drift and refit check
10. Summary of decisions


# 1. Setup and data

## 1.1. Paths and experiment settings

In [1]:
# Handle paths without computer-specific locations.
from pathlib import Path

PROJECT_ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "data").is_dir() and (folder / "notebooks").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate the project. Open the notebook within the repository.")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "preprocessed"
SUBMISSION_DIR = DATA_DIR / "submissions"
OPTUNA_DIR = DATA_DIR / "optuna"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

for folder in (SUBMISSION_DIR, OPTUNA_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# decision: a new experiment version, so 04 searches never mix with the homework ones
EXPERIMENT_VERSION = "project_v1"

N_TRIALS = 60         # completed trials per search
FRESH_SEARCH = False  # True deletes and restarts the named searches of this version

NEEDED = [
    PROCESSED_DIR / "train_clean.parquet",
    PROCESSED_DIR / "test_clean.parquet",
    PROCESSED_DIR / "decision_log_modelling.json",
    PROCESSED_DIR / "column_roles.json",
    PROCESSED_DIR / "preprocessing_config.json",
    RAW_DIR / "sample_submission.csv",
]

missing = [path.name for path in NEEDED if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}. Run 02_preprocessing and 03_modelling_homework first.")

print(f"Project root: {PROJECT_ROOT.name}")
print(f"Experiment version: {EXPERIMENT_VERSION} | trials per search: {N_TRIALS}")

Project root: ml-repeat-purchase
Experiment version: project_v1 | trials per search: 60


## 1.2. Imports, display and plot settings

In [2]:
# Standard library
import sys
import json
import hashlib
import warnings
from time import perf_counter

# Data and numerical calculations
import numpy as np
import pandas as pd

# Charts
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Notebook display
from IPython.display import display

# Hyperparameter search
import optuna
from optuna.importance import FanovaImportanceEvaluator
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice

# Pipelines and custom transformers
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, SplineTransformer
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils.validation import check_is_fitted
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, cross_val_predict

# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# Joblib
import joblib
from importlib.metadata import version

# Metrics
from sklearn.metrics import (
    f1_score,
    log_loss,
    roc_auc_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.exceptions import ConvergenceWarning
from scipy.optimize import minimize

# Project modules
project_path = str(PROJECT_ROOT)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import src.repeat_purchase_prep as rpp
from src.decision_log import DecisionLog

RANDOM_STATE = rpp.RANDOM_STATE
target = rpp.TARGET

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
# Plot definitions shared by every chart in this notebook
ACCENT = {"burgundy": "#801333", "orange": "#ff8600", "near_black": "#1d212a"}
SERIES_COLOURS = ["#801333", "#f3a481", "#35193e", "#ff8600", "#e13342"]  # one colour per model or group
GRID_COLOUR = "#eeeeee"
FONT = "Inter, Arial, sans-serif"

# Seaborn's rocket palette as Plotly colour scales
ROCKET = [[i / 10, colour] for i, colour in enumerate(sns.color_palette("rocket", 11).as_hex())]
ROCKET_R = [[i / 10, colour] for i, colour in enumerate(sns.color_palette("rocket_r", 11).as_hex())]
# Correlations and loadings: always plotted with zmin=-1, zmax=1
DIVERGING = [[0.0, ACCENT["orange"]], [0.5, "#ffffff"], [1.0, ACCENT["burgundy"]]]

pio.templates["project"] = go.layout.Template(
    layout=dict(
        font=dict(family=FONT, size=13, color=ACCENT["near_black"]),
        title=dict(x=0, xanchor="left", font=dict(size=17)),
        paper_bgcolor="white",
        plot_bgcolor="white",
        colorway=SERIES_COLOURS,
        colorscale=dict(sequential=ROCKET_R, diverging=DIVERGING),
        xaxis=dict(gridcolor=GRID_COLOUR, zeroline=False, linecolor="#cccccc", ticks="outside"),
        yaxis=dict(gridcolor=GRID_COLOUR, zeroline=False, linecolor="#cccccc", ticks="outside"),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(font_family=FONT),
        margin=dict(l=60, r=30, t=70, b=60),
    )
)
pio.templates.default = "project"

# decision: figures saved as HTML (plotly.js from the CDN) and PNG at 300 dpi equivalent
HTML_OPTIONS = dict(include_plotlyjs="cdn")
PNG_OPTIONS = dict(scale=300 / 72)

## 1.3. Data and folds

The cleaned data, column roles and validation settings come from notebook 2. The folds are the four expanding years used in notebook 3, so every model here is scored on the same rows. Only the key checks are repeated.

In [4]:
train = pd.read_parquet(PROCESSED_DIR / "train_clean.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_clean.parquet")
sample_submission = pd.read_csv(RAW_DIR / "sample_submission.csv")

config = json.loads((PROCESSED_DIR / "preprocessing_config.json").read_text(encoding="utf-8"))
roles = json.loads((PROCESSED_DIR / "column_roles.json").read_text(encoding="utf-8"))

# Validation settings saved by notebook 2
GAP_DAYS = int(config["validation"]["gap_days"])
HOLDOUT_START = pd.Timestamp(config["validation"]["holdout_start"])
DEV_END = HOLDOUT_START - pd.Timedelta(days=GAP_DAYS)
VAL_YEARS = list(config["validation"]["validation_years"])

dev = train.loc[train["purchase_date"] < DEV_END].reset_index(drop=True)
holdout = train.loc[train["purchase_date"] >= HOLDOUT_START].reset_index(drop=True)

# Expanding yearly folds: training outcomes close before the validation year starts
folds = []
for year in VAL_YEARS:
    start = pd.Timestamp(year=year, month=1, day=1)
    stop = min(pd.Timestamp(year=year + 1, month=1, day=1), DEV_END)
    train_idx = np.flatnonzero(dev["purchase_date"] < start - pd.Timedelta(days=GAP_DAYS))
    val_idx = np.flatnonzero((dev["purchase_date"] >= start) & (dev["purchase_date"] < stop))
    folds.append((train_idx, val_idx))

# Key checks only; the full set ran in notebook 3
assert set(train.columns) - {target} == set(test.columns)
assert set(sample_submission["ID"]) == set(test["ID"])
replaced = config.get("review_features", {}).get("replaced", [])
assert set(roles["num"]) - {"nbd_repeat_prob"} <= set(dev.columns) and not set(replaced) & set(roles["num"]), \
    "Saved column roles no longer match the data."

log = DecisionLog.load(PROCESSED_DIR / "decision_log_modelling.json")

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Development: {len(dev):,} | Holdout: {len(holdout):,}")
print("Fold sizes (train / validation):", {y: (len(tr), len(va)) for y, (tr, va) in zip(VAL_YEARS, folds)})
print(f"Decisions loaded: {len(log)}")

Train: (6469, 55) | Test: (986, 54)
Development: 5,403 | Holdout: 835
Fold sizes (train / validation): {2020: (2578, 670), 2021: (3234, 782), 2022: (4004, 628), 2023: (4705, 536)}
Decisions loaded: 44


## 1.4. Pipeline builders

Two input branches, as decided in notebook 2. The linear branch fills, scales and encodes the columns, and can add the elastic-net selection. The tree branch keeps missing values and codes the categories. Both branches add the purchase-process probability, fitted on each training part from the prior purchase count and tenure. Distance and probability models (KNN, Naive Bayes, LDA) use the linear branch; trees use the tree branch.


In [5]:
# Saved branch decisions from notebook 2
NOISE = list(config["excluded_noise"])
TREE_DROP = list(config["hist_gradient_boosting"]["drop_columns"])
NBD_BLOCK = config.get("review_features", {}).get("fitted_block", {})  # fitted inside each training part
NBD_INPUTS = NBD_BLOCK.get("nbd_repeat_prob", [])


class NBDRepeatProbability(BaseEstimator, TransformerMixin):
    """BG/NBD-style probability of another purchase within the horizon, fitted on the training rows (notebook 2, 6.4)."""

    def __init__(self, horizon=90):
        self.horizon = horizon

    @staticmethod
    def _prob(theta, x, T, h):
        r, alpha, a, b = np.exp(theta)  # log scale keeps the parameters positive
        stop = np.where(x >= 1, a / (a + b + np.maximum(x - 1, 0)), 0.0)
        buy = 1 - np.exp((r + x) * (np.log(alpha + T) - np.log(alpha + T + h)))
        return np.clip((1 - stop) * buy, 1e-6, 1 - 1e-6)

    @staticmethod
    def _inputs(X):
        X = np.asarray(X, dtype=float)
        return np.nan_to_num(X[:, 0]), np.nan_to_num(np.maximum(X[:, 1], 0))

    def fit(self, X, y):
        x, T = self._inputs(X)
        y = np.asarray(y, dtype=float)

        def neg_log_lik(theta):
            p = self._prob(theta, x, T, self.horizon)
            return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

        result = minimize(neg_log_lik, x0=np.log([1.0, 50.0, 1.0, 5.0]), method="Nelder-Mead",
                          options={"maxiter": 4000, "xatol": 1e-6, "fatol": 1e-9})
        self.theta_ = result.x
        return self

    def transform(self, X):
        x, T = self._inputs(X)
        return self._prob(self.theta_, x, T, self.horizon).reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array(["nbd_repeat_prob"])


def branch_roles(branch):
    """Numeric, large and small categorical inputs of one branch; the fitted probability is added by the builders."""
    excluded = set(NOISE + (TREE_DROP if branch == "tree" else []) + ["nbd_repeat_prob"])
    return {group: [c for c in roles[group] if c not in excluded] for group in ("num", "high", "low")}


def branch_inputs(branch):
    r = branch_roles(branch)
    columns = r["num"] + r["high"] + r["low"]
    return columns + [c for c in NBD_INPUTS if c not in columns]  # raw inputs of the fitted probability


LOG_EXCLUDE = NOISE + ["customer_since_year"]  # as in notebook 3: the start year is not a skewed amount


class FoldLog1p(BaseEstimator, TransformerMixin):
    """log1p on non-negative columns whose skew exceeds the threshold, chosen on the training rows (notebook 3)."""

    def __init__(self, threshold=1.0, exclude=None):
        self.threshold = threshold
        self.exclude = exclude

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        candidates = [c for c in X.columns if c not in set(self.exclude or []) and X[c].notna().any() and X[c].min() >= 0]
        skewness = X[candidates].skew()
        self.columns_ = skewness[skewness > self.threshold].index.tolist()
        return self

    def transform(self, X):
        check_is_fitted(self, "columns_")
        X = pd.DataFrame(X, columns=self.feature_names_in_).copy()
        X[self.columns_] = np.log1p(X[self.columns_].clip(lower=0))
        return X

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_.copy()


def build_linear_pipeline(model, select=True, log=False, select_C=None, l1_ratio=None, splines=0):
    """Median fill and scaling, frequency and one-hot encoding, optional elastic-net selection.

    log: notebook 3's log transform of skewed columns; select_C, l1_ratio: selector settings (config by default);
    splines: number of knots of a cubic spline basis on every numeric column (0 = none).
    """
    settings, r = config["linear"], branch_roles("linear")
    num_steps = [("log", FoldLog1p(exclude=LOG_EXCLUDE))] if log else []
    num_steps += [("impute", SimpleImputer(strategy="median", keep_empty_features=True)), ("scale", StandardScaler())]
    if splines:
        num_steps.append(("spline", SplineTransformer(n_knots=splines, degree=3, include_bias=False)))
    blocks = [
        ("num", Pipeline(num_steps), r["num"]),
        ("high", Pipeline([("encode", rpp.FrequencyEncoder()), ("scale", StandardScaler())]), r["high"]),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=settings["rare_min_rows"]), r["low"]),
    ]
    if NBD_INPUTS:
        blocks.append(("nbd", Pipeline([("fit", NBDRepeatProbability()), ("scale", StandardScaler())]), NBD_INPUTS))
    steps = [("prep", ColumnTransformer(blocks, sparse_threshold=0))]
    if select:
        steps.append(("select", make_selector(select_C, l1_ratio)))
    steps.append(("model", clone(model)))
    return Pipeline(steps)


def make_selector(select_C=None, l1_ratio=None):
    """Elastic-net selection with the saved mean-importance threshold."""
    settings = config["linear"]
    return SelectFromModel(
        LogisticRegression(solver="saga", C=settings["selector_C"] if select_C is None else select_C,
                           l1_ratio=settings["selector_l1_ratio"] if l1_ratio is None else l1_ratio,
                           max_iter=20000, random_state=RANDOM_STATE),
        threshold=settings["selector_threshold"],
    )


def build_tree_pipeline(model):
    """Small categories as codes (first), large ones as frequencies, numbers untouched; missing values kept."""
    r = branch_roles("tree")
    blocks = [
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), r["low"]),
        ("high", rpp.FrequencyEncoder(), r["high"]),
        ("num", "passthrough", r["num"]),
    ]
    if NBD_INPUTS:
        blocks.append(("nbd", NBDRepeatProbability(), NBD_INPUTS))  # last, so category positions stay first
    prep = ColumnTransformer(blocks, sparse_threshold=0, verbose_feature_names_out=False)
    model = clone(model)
    if isinstance(model, HistGradientBoostingClassifier):
        # decision: native categories and no internal random split, as in notebook 3
        model.set_params(categorical_features=list(range(len(r["low"]))), early_stopping=False)
    return Pipeline([("prep", prep), ("model", model)])


print("Linear inputs:", len(branch_inputs("linear")), "| Tree inputs:", len(branch_inputs("tree")),
      "| fitted probability:", bool(NBD_INPUTS))


Linear inputs: 42 | Tree inputs: 42 | fitted probability: True


## 1.5. Scoring on the folds

Each model is fitted in every fold and scored on the later year. Its threshold is chosen once, on the four years' predictions pooled, with the same rule as notebook 3. The predictions keep the customer and whether that customer appeared in the training part, for the error analysis.

In [6]:
def best_threshold(y, p, tol=0.005):
    """Centre of the longest run of thresholds within tol of the best F1 (grid 0.05 to 0.95)."""
    grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
    actual = np.asarray(y).astype(bool)[:, None]
    predicted = np.asarray(p)[:, None] >= grid
    tp = (predicted & actual).sum(axis=0)
    fp = (predicted & ~actual).sum(axis=0)
    fn = (~predicted & actual).sum(axis=0)
    f1s = 2 * tp / np.maximum(2 * tp + fp + fn, 1)

    near = np.flatnonzero(f1s >= f1s.max() - tol)
    runs = [r for r in np.split(near, np.flatnonzero(np.diff(near) > 1) + 1) if np.any(f1s[r] == f1s.max())]
    chosen = max(runs, key=lambda r: (len(r), -int(r[0])))
    return float(grid[chosen[(len(chosen) - 1) // 2]]), grid, f1s


def f1_floor(y):
    """F1 of predicting a repeat for every row."""
    share = np.mean(y)
    return 2 * share / (1 + share)


def score_folds(pipe, inputs, data=dev):
    """Fit in every fold; return (fold table, out-of-fold predictions, pooled threshold)."""
    rows, oof = [], []
    for (tr, va), year in zip(folds, VAL_YEARS):
        train_part, val_part = data.iloc[tr], data.iloc[va]
        start = perf_counter()
        model = clone(pipe).fit(train_part[inputs], train_part[target])
        seconds = perf_counter() - start

        p = model.predict_proba(val_part[inputs])[:, 1]
        p_train = model.predict_proba(train_part[inputs])[:, 1]
        y = val_part[target].to_numpy()
        steps = model.named_steps
        rows.append({
            "fold": year,
            "n_features": (int(steps["select"].get_support().sum()) if "select" in steps
                           else len(steps["prep"].get_feature_names_out())),
            "log_loss": log_loss(y, p, labels=[0, 1]),
            "train_log_loss": log_loss(train_part[target], p_train, labels=[0, 1]),
            "roc_auc": roc_auc_score(y, p),
            "fit_seconds": seconds,
        })
        # Kept for later error analysis: who the customer is and whether training saw them
        oof.append(pd.DataFrame({
            "fold": year, "ID": val_part["ID"].to_numpy(), "customer_id": val_part["customer_id"].to_numpy(),
            "seen": val_part["customer_id"].isin(train_part["customer_id"]).to_numpy(), "y": y, "p": p,
        }))

    oof = pd.concat(oof, ignore_index=True)
    threshold, _, _ = best_threshold(oof["y"], oof["p"])
    table = pd.DataFrame(rows).set_index("fold")
    table["f1"] = [f1_score(part["y"], part["p"] >= threshold) for _, part in oof.groupby("fold")]
    return table, oof, threshold


def summary_row(table, oof, threshold):
    """One line per model: pooled F1 at its own threshold plus fold averages."""
    predicted = oof["p"] >= threshold
    return {
        "n_features": table["n_features"].mean(),
        "f1": f1_score(oof["y"], predicted),
        "floor": f1_floor(oof["y"]),
        "balanced_accuracy": balanced_accuracy_score(oof["y"], predicted),
        "roc_auc": table["roc_auc"].mean(),
        "log_loss": table["log_loss"].mean(),
        "overfit_gap": (table["log_loss"] - table["train_log_loss"]).mean(),
        "threshold": threshold,
        "fit_seconds": table["fit_seconds"].sum(),
    }

## 1.6. Decision rule

Same rule as notebook 2: an option beats the reference only if the 95% customer-bootstrap interval of its log-loss difference sits below zero, its F1 is not clearly worse (the F1 interval does not sit below zero) and its log-loss is higher in at most one fold. It is worse when its log-loss is clearly higher or its F1 clearly lower. Log-loss uses every row, so it detects smaller real differences than F1 at one threshold; F1 stays the reported metric and sets the threshold.


In [7]:
def fast_f1(y, predicted):
    tp = np.sum(y & predicted)
    return 2 * tp / (2 * tp + np.sum(~y & predicted) + np.sum(y & ~predicted))


def row_log_loss(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


def bootstrap_diffs(oof_ref, oof_opt, thr_ref, thr_opt, n_boot=2000, seed=RANDOM_STATE):
    """95% intervals of the F1 and log-loss differences (option - reference), resampling whole customers."""
    assert oof_ref["ID"].equals(oof_opt["ID"]), "Options must be scored on the same rows."
    y = oof_ref["y"].to_numpy(dtype=bool)
    pred_ref, pred_opt = oof_ref["p"].to_numpy() >= thr_ref, oof_opt["p"].to_numpy() >= thr_opt
    ll_diff = row_log_loss(y, oof_opt["p"].to_numpy()) - row_log_loss(y, oof_ref["p"].to_numpy())
    groups = list(oof_ref.groupby("customer_id", sort=False).indices.values())

    rng = np.random.default_rng(seed)
    f1_d, ll_d = np.empty(n_boot), np.empty(n_boot)
    for b in range(n_boot):
        rows = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        f1_d[b] = fast_f1(y[rows], pred_opt[rows]) - fast_f1(y[rows], pred_ref[rows])
        ll_d[b] = ll_diff[rows].mean()
    return np.percentile(f1_d, [2.5, 97.5]), np.percentile(ll_d, [2.5, 97.5])


def judge(runs, reference):
    """Scoreboard against the reference: log-loss clearly lower, F1 not clearly worse, at most one worse fold."""
    ref_table, ref_oof, ref_thr = runs[reference]
    ref_ll = row_log_loss(ref_oof["y"].to_numpy(), ref_oof["p"].to_numpy()).mean()
    rows = []
    for name, (table, oof, thr) in runs.items():
        row = {"model": name, **summary_row(table, oof, thr)}
        row["ll_diff"] = row_log_loss(oof["y"].to_numpy(), oof["p"].to_numpy()).mean() - ref_ll
        row["folds_worse"] = int((table["log_loss"] > ref_table["log_loss"]).sum())
        if name == reference:
            row.update(ll_ci_low=np.nan, ll_ci_high=np.nan, ci_low=np.nan, ci_high=np.nan, verdict="reference")
        else:
            (lo, hi), (ll_lo, ll_hi) = bootstrap_diffs(ref_oof, oof, ref_thr, thr)
            verdict = ("better" if ll_hi < 0 and hi >= 0 and row["folds_worse"] <= 1
                       else "worse" if ll_lo > 0 or hi < 0 else "no clear difference")
            row.update(ll_ci_low=ll_lo, ll_ci_high=ll_hi, ci_low=lo, ci_high=hi, verdict=verdict)
        rows.append(row)

    board = pd.DataFrame(rows).set_index("model")
    board.insert(2, "f1_diff", board["f1"] - board.loc[reference, "f1"])
    return board


# 2. Reference models

Three models set the bar for everything that follows. The dummy always predicts the training share of repeats, so its F1 is the "always repeat" floor. The untuned logistic regression and gradient boosting are the starting points of notebook 3. Logistic regression is the reference: a new model has to beat it under the decision rule.

## 2.1. Scoreboard

In [8]:
linear_inputs, tree_inputs = branch_inputs("linear"), branch_inputs("tree")

reference_pipes = {
    # decision: the dummy predicts the training share, so its F1 is the always-repeat floor
    "dummy": (build_linear_pipeline(DummyClassifier(strategy="prior"), select=False), linear_inputs),
    "logistic": (build_linear_pipeline(LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)), linear_inputs),
    "hist_gb": (build_tree_pipeline(HistGradientBoostingClassifier(random_state=RANDOM_STATE)), tree_inputs),
}

reference_runs = {name: score_folds(pipe, inputs) for name, (pipe, inputs) in reference_pipes.items()}

board_reference = judge(reference_runs, reference="logistic")
board_reference.loc["dummy", "n_features"] = 0  # it ignores every column
display(board_reference)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
dummy,0.000,0.709,-0.040,0.709,0.500,0.500,0.689,0.002,0.300,0.585,0.133,4,0.074,0.193,-0.058,-0.018,worse
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.923,0.000,0,NaN,NaN,NaN,NaN,reference
hist_gb,41.000,0.744,-0.004,0.709,0.675,0.774,0.589,0.344,0.350,1.530,0.033,4,0.017,0.050,-0.018,0.010,worse


Logistic regression reaches F1 0.749 against the floor of 0.709, with ROC-AUC 0.785 and balanced accuracy 0.683. Untuned gradient boosting is worse under the rule: its log-loss is 0.589 against 0.558 (interval of the difference [0.017, 0.050]) and higher in all four years, while its F1 of 0.744 is not clearly different. The dummy is clearly worse, as expected, with balanced accuracy 0.500 and ROC-AUC 0.500.


## 2.2. F1 by validation year

In [9]:
fold_f1 = pd.DataFrame({name: reference_runs[name][0]["f1"] for name in ["logistic", "hist_gb"]})
fold_f1["always repeat"] = [f1_floor(part["y"]) for _, part in reference_runs["logistic"][1].groupby("fold")]

fig = go.Figure()
for name in ["logistic", "hist_gb", "always repeat"]:
    fig.add_trace(go.Scatter(
        x=fold_f1.index, y=fold_f1[name], mode="lines+markers" if name != "always repeat" else "lines",
        name=name, line=dict(dash="dash", color="grey") if name == "always repeat" else None,
        hovertemplate="%{x}: F1 %{y:.3f}<extra>" + name + "</extra>",
    ))
fig.update_layout(
    title="Reference models: F1 in each validation year",
    xaxis=dict(title="Validation year", tickmode="array", tickvals=VAL_YEARS),
    yaxis=dict(title="F1 at each model's pooled threshold"),
    height=420,
)
fig.write_html(FIGURES_DIR / "04_reference_f1_by_year.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_reference_f1_by_year.png", **PNG_OPTIONS)
fig.show()

In [10]:
display(fold_f1.round(3))

,logistic,hist_gb,always repeat
fold,,,
2020,0.764,0.753,0.745
2021,0.762,0.751,0.709
2022,0.709,0.706,0.649
2023,0.755,0.768,0.728


Logistic regression stays above the floor in every year, from 0.709 in 2022 to 0.764 in 2020. Its smallest lift is in 2020 (0.764 against 0.745). Gradient boosting beats it only in 2023 (0.768 against 0.755) and is lowest in 2022 (0.706), the year with the lowest floor (0.649).


## Findings

| Model | F1 | ROC-AUC | Balanced accuracy | Log-loss | Verdict against logistic |
|---|---|---|---|---|---|
| Dummy (always repeat) | 0.709 | 0.500 | 0.500 | 0.689 | worse |
| Logistic regression | 0.749 | 0.785 | 0.683 | 0.558 | reference |
| Gradient boosting | 0.744 | 0.774 | 0.675 | 0.589 | worse |

- The untuned logistic regression is the reference for the default models in section 3, at F1 0.749.
- The floor is high (0.709), because about 55% of development purchases are followed by a repeat. Balanced accuracy and ROC-AUC show the gap to a model without information more clearly than F1.
- Gradient boosting's log-loss gap between training and validation (0.344, against 0.008 for logistic regression) shows its default settings follow the training years closely.


# 3. New models with default settings

Each model runs with its library defaults on the same folds, so the comparison shows what each model family brings before any tuning. A short check before each model tests the assumption it relies on.

## 3.1. Naive Bayes

Gaussian Naive Bayes assumes that, within each class, every column is normal and independent of the others. The check measures how far the prepared columns are from that.

In [11]:
# Prepared linear-branch columns of the last fold, the same matrix Naive Bayes sees
tr, _ = folds[-1]
nb_prep = build_linear_pipeline(GaussianNB(), select=False).named_steps["prep"]
X_nb = pd.DataFrame(
    nb_prep.fit_transform(dev.iloc[tr][linear_inputs], dev.iloc[tr][target]),
    columns=nb_prep.get_feature_names_out(),
)
y_nb = dev.iloc[tr][target].to_numpy()

binary = [c for c in X_nb if set(np.unique(X_nb[c])) <= {0.0, 1.0}]
numeric = [c for c in X_nb if c not in binary]
skew_by_class = pd.DataFrame({
    "skew_no_repeat": X_nb.loc[y_nb == 0, numeric].skew(),
    "skew_repeat": X_nb.loc[y_nb == 1, numeric].skew(),
})
skew_by_class["max_abs_skew"] = skew_by_class.abs().max(axis=1)

# Strongest correlation of each numeric column with another, inside the repeat class
corr = X_nb.loc[y_nb == 1, numeric].corr().abs().where(lambda m: ~np.eye(len(m), dtype=bool))
skew_by_class["max_abs_corr"] = corr.max()

print(f"{len(X_nb.columns)} prepared columns: {len(binary)} take only 0 and 1, {len(numeric)} are numeric")
print(f"Numeric columns with |skew| > 1 in either class: {(skew_by_class['max_abs_skew'] > 1).sum()}")
print(f"Numeric columns correlated above 0.8 with another: {(skew_by_class['max_abs_corr'] > 0.8).sum()}")
display(skew_by_class.sort_values("max_abs_skew", ascending=False).head(8).round(2))

73 prepared columns: 39 take only 0 and 1, 34 are numeric
Numeric columns with |skew| > 1 in either class: 30
Numeric columns correlated above 0.8 with another: 16


,skew_no_repeat,skew_repeat,max_abs_skew,max_abs_corr
num__spend_per_year,29.370,42.270,42.270,0.470
num__Total Desc_ Global,31.360,24.280,31.360,0.130
num__Total Portes,15.700,24.020,24.020,0.270
num__purchases_per_year,21.400,7.160,21.400,0.680
num__prior_std_gap_days,5.080,9.020,9.020,0.640
num__Total Desc_ Linha,7.840,6.460,7.840,0.630
num__invoice_count_current_purchase,7.610,5.690,7.610,0.170
num__spend_30d,7.100,3.820,7.100,0.710


Both assumptions fail. Of the 73 prepared columns, 39 take only the values 0 and 1, so they cannot be normal. Of the 34 numeric columns, 30 have a skew above 1 in at least one class (up to 42.3 for `spend_per_year`), and 16 correlate above 0.8 with another column, so the same information is counted several times.


In [12]:
nb_runs = {
    "naive_bayes": score_folds(build_linear_pipeline(GaussianNB(), select=False), linear_inputs),
    "naive_bayes_selected": score_folds(build_linear_pipeline(GaussianNB()), linear_inputs),
}
board_nb = judge({"logistic": reference_runs["logistic"], **nb_runs}, reference="logistic")
display(board_nb)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.923,0.000,0,NaN,NaN,NaN,NaN,reference
naive_bayes,69.250,0.661,-0.088,0.709,0.646,0.727,6.273,2.071,0.070,0.486,5.485,4,4.591,6.450,-0.138,-0.056,worse
naive_bayes_selected,20.250,0.687,-0.062,0.709,0.703,0.754,2.443,0.371,0.050,3.948,1.857,4,1.553,2.197,-0.117,-0.031,worse


On all prepared columns Naive Bayes scores F1 0.661, below the always-repeat floor of 0.709. With the elastic-net columns it reaches 0.687, still below the floor. Both are clearly worse than logistic regression. The probabilities are badly overconfident: log-loss is 6.27 and 2.44 against 0.558, and the selected thresholds sit at 0.07 and 0.05, at the bottom of the grid.


## 3.2. Linear discriminant analysis

LDA fits one normal distribution per class with a shared covariance matrix, and it has to invert that matrix. Strongly or exactly collinear columns make the inversion unstable, so the check measures collinearity.

In [13]:
# Collinearity of the same prepared matrix (last fold), all columns together
X_all = X_nb.to_numpy()
X_all = X_all[:, X_all.std(axis=0) > 0]  # constant columns carry no information
singular_values = np.linalg.svd(X_all - X_all.mean(axis=0), compute_uv=False)
condition_number = singular_values[0] / singular_values[-1]

# Pairs of numeric columns that move almost together
pairs = (
    X_nb[numeric].corr().abs()
    .where(np.triu(np.ones((len(numeric), len(numeric)), dtype=bool), k=1))
    .stack().sort_values(ascending=False)
)

print(f"Condition number of the centred matrix: {condition_number:,.0f}")
print(f"Numeric pairs with |correlation| > 0.9: {(pairs > 0.9).sum()}")
display(pairs.head(6).round(3).rename("abs_correlation").to_frame())

Condition number of the centred matrix: 144,775,749,955,350,720
Numeric pairs with |correlation| > 0.9: 5


,,abs_correlation
num__Total a Pagar,num__Total Liquido,0.996
num__Total Bruto,num__Total Liquido,0.982
num__Total a Pagar,num__Total Bruto,0.972
num__purchase_count_180d,num__purchase_count_365d,0.954
num__purchase_count_90d,num__purchase_count_180d,0.936
num__Total a Pagar,num__Total Imp_,0.894


The condition number is about 1.4 × 10¹⁷, which means the matrix is singular: some columns are exact combinations of others. Two causes: the one-hot encoding keeps every level, so each group of indicators sums to 1, and the accounting identity from the EDA makes `Total a Pagar` equal to `Total Liquido` plus `Total Imp_` (correlation 0.996). Five numeric pairs correlate above 0.9, including the 90-, 180- and 365-day purchase counts.


In [14]:
lda_runs = {
    "lda": score_folds(build_linear_pipeline(LinearDiscriminantAnalysis(), select=False), linear_inputs),
    "lda_selected": score_folds(build_linear_pipeline(LinearDiscriminantAnalysis()), linear_inputs),
}
board_lda = judge({"logistic": reference_runs["logistic"], **lda_runs}, reference="logistic")
display(board_lda)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.923,0.000,0,NaN,NaN,NaN,NaN,reference
lda,69.250,0.739,-0.009,0.709,0.650,0.762,0.591,0.036,0.330,0.608,0.034,4,0.024,0.045,-0.020,0.001,worse
lda_selected,20.250,0.745,-0.003,0.709,0.666,0.778,0.571,0.011,0.340,3.930,0.013,4,0.006,0.020,-0.012,0.005,worse


LDA scores F1 0.739 on all columns and 0.745 on the selected ones, above the floor of 0.709. Both are worse than logistic regression on log-loss (0.591 and 0.571 against 0.558, intervals [0.024, 0.045] and [0.006, 0.020]), and higher in all four years. The default `svd` solver runs despite the singular matrix, because it never inverts the covariance directly.


## 3.3. Quadratic discriminant analysis

QDA keeps LDA's normal distributions but gives each class its own covariance matrix, so the boundary between repeaters and non-repeaters can curve. The price is that it must invert one matrix per class, estimated from that class's rows only. A column that never varies inside one class (common for rare one-hot levels) makes that class's matrix impossible to invert. The check counts, for each class, the rank of its covariance against the number of columns.

If the default cannot be fitted, the smallest regularisation the search can try (reg_param = 0.001) stands in for it.


In [15]:
# Rank of each class's covariance on the last fold: all prepared columns and the elastic-net selection
tr, _ = folds[-1]
y_last = dev.iloc[tr][target].to_numpy()
qda_check = {}
for label, pipe in {
    "all columns": build_linear_pipeline(QuadraticDiscriminantAnalysis(), select=False),
    "selected columns": build_linear_pipeline(QuadraticDiscriminantAnalysis()),
}.items():
    X_prep = Pipeline(pipe.steps[:-1]).fit_transform(dev.iloc[tr][linear_inputs], y_last)
    for cls in (0, 1):
        rows = X_prep[y_last == cls]
        qda_check[(label, cls)] = {
            "columns": X_prep.shape[1],
            "rows": len(rows),
            "covariance_rank": np.linalg.matrix_rank(np.cov(rows, rowvar=False)),
            "constant_columns": int((rows.std(axis=0) == 0).sum()),
        }
qda_check = pd.DataFrame(qda_check).T.rename_axis(["inputs", "class"])
display(qda_check)


columns  rows  covariance_rank  constant_columns
inputs           class                                                  
all columns      0           73  2101               64                 0
                 1           73  2604               63                 1
selected columns 0           22  2101               22                 0
                 1           22  2604               22                 0

In [16]:
# The library default (reg_param = 0) needs every class covariance to be invertible
try:
    score_folds(build_linear_pipeline(QuadraticDiscriminantAnalysis(), select=False), linear_inputs)
except np.linalg.LinAlgError as error:
    print(f"Default QDA cannot be fitted: {error}")

# decision: the smallest regularisation the search can try stands in for the default
QDA_MIN_REG = 1e-3  # must exceed the rank tolerance (1e-4) of scikit-learn's check
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)  # "Variables are collinear": the check above shows why
    qda_runs = {
        "qda": score_folds(build_linear_pipeline(QuadraticDiscriminantAnalysis(reg_param=QDA_MIN_REG), select=False),
                           linear_inputs),
        "qda_selected": score_folds(build_linear_pipeline(QuadraticDiscriminantAnalysis(reg_param=QDA_MIN_REG)),
                                    linear_inputs),
    }
board_qda = judge({"logistic": reference_runs["logistic"], **qda_runs}, reference="logistic")
display(board_qda)


Default QDA cannot be fitted: The covariance matrix of class 0 is not full rank. Increase the value of `reg_param` to reduce the collinearity.


,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.923,0.000,0,NaN,NaN,NaN,NaN,reference
qda,69.250,0.686,-0.063,0.709,0.686,0.726,5.319,2.430,0.070,0.546,4.542,4,3.716,5.498,-0.113,-0.032,worse
qda_selected,20.250,0.700,-0.049,0.709,0.703,0.756,2.208,0.582,0.050,3.891,1.614,4,1.343,1.928,-0.096,-0.022,worse


QDA with default settings cannot be fitted on all prepared columns: in the last fold the covariance of each class has rank 63–64 for 73 columns, so it cannot be inverted. The selected columns have full rank in that fold. With the smallest regularisation (0.001) it runs, but it is clearly worse than logistic regression: F1 0.686 on all columns and 0.700 on the selected ones, both at or below the floor of 0.709. Its probabilities are badly overconfident (log-loss 5.32 and 2.21, thresholds 0.07 and 0.05), the same failure as Naive Bayes.



## 3.4. K-nearest neighbours

KNN predicts from the training rows closest to each new row, so it relies on distances being meaningful. With many columns, the nearest and farthest neighbours can end up almost equally far away. The check compares the two distances on all prepared columns and on the elastic-net selection.

In [17]:
# Distance concentration on the last fold: as columns grow, nearest and farthest neighbours look alike
tr, va = folds[-1]
selected_pipe = build_linear_pipeline(KNeighborsClassifier()).fit(dev.iloc[tr][linear_inputs], dev.iloc[tr][target])

matrices = {
    "all columns": (X_nb.to_numpy(), nb_prep.transform(dev.iloc[va][linear_inputs])),
    "selected columns": tuple(
        selected_pipe.named_steps["select"].transform(selected_pipe.named_steps["prep"].transform(dev.iloc[part][linear_inputs]))
        for part in (tr, va)
    ),
}

rows = []
for name, (X_train_fold, X_val_fold) in matrices.items():
    distances = np.linalg.norm(X_val_fold[:, None, :] - X_train_fold[None, :, :], axis=2)
    nearest, farthest = distances.min(axis=1), distances.max(axis=1)
    rows.append({
        "columns": X_train_fold.shape[1],
        "median_nearest": np.median(nearest),
        "median_farthest": np.median(farthest),
        "median_contrast": np.median((farthest - nearest) / nearest),
    })
display(pd.DataFrame(rows, index=list(matrices)).round(2))

,columns,median_nearest,median_farthest,median_contrast
all columns,73,2.010,61.750,29.670
selected columns,22,0.810,16.720,19.570


The contrast between the farthest and nearest neighbour is higher with all 73 columns (29.7) than with the 22 selected ones (19.6), the opposite of the expected concentration. The farthest distances are driven by the extreme values seen in the Naive Bayes check (skew up to 42), so this measure mostly reflects outliers rather than the number of columns. The selection more than halves the distance to the nearest neighbour (median 0.81 against 2.01).


In [18]:
knn_runs = {
    "knn": score_folds(build_linear_pipeline(KNeighborsClassifier(), select=False), linear_inputs),
    "knn_selected": score_folds(build_linear_pipeline(KNeighborsClassifier()), linear_inputs),
}
board_knn = judge({"logistic": reference_runs["logistic"], **knn_runs}, reference="logistic")
display(board_knn)

print("Distinct probabilities from the selected model:", np.sort(knn_runs["knn_selected"][1]["p"].unique()))

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.923,0.000,0,NaN,NaN,NaN,NaN,reference
knn,69.250,0.721,-0.027,0.709,0.627,0.737,2.200,1.775,0.220,0.605,1.552,4,1.240,1.906,-0.044,-0.012,worse
knn_selected,20.250,0.731,-0.018,0.709,0.644,0.737,2.169,1.745,0.300,4.085,1.557,4,1.235,1.916,-0.033,-0.003,worse


Distinct probabilities from the selected model: [0.  0.2 0.4 0.6 0.8 1. ]


With five neighbours, KNN scores F1 0.721 on all columns and 0.731 on the selected ones, both above the floor of 0.709 and clearly worse than logistic regression. It gives only six probabilities (0, 0.2, … 1), so its threshold of 0.30 means "at least two of the five neighbours repeated". Many predictions at exactly 0 or 1 explain the high log-loss (about 2.2), and the training–validation gap of about 1.75 shows each training row votes for itself.


## 3.5. Decision tree

A tree with default settings keeps splitting until every leaf is pure, so it can memorise the training rows. The check measures how large the default tree grows in each fold and how many leaves hold a single row. Missing values stay in: the tree sends them down whichever branch fits the training rows better.

In [19]:
# Size of the default tree in each fold, fitted on the tree branch with missing values kept
tree_rows = []
for (tr, va), year in zip(folds, VAL_YEARS):
    fitted = build_tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)).fit(
        dev.iloc[tr][tree_inputs], dev.iloc[tr][target]
    )
    tree = fitted.named_steps["model"]
    leaf_sizes = tree.tree_.n_node_samples[tree.tree_.children_left == -1]
    tree_rows.append({
        "fold": year,
        "training_rows": len(tr),
        "depth": tree.get_depth(),
        "leaves": tree.get_n_leaves(),
        "median_leaf_size": np.median(leaf_sizes),
        "single_row_leaves": np.mean(leaf_sizes == 1),
    })
display(pd.DataFrame(tree_rows).set_index("fold").round(3))

,training_rows,depth,leaves,median_leaf_size,single_row_leaves
fold,,,,,
2020,2578,26,443,2.000,0.359
2021,3234,22,568,2.000,0.377
2022,4004,30,685,2.000,0.364
2023,4705,29,790,2.000,0.359


The default tree grows with the data: from depth 26 and 443 leaves in the 2020 fold to depth 29 and 790 leaves in 2023. The median leaf holds two training rows, and 36–38% of the leaves hold a single row, so most answers rest on one or two purchases.


In [20]:
tree_runs = {"decision_tree": score_folds(build_tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)), tree_inputs)}
board_tree = judge({"logistic": reference_runs["logistic"], **tree_runs}, reference="logistic")
display(board_tree)

print("Distinct probabilities:", tree_runs["decision_tree"][1]["p"].nunique())

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.923,0.000,0,NaN,NaN,NaN,NaN,reference
decision_tree,41.000,0.631,-0.118,0.709,0.599,0.597,14.478,14.478,0.500,0.572,13.228,4,11.601,14.763,-0.156,-0.086,worse


Distinct probabilities: 2


The default tree scores F1 0.631, well below the floor of 0.709, with ROC-AUC 0.597 and balanced accuracy 0.599. Its pure leaves give only two probabilities, 0 and 1, so the threshold has no effect and log-loss is extreme (14.5). The training–validation gap equals the validation log-loss, because the tree fits its training rows perfectly.


## 3.6. Random forest

A random forest averages many deep trees, each grown on a bootstrap sample of the rows. The rows left out of each sample give an out-of-bag score at no extra cost, but those rows are drawn at random from the same years. The check compares that score with F1 on the later validation year, to see whether random-row scoring would flatter the model here.

In [21]:
# Out-of-bag F1 (random rows of the training years) against F1 on the later validation year
oob_rows = []
for (tr, va), year in zip(folds, VAL_YEARS):
    forest_pipe = build_tree_pipeline(RandomForestClassifier(oob_score=True, random_state=RANDOM_STATE, n_jobs=-1))
    forest_pipe.fit(dev.iloc[tr][tree_inputs], dev.iloc[tr][target])
    forest = forest_pipe.named_steps["model"]

    p_oob = forest.oob_decision_function_[:, 1]
    p_val = forest_pipe.predict_proba(dev.iloc[va][tree_inputs])[:, 1]
    oob_rows.append({
        "fold": year,
        "oob_f1": f1_score(dev.iloc[tr][target], p_oob >= 0.5),
        "validation_f1": f1_score(dev.iloc[va][target], p_val >= 0.5),
        "median_tree_depth": np.median([t.get_depth() for t in forest.estimators_]),
    })
oob_table = pd.DataFrame(oob_rows).set_index("fold")
oob_table["optimism"] = oob_table["oob_f1"] - oob_table["validation_f1"]
display(oob_table.round(3))

,oob_f1,validation_f1,median_tree_depth,optimism
fold,,,,
2020,0.734,0.713,25.000,0.021
2021,0.735,0.743,25.500,-0.009
2022,0.733,0.706,26.000,0.027
2023,0.735,0.728,27.000,0.007


At a 0.5 cut-off, the out-of-bag F1 differs from the later year's F1 by -0.009 to +0.027, mostly in the forest's favour. Random-row scoring therefore tends to flatter the forest slightly here, which supports keeping the chronological folds as the basis for every decision. The trees grow to a median depth of 25–27, as deep as the single default tree.


In [22]:
forest_runs = {
    "random_forest": score_folds(
        build_tree_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)), tree_inputs
    )
}
board_forest = judge({"logistic": reference_runs["logistic"], **forest_runs}, reference="logistic")
display(board_forest)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.923,0.000,0,NaN,NaN,NaN,NaN,reference
random_forest,41.000,0.745,-0.004,0.709,0.661,0.774,0.588,0.435,0.400,1.138,0.028,3,-0.002,0.064,-0.016,0.009,no clear difference


The default forest scores F1 0.745, close to logistic regression and far above the single tree's 0.631: averaging 100 deep trees removes most of their individual noise. It shows no clear difference against logistic regression, but its log-loss is higher (0.588 against 0.558) in three of four years, and its training–validation gap of 0.435 shows the deep trees still follow the training years closely.


## 3.7. Support vector machine (pending)

## 3.8. Multilayer perceptron

A multilayer perceptron passes the inputs through one or more layers of units, each a weighted sum followed by a non-linear activation, so it can learn curved boundaries and interactions without being told which ones. It needs scaled inputs, which the linear branch provides, and its result depends on the random starting weights.

The library default has one hidden layer of 100 units and trains for up to 200 passes over the data. It is scored on all prepared columns and on the elastic-net selection. Training stops after 200 passes even if the fit has not settled; that limit acts like early stopping, so the warning is noted and not treated as an error.


In [23]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)  # 200 passes may stop before the fit settles
    mlp_runs = {
        "mlp": score_folds(build_linear_pipeline(MLPClassifier(random_state=RANDOM_STATE), select=False), linear_inputs),
        "mlp_selected": score_folds(build_linear_pipeline(MLPClassifier(random_state=RANDOM_STATE)), linear_inputs),
    }
board_mlp = judge({"logistic": reference_runs["logistic"], **mlp_runs}, reference="logistic")
display(board_mlp)


,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,ll_diff,folds_worse,ll_ci_low,ll_ci_high,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,0.709,0.683,0.785,0.558,0.008,0.390,3.923,0.000,0,NaN,NaN,NaN,NaN,reference
mlp,69.250,0.727,-0.021,0.709,0.624,0.731,0.832,0.514,0.190,8.754,0.271,4,0.215,0.342,-0.038,-0.005,worse
mlp_selected,20.250,0.738,-0.011,0.709,0.622,0.767,0.595,0.124,0.230,9.000,0.035,4,0.019,0.051,-0.025,0.006,worse


The default MLP scores F1 0.727 on all columns and 0.738 on the selected ones, both above the floor and both worse than logistic regression on log-loss (0.832 and 0.595 against 0.558, higher in all four years). On all 69 inputs it follows the training years closely (log-loss gap 0.514); on the 20 selected inputs the gap falls to 0.124. Its low thresholds (0.19 and 0.23) show its probabilities sit too low for repeats.



## 3.9. Default scoreboard

In [24]:
default_runs = {**reference_runs, **nb_runs, **lda_runs, **qda_runs, **knn_runs, **tree_runs, **forest_runs, **mlp_runs}
board_default = judge(default_runs, reference="logistic").sort_values("f1", ascending=False)
board_default.loc["dummy", "n_features"] = 0
display(board_default[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse",
                       "balanced_accuracy", "roc_auc", "log_loss", "threshold", "verdict"]])

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,20.250,0.749,0.000,NaN,NaN,0.000,NaN,NaN,0,0.683,0.785,0.558,0.390,reference
lda_selected,20.250,0.745,-0.003,-0.012,0.005,0.013,0.006,0.020,4,0.666,0.778,0.571,0.340,worse
random_forest,41.000,0.745,-0.004,-0.016,0.009,0.028,-0.002,0.064,3,0.661,0.774,0.588,0.400,no clear difference
hist_gb,41.000,0.744,-0.004,-0.018,0.010,0.033,0.017,0.050,4,0.675,0.774,0.589,0.350,worse
lda,69.250,0.739,-0.009,-0.020,0.001,0.034,0.024,0.045,4,0.650,0.762,0.591,0.330,worse
mlp_selected,20.250,0.738,-0.011,-0.025,0.006,0.035,0.019,0.051,4,0.622,0.767,0.595,0.230,worse
knn_selected,20.250,0.731,-0.018,-0.033,-0.003,1.557,1.235,1.916,4,0.644,0.737,2.169,0.300,worse
mlp,69.250,0.727,-0.021,-0.038,-0.005,0.271,0.215,0.342,4,0.624,0.731,0.832,0.190,worse
knn,69.250,0.721,-0.027,-0.044,-0.012,1.552,1.240,1.906,4,0.627,0.737,2.200,0.220,worse


In [25]:
order = board_default.index[::-1]  # best at the top of the chart
fig = go.Figure(go.Scatter(
    x=board_default.loc[order, "f1_diff"], y=order, mode="markers",
    marker=dict(size=10, color=ACCENT["burgundy"]),
    error_x=dict(
        type="data", symmetric=False,
        array=(board_default.loc[order, "ci_high"] - board_default.loc[order, "f1_diff"]).fillna(0),
        arrayminus=(board_default.loc[order, "f1_diff"] - board_default.loc[order, "ci_low"]).fillna(0),
        color=ACCENT["near_black"], thickness=1.2,
    ),
    customdata=board_default.loc[order, ["f1", "ci_low", "ci_high"]],
    hovertemplate="%{y}<br>F1 %{customdata[0]:.3f}<br>difference %{x:+.3f}"
                  "<br>95% interval [%{customdata[1]:.3f}, %{customdata[2]:.3f}]<extra></extra>",
))
fig.add_vline(x=0, line=dict(color="grey", dash="dash"))
fig.update_layout(
    title="Default models: F1 difference from logistic regression, 95% customer-bootstrap interval",
    xaxis_title="F1 minus logistic regression's F1",
    height=460,
)
fig.write_html(FIGURES_DIR / "04_default_scoreboard.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_default_scoreboard.png", **PNG_OPTIONS)
fig.show()

No default model beats logistic regression. The random forest is the only one with no clear difference; every other model is worse. LDA on the selected columns, the random forest and gradient boosting cluster at F1 0.744–0.745, with the MLP on the selected columns next (0.738). QDA, Naive Bayes and the single tree fall to or below the always-repeat floor.



## Findings

| Model | F1 | F1 difference [95% interval] | Log-loss | ROC-AUC | Main issue at defaults |
|---|---|---|---|---|---|
| Logistic regression | 0.749 | reference | 0.558 | 0.785 | — |
| LDA (selected) | 0.745 | -0.003 [-0.012, 0.005] | 0.571 | 0.778 | singular covariance |
| Random forest | 0.745 | -0.004 [-0.016, 0.009] | 0.588 | 0.774 | deep trees, gap 0.435 |
| Gradient boosting | 0.744 | -0.004 [-0.018, 0.010] | 0.589 | 0.774 | gap 0.344 |
| MLP (selected) | 0.738 | -0.011 [-0.025, 0.006] | 0.595 | 0.767 | low probabilities for repeats |
| KNN (selected) | 0.731 | -0.018 [-0.033, -0.003] | 2.169 | 0.737 | six probability values |
| QDA (selected) | 0.700 | -0.049 [-0.096, -0.022] | 2.208 | 0.756 | rank-deficient class covariances |
| Naive Bayes (selected) | 0.687 | -0.062 [-0.117, -0.031] | 2.443 | 0.754 | overconfident probabilities |
| Decision tree | 0.631 | -0.118 [-0.156, -0.086] | 14.478 | 0.597 | one-row leaves, 0/1 probabilities |

- At default settings, logistic regression stays the best model. Each other family fails in a way its check predicted: overconfidence for Naive Bayes and QDA, collinearity for LDA, coarse votes for KNN and overfitting for the trees.
- The elastic-net selection helped Naive Bayes, LDA, QDA, KNN and the MLP, so the selected versions are their defaults from here on.
- Every failure has a matching setting to tune: calibration and smoothing, shrinkage, covariance regularisation, the number of neighbours, the weight penalty, and pruning or depth limits.



# 4. Tuning

Each model family is tuned on the development folds with two Optuna searches of 60 trials each: one steered by log-loss and one by F1. Every trial records both. The preprocessing is fitted once per fold and reused by every trial, because only the model's settings change. The five best distinct trials of each search are rescored and compared with the family's default model under the decision rule.

## 4.0. Search setup

In [26]:
# Fingerprint of the saved inputs: a change in data or configuration starts new searches
fingerprint = hashlib.sha256(EXPERIMENT_VERSION.encode("utf-8"))
for filename in ["train_clean.parquet", "test_clean.parquet", "column_roles.json", "preprocessing_config.json"]:
    fingerprint.update((PROCESSED_DIR / filename).read_bytes())
INPUT_FINGERPRINT = fingerprint.hexdigest()

STORAGE = f"sqlite:///{(OPTUNA_DIR / f'studies_{EXPERIMENT_VERSION}_{INPUT_FINGERPRINT[:12]}.db').as_posix()}"
STUDY_METADATA = {
    "experiment_version": EXPERIMENT_VERSION,
    "input_fingerprint": INPUT_FINGERPRINT,
    "validation_years": [int(year) for year in VAL_YEARS],
    "gap_days": GAP_DAYS,
}
OBJECTIVES = {"log_loss": "minimize", "f1": "maximize"}


class FoldCache:
    """Preprocessing fitted once per fold on the training rows; matrices reused by every trial."""

    def __init__(self, prep, inputs, data=dev):
        self.folds = []
        for (tr, va), year in zip(folds, VAL_YEARS):
            train_part, val_part = data.iloc[tr], data.iloc[va]
            fitted = clone(prep).fit(train_part[inputs], train_part[target])
            self.folds.append({
                "year": year,
                "X_train": np.asarray(fitted.transform(train_part[inputs]), dtype=float),
                "y_train": train_part[target].to_numpy(),
                "X_val": np.asarray(fitted.transform(val_part[inputs]), dtype=float),
                "y_val": val_part[target].to_numpy(),
                "ID": val_part["ID"].to_numpy(),
                "customer_id": val_part["customer_id"].to_numpy(),
                "seen": val_part["customer_id"].isin(train_part["customer_id"]).to_numpy(),
            })


def evaluate_cached(cache, model):
    """Same output as score_folds, on cached matrices."""
    rows, oof = [], []
    for fold in cache.folds:
        start = perf_counter()
        fitted = clone(model).fit(fold["X_train"], fold["y_train"])
        seconds = perf_counter() - start
        p = fitted.predict_proba(fold["X_val"])[:, 1]
        p_train = fitted.predict_proba(fold["X_train"])[:, 1]
        rows.append({
            "fold": fold["year"],
            "n_features": fold["X_train"].shape[1],
            "log_loss": log_loss(fold["y_val"], p, labels=[0, 1]),
            "train_log_loss": log_loss(fold["y_train"], p_train, labels=[0, 1]),
            "roc_auc": roc_auc_score(fold["y_val"], p),
            "fit_seconds": seconds,
        })
        oof.append(pd.DataFrame({key: fold[key] for key in ["ID", "customer_id", "seen"]}
                                | {"fold": fold["year"], "y": fold["y_val"], "p": p}))

    oof = pd.concat(oof, ignore_index=True)[["fold", "ID", "customer_id", "seen", "y", "p"]]
    threshold, _, _ = best_threshold(oof["y"], oof["p"])
    table = pd.DataFrame(rows).set_index("fold")
    table["f1"] = [f1_score(part["y"], part["p"] >= threshold) for _, part in oof.groupby("fold")]
    return table, oof, threshold

def pick_cache(cache, params):
    """A search can switch between caches (e.g. all or selected columns) through a function of its settings."""
    return cache(params) if callable(cache) else cache

def tune(cache, make_model, suggest, objective, name, n_trials=N_TRIALS):
    """Run or resume a stored Optuna search steered by log-loss or F1; every trial records both."""
    if FRESH_SEARCH:
        try:
            optuna.delete_study(study_name=name, storage=STORAGE)
        except KeyError:
            pass

    study = optuna.create_study(
        study_name=name, storage=STORAGE, load_if_exists=True, direction=OBJECTIVES[objective],
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),  # same seed, same trials on a rerun
    )
    expected = {**STUDY_METADATA, "objective": objective}
    if study.trials and any(study.user_attrs.get(k) != v for k, v in expected.items()):
        raise ValueError(f"{name}: stored search was run with other settings. Use a new experiment version.")
    for key, value in expected.items():
        study.set_user_attr(key, value)

    def run_trial(trial):
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)  # a model that does not converge stops the search
            params = suggest(trial)
            table, oof, threshold = evaluate_cached(pick_cache(cache, params), make_model(params))
        metrics = summary_row(table, oof, threshold)
        for key in ["f1", "log_loss", "roc_auc", "balanced_accuracy", "threshold", "overfit_gap"]:
            trial.set_user_attr(key, float(metrics[key]))
        return metrics[objective]

    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    if done < n_trials:
        study.optimize(run_trial, n_trials=n_trials - done, show_progress_bar=False)
    print(f"{name}: {len(study.trials)} trials")
    return study


def trials_table(study):
    """Completed trials, best first by the steering objective."""
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    table = pd.DataFrame([{"trial": t.number, **t.params, **t.user_attrs} for t in done]).set_index("trial")
    objective = study.user_attrs["objective"]
    return table.sort_values(objective, ascending=OBJECTIVES[objective] == "minimize", kind="stable")


def shortlist(studies, cache, make_model, k=5):
    """Rescore the k best trials of each search, named like 'f1_trial_12'."""
    runs = {}
    for objective, study in studies.items():
        # Trials with identical scores are the same model in practice; keep one of each
        distinct = trials_table(study).round({"f1": 4, "log_loss": 4}).drop_duplicates(subset=["f1", "log_loss"])
        for number in distinct.head(k).index:
            params = study.trials[number].params
            runs[f"{objective}_trial_{number}"] = (evaluate_cached(pick_cache(cache, params), make_model(params)), params)
    return runs


def choose_winner(runs, reference):
    """Keep the reference unless a candidate is better; among better ones, fewer columns then lower log-loss."""
    board = judge(runs, reference)
    better = board.index[board["verdict"] == "better"].tolist()
    if not better:
        return reference, board
    best = board.loc[better].sort_values(["n_features", "ll_diff"], ascending=[True, True]).index[0]
    return best, board

### Tuned references and tracking

The tuned models are compared with the two models tuned in notebook 3, read from its decision log: logistic regression (log transform and stronger regularisation, log-loss trial 51) and gradient boosting (log-loss trial 59). Tuned logistic regression is the reference from here on. After each family is tuned, its chosen model joins one comparison against it, shown beside its default to make the effect of tuning visible.


In [27]:
# Tuned references from notebook 3: both models' chosen settings, read from its decision log
hgb_params = next(d.carries["tree_params"] for d in log.items if d.step == "D33")
logistic_params = next(d.carries["linear_params"] for d in log.items if d.step == "D33")


def tuned_logistic():
    """Notebook 3's tuned logistic regression: log transform, selector settings and model C."""
    return build_linear_pipeline(
        LogisticRegression(C=logistic_params.get("model_C", 1.0), max_iter=5000, random_state=RANDOM_STATE),
        log=logistic_params.get("use_log", False), select_C=logistic_params.get("select_C"),
        l1_ratio=logistic_params.get("l1_ratio"),
    )


tuned_runs = {
    "logistic": score_folds(tuned_logistic(), linear_inputs),
    "hist_gb": score_folds(
        build_tree_pipeline(HistGradientBoostingClassifier(**hgb_params, random_state=RANDOM_STATE)), tree_inputs
    ),
}
# Each family's default run, to show what tuning added
default_of = {"logistic": reference_runs["logistic"], "hist_gb": reference_runs["hist_gb"]}


def plot_search(studies, model_name):
    """Search diagnostics, as in notebook 3: history, setting importance, and every trial's scores."""
    for objective, study in studies.items():
        plot_optimization_history(study).update_layout(
            title=f"{model_name}, steered by {objective}: search history").show()
        plot_param_importances(study, evaluator=FanovaImportanceEvaluator(seed=RANDOM_STATE)).update_layout(
            title=f"{model_name}, steered by {objective}: which settings mattered").show()
    plot_slice(studies["log_loss"]).update_layout(title=f"{model_name}: log-loss against each setting").show()

    # Left: log-loss against F1 for every trial. Right: training against validation log-loss.
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Development performance", "Training and validation loss"])
    for i, (objective, study) in enumerate(studies.items()):
        trials = trials_table(study)
        train_loss = trials["log_loss"] - trials["overfit_gap"]
        style = dict(mode="markers", marker=dict(color=SERIES_COLOURS[i], size=7, opacity=0.7),
                     legendgroup=objective, name=f"steered by {objective}")
        fig.add_trace(go.Scatter(x=trials["log_loss"], y=trials["f1"], **style,
                                 hovertemplate="log-loss %{x:.3f}<br>F1 %{y:.3f}<extra></extra>"), row=1, col=1)
        fig.add_trace(go.Scatter(x=train_loss, y=trials["log_loss"], **style, showlegend=False,
                                 hovertemplate="training %{x:.3f}<br>validation %{y:.3f}<extra></extra>"), row=1, col=2)
    fig.add_trace(go.Scatter(x=[0.4, 0.8], y=[0.4, 0.8], mode="lines", line=dict(dash="dash", color="grey"),
                             name="equal losses"), row=1, col=2)
    fig.update_xaxes(title_text="Mean validation log-loss", row=1, col=1)
    fig.update_yaxes(title_text="Pooled F1", row=1, col=1)
    fig.update_xaxes(title_text="Mean training log-loss", row=1, col=2)
    fig.update_yaxes(title_text="Mean validation log-loss", row=1, col=2)
    fig.update_layout(title=f"{model_name}: every trial of both searches", height=450)
    fig.show()


def plot_tuning_progress():
    """Tuned families so far against logistic regression, with each family's default for contrast."""
    board = judge(tuned_runs, reference="logistic")
    ref_f1 = board.loc["logistic", "f1"]
    default_diff = pd.Series({name: summary_row(*run)["f1"] - ref_f1 for name, run in default_of.items()})
    order = board.sort_values("f1").index

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=default_diff[order], y=order, mode="markers", name="default",
        marker=dict(symbol="circle-open", size=11, color=ACCENT["near_black"]),
        hovertemplate="%{y} default<br>difference %{x:+.3f}<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=board.loc[order, "f1_diff"], y=order, mode="markers", name="tuned",
        marker=dict(size=11, color=ACCENT["burgundy"]),
        error_x=dict(type="data", symmetric=False,
                     array=(board.loc[order, "ci_high"] - board.loc[order, "f1_diff"]).fillna(0),
                     arrayminus=(board.loc[order, "f1_diff"] - board.loc[order, "ci_low"]).fillna(0),
                     color=ACCENT["burgundy"], thickness=1.2),
        customdata=board.loc[order, ["f1", "ci_low", "ci_high", "verdict"]],
        hovertemplate="%{y} tuned<br>F1 %{customdata[0]:.3f}<br>difference %{x:+.3f}"
                      "<br>95% interval [%{customdata[1]:.3f}, %{customdata[2]:.3f}]<br>%{customdata[3]}<extra></extra>",
    ))
    fig.add_vline(x=0, line=dict(color="grey", dash="dash"))
    fig.update_layout(
        title=f"Tuned models against logistic regression ({len(board)} families so far)",
        xaxis_title="F1 minus logistic regression's F1 (bars: 95% customer-bootstrap interval)",
        height=150 + 55 * len(board),
    )
    fig.write_html(FIGURES_DIR / "04_tuning_progress.html", **HTML_OPTIONS)
    fig.write_image(FIGURES_DIR / "04_tuning_progress.png", **PNG_OPTIONS)
    fig.show()
    display(board[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "balanced_accuracy",
                   "roc_auc", "log_loss", "threshold", "verdict"]])


def plot_validation_curve(cache, make_model, params, setting, values, model_name, log_x=False):
    """Training and validation F1 and log-loss as one setting varies; the other settings stay as chosen."""
    rows = []
    for value in values:
        settings = {**params, setting: value}
        folds_used = pick_cache(cache, settings).folds
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # convergence and collinearity notes at extreme values
            fitted = [clone(make_model(settings)).fit(f["X_train"], f["y_train"]) for f in folds_used]
            p_train = [m.predict_proba(f["X_train"])[:, 1] for m, f in zip(fitted, folds_used)]
            p_val = [m.predict_proba(f["X_val"])[:, 1] for m, f in zip(fitted, folds_used)]
        y_val = np.concatenate([f["y_val"] for f in folds_used])
        threshold, _, _ = best_threshold(pd.Series(y_val), pd.Series(np.concatenate(p_val)))  # pooled, as everywhere
        rows.append({
            setting: value,
            "train_f1": np.mean([f1_score(f["y_train"], p >= threshold) for f, p in zip(folds_used, p_train)]),
            "validation_f1": f1_score(y_val, np.concatenate(p_val) >= threshold),
            "train_log_loss": np.mean([log_loss(f["y_train"], np.clip(p, 1e-15, 1 - 1e-15), labels=[0, 1])
                                       for f, p in zip(folds_used, p_train)]),
            "validation_log_loss": np.mean([log_loss(f["y_val"], np.clip(p, 1e-15, 1 - 1e-15), labels=[0, 1])
                                            for f, p in zip(folds_used, p_val)]),
        })
    curve = pd.DataFrame(rows).set_index(setting)

    fig = make_subplots(rows=1, cols=2, subplot_titles=["F1 (higher is better)", "Log-loss (lower is better)"])
    for col, metric in enumerate(["f1", "log_loss"], start=1):
        for part, colour in [("train", ACCENT["near_black"]), ("validation", ACCENT["burgundy"])]:
            fig.add_trace(go.Scatter(
                x=curve.index, y=curve[f"{part}_{metric}"], mode="lines+markers", name=part.capitalize(),
                line=dict(color=colour), marker=dict(color=colour, size=8), legendgroup=part, showlegend=col == 1,
                hovertemplate=f"{setting} %{{x}}<br>{metric} %{{y:.3f}}<extra>{part}</extra>",
            ), row=1, col=col)
    if setting in params:  # the chosen value
        fig.add_vline(x=np.log10(params[setting]) if log_x else params[setting],
                      line=dict(dash="dot", color="grey"), annotation_text="chosen")
    fig.update_xaxes(title_text=setting, type="log" if log_x else "linear")
    fig.update_layout(title=f"{model_name}: training and validation scores against {setting}", height=430)
    slug = model_name.lower().replace(" ", "_")
    fig.write_html(FIGURES_DIR / f"04_validation_curve_{slug}.html", **HTML_OPTIONS)
    fig.write_image(FIGURES_DIR / f"04_validation_curve_{slug}.png", **PNG_OPTIONS)
    fig.show()
    display(curve.round(3))
    return curve


## 4.1. Naive Bayes

The default model's main problem was overconfidence. Two settings address it: `var_smoothing`, which adds a share of the largest variance to every variance, and calibration, which relearns the probabilities on held-out parts of the training years (sigmoid or isotonic). The elastic-net columns are kept. `var_smoothing` is searched from 10⁻¹² to 100: an earlier search with an upper limit of 1 placed its best F1 trials at that limit, so the range is widened.

In [28]:
# decision: Naive Bayes keeps the elastic-net selection (better than all columns at defaults)
nb_cache = FoldCache(Pipeline(build_linear_pipeline(GaussianNB()).steps[:-1]), linear_inputs)


def suggest_nb(trial):
    return {
        "var_smoothing": trial.suggest_float("var_smoothing", 1e-12, 100.0, log=True),
        "calibration": trial.suggest_categorical("calibration", ["none", "sigmoid", "isotonic"]),
    }


def make_nb(params):
    model = GaussianNB(var_smoothing=params["var_smoothing"])
    if params["calibration"] == "none":
        return model
    # Calibration learns on 5 internal splits of the training years only
    return CalibratedClassifierCV(model, method=params["calibration"], cv=5)

In [29]:
study_nb = {objective: tune(nb_cache, make_nb, suggest_nb, objective, f"naive_bayes_wide_{objective}")
            for objective in OBJECTIVES}

for objective, study in study_nb.items():
    print(f"\nSteered by {objective}")
    top = trials_table(study).head(5)[["var_smoothing", "calibration", "f1", "log_loss", "roc_auc", "threshold"]]
    display(top.style.format({"var_smoothing": "{:.1e}"}, precision=3))

naive_bayes_wide_log_loss: 60 trials
naive_bayes_wide_f1: 60 trials

Steered by log_loss


,var_smoothing,calibration,f1,log_loss,roc_auc,threshold
trial,,,,,,
32,9.5e+01,sigmoid,0.736,0.573,0.768,0.350
21,9.0e+01,sigmoid,0.736,0.573,0.768,0.350
29,9.0e+01,sigmoid,0.736,0.573,0.768,0.350
19,8.9e+01,sigmoid,0.736,0.573,0.768,0.350
24,8.7e+01,sigmoid,0.736,0.573,0.768,0.350



Steered by f1


,var_smoothing,calibration,f1,log_loss,roc_auc,threshold
trial,,,,,,
22,3.4e+01,sigmoid,0.738,0.574,0.767,0.340
23,3.5e+01,sigmoid,0.738,0.574,0.767,0.340
49,3.9e+00,none,0.738,0.621,0.762,0.310
55,4.7e+01,sigmoid,0.737,0.574,0.767,0.340
56,6.4e+01,sigmoid,0.737,0.573,0.768,0.340


In [30]:
plot_search(study_nb, "Naive Bayes")

Both searches settle on sigmoid calibration (34 and 29 of 60 trials) with heavy smoothing. The log-loss search keeps smoothing between 41 and 95, all at log-loss 0.573–0.574 and F1 0.736–0.737. The F1 search reaches F1 0.737–0.738 with smoothing 13–64; its best uncalibrated trial (smoothing 3.9) has the same F1 but log-loss 0.621. The best log-loss trial (95) again sits near the new upper limit of 100, but the gain has flattened: every log-loss trial above 40 lies within 0.002 of it. The default had F1 0.687 and log-loss 2.443.



In [31]:
nb_candidates = shortlist(study_nb, nb_cache, make_nb)
runs = {"default": nb_runs["naive_bayes_selected"], **{name: run for name, (run, _) in nb_candidates.items()}}
chosen_nb, board_nb_tuned = choose_winner(runs, reference="default")
nb_params = nb_candidates[chosen_nb][1] if chosen_nb != "default" else {}

display(board_nb_tuned[["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_nb} {nb_params}")

,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,
default,0.687,0.000,NaN,NaN,0.000,NaN,NaN,0,2.443,0.754,reference
log_loss_trial_32,0.736,0.049,0.015,0.109,-1.842,-2.178,-1.539,0,0.573,0.768,better
log_loss_trial_21,0.736,0.050,0.015,0.109,-1.842,-2.178,-1.539,0,0.573,0.768,better
log_loss_trial_54,0.736,0.050,0.015,0.109,-1.842,-2.178,-1.539,0,0.573,0.768,better
log_loss_trial_51,0.737,0.051,0.015,0.110,-1.841,-2.177,-1.538,0,0.573,0.768,better
log_loss_trial_16,0.737,0.051,0.015,0.110,-1.840,-2.176,-1.538,0,0.574,0.767,better
f1_trial_22,0.738,0.051,0.016,0.111,-1.840,-2.175,-1.538,0,0.574,0.767,better
f1_trial_23,0.738,0.051,0.016,0.111,-1.840,-2.175,-1.538,0,0.574,0.767,better
f1_trial_49,0.738,0.051,0.021,0.102,-1.794,-2.138,-1.483,0,0.621,0.762,better


Chosen: log_loss_trial_32 {'var_smoothing': 94.98576391975146, 'calibration': 'sigmoid'}


In [32]:
tuned_runs["naive_bayes"] = runs[chosen_nb]
default_of["naive_bayes"] = nb_runs["naive_bayes_selected"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,22.000,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
hist_gb,41.000,0.748,0.001,-0.013,0.021,-0.006,-0.013,0.002,1,0.670,0.790,0.541,0.360,no clear difference
naive_bayes,20.250,0.736,-0.012,-0.027,0.011,0.026,0.014,0.039,4,0.624,0.768,0.573,0.350,worse


### Validation curve

The validation curve varies one setting while the others stay at their chosen values. The dark line is the training score (mean over the four folds), the red line the validation score on the later years. Where the lines move apart, the model is learning the training years rather than patterns that carry over.


In [33]:
nb_curve = plot_validation_curve(nb_cache, make_nb, nb_params or {"var_smoothing": 1e-9, "calibration": "none"},
                                 "var_smoothing", np.logspace(-12, 2, 8), "Naive Bayes", log_x=True)

,train_f1,validation_f1,train_log_loss,validation_log_loss
var_smoothing,,,,
0.000,0.703,0.716,0.599,0.597
0.000,0.703,0.716,0.599,0.597
0.000,0.703,0.716,0.598,0.597
0.000,0.703,0.716,0.595,0.597
0.000,0.703,0.716,0.595,0.597
0.010,0.705,0.716,0.593,0.593
1.000,0.732,0.729,0.580,0.588
100.000,0.733,0.737,0.571,0.573


All ten candidates pass the rule against the default: log-loss falls from 2.443 to 0.573–0.621 and every F1 interval lies above zero. Log-loss trial 32 is chosen as the lowest log-loss at the same column count: sigmoid calibration with smoothing 95, F1 0.736 against 0.687. Against logistic regression, tuned Naive Bayes stays worse (log-loss 0.573 against 0.547, interval [0.014, 0.039], higher in all four years).

The validation curve shows why such large smoothing helps. Training and validation scores stay together at every value (log-loss 0.599 and 0.597 at 10⁻¹², 0.571 and 0.573 at 100), so Naive Bayes is not overfitting. The smoothing removes bias instead: it flattens the badly skewed class distributions, and validation F1 rises from 0.716 to 0.737 while log-loss falls from 0.597 to 0.573.



In [34]:
# Reliability on the pooled development predictions: default against the best calibrated trial
best_calibrated = board_nb_tuned.drop(index="default").sort_values("f1", ascending=False).index[0]
curves = {"default (uncalibrated)": runs["default"][1], f"calibrated ({best_calibrated})": runs[best_calibrated][1]}

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration",
                         line=dict(dash="dash", color="grey")))
brier = {}
for name, oof in curves.items():
    observed, predicted = calibration_curve(oof["y"], oof["p"], n_bins=10, strategy="quantile")
    brier[name] = brier_score_loss(oof["y"], oof["p"])
    fig.add_trace(go.Scatter(x=predicted, y=observed, mode="lines+markers", name=name,
                             hovertemplate="predicted %{x:.2f}<br>observed %{y:.2f}<extra>" + name + "</extra>"))
fig.update_layout(
    title="Naive Bayes: predicted against observed repeat share (deciles of the predictions)",
    xaxis=dict(title="Mean predicted probability", range=[0, 1]),
    yaxis=dict(title="Observed repeat share", range=[0, 1]),
    height=480,
)
fig.write_html(FIGURES_DIR / "04_naive_bayes_calibration.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_naive_bayes_calibration.png", **PNG_OPTIONS)
fig.show()

print({name: round(value, 3) for name, value in brier.items()})

{'default (uncalibrated)': 0.305, 'calibrated (f1_trial_23)': 0.195}


Calibration repairs the probabilities: the Brier score falls from 0.305 for the uncalibrated default to 0.195 for the calibrated F1 trial 23 (sigmoid, smoothing 35). The uncalibrated model pushes most predictions to 0 or 1; after calibration the deciles lie much closer to the diagonal of the reliability chart.



## 4.2. Linear discriminant analysis

The default LDA ran on a singular covariance matrix. Shrinkage pulls the estimated covariance towards a diagonal one, which makes it invertible and more stable. The search tries no shrinkage, the automatic Ledoit–Wolf amount and fixed amounts from 0.0001 to 1, on all prepared columns or on the elastic-net selection.

In [35]:
# Two cached input sets: all prepared columns, or the elastic-net selection
lda_caches = {
    "all": FoldCache(build_linear_pipeline(LinearDiscriminantAnalysis(), select=False).named_steps["prep"], linear_inputs),
    "selected": FoldCache(Pipeline(build_linear_pipeline(LinearDiscriminantAnalysis()).steps[:-1]), linear_inputs),
}


def suggest_lda(trial):
    params = {
        "columns": trial.suggest_categorical("columns", ["all", "selected"]),
        "shrinkage_mode": trial.suggest_categorical("shrinkage_mode", ["none", "auto", "value"]),
    }
    if params["shrinkage_mode"] == "value":
        params["shrinkage"] = trial.suggest_float("shrinkage", 1e-4, 1.0, log=True)
    return params


def make_lda(params):
    if params["shrinkage_mode"] == "none":
        return LinearDiscriminantAnalysis(solver="svd")
    # Shrinkage needs a solver that estimates the covariance: lsqr
    shrinkage = "auto" if params["shrinkage_mode"] == "auto" else params["shrinkage"]
    return LinearDiscriminantAnalysis(solver="lsqr", shrinkage=shrinkage)

In [36]:
get_lda_cache = lambda params: lda_caches[params["columns"]]

study_lda = {objective: tune(get_lda_cache, make_lda, suggest_lda, objective, f"lda_{objective}")
             for objective in OBJECTIVES}

for objective, study in study_lda.items():
    print(f"\nSteered by {objective}")
    top = trials_table(study).head(5).reindex(
        columns=["columns", "shrinkage_mode", "shrinkage", "f1", "log_loss", "roc_auc", "threshold"]
    )
    display(top.style.format({"shrinkage": "{:.3g}"}, precision=3, na_rep="-"))

lda_log_loss: 60 trials
lda_f1: 60 trials

Steered by log_loss


,columns,shrinkage_mode,shrinkage,f1,log_loss,roc_auc,threshold
trial,,,,,,,
54,selected,value,0.114,0.743,0.565,0.780,0.330
56,selected,value,0.118,0.743,0.565,0.780,0.320
52,selected,value,0.1,0.744,0.565,0.780,0.440
53,selected,value,0.0988,0.744,0.565,0.780,0.440
59,selected,value,0.0906,0.745,0.565,0.780,0.440



Steered by f1


,columns,shrinkage_mode,shrinkage,f1,log_loss,roc_auc,threshold
trial,,,,,,,
10,selected,auto,-,0.748,0.569,0.779,0.320
11,selected,auto,-,0.748,0.569,0.779,0.320
12,selected,auto,-,0.748,0.569,0.779,0.320
13,selected,auto,-,0.748,0.569,0.779,0.320
15,selected,auto,-,0.748,0.569,0.779,0.320


In [37]:
plot_search(study_lda, "LDA")

The two searches find different optima. Log-loss prefers the selected columns with a fixed, moderate shrinkage (about 0.09–0.12), at log-loss 0.565. F1 prefers the automatic Ledoit–Wolf amount on the selected columns (41 of 60 trials), at F1 0.748 and log-loss 0.569.


In [38]:
lda_candidates = shortlist(study_lda, get_lda_cache, make_lda)
runs = {"default": lda_runs["lda_selected"], **{name: run for name, (run, _) in lda_candidates.items()}}
chosen_lda, board_lda_tuned = choose_winner(runs, reference="default")
lda_params = lda_candidates[chosen_lda][1] if chosen_lda != "default" else {}

display(board_lda_tuned[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_lda} {lda_params}")

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,,
default,20.250,0.745,0.000,NaN,NaN,0.000,NaN,NaN,0,0.571,0.778,reference
log_loss_trial_54,20.250,0.743,-0.002,-0.008,0.003,-0.006,-0.012,-0.000,0,0.565,0.780,better
log_loss_trial_56,20.250,0.743,-0.002,-0.008,0.005,-0.006,-0.012,-0.000,0,0.565,0.780,better
log_loss_trial_52,20.250,0.744,-0.001,-0.017,0.011,-0.006,-0.012,-0.001,0,0.565,0.780,better
log_loss_trial_59,20.250,0.745,-0.000,-0.017,0.011,-0.006,-0.011,-0.001,0,0.565,0.780,better
log_loss_trial_51,20.250,0.747,0.001,-0.015,0.013,-0.006,-0.011,-0.001,0,0.565,0.780,better
f1_trial_10,20.250,0.748,0.003,-0.002,0.009,-0.002,-0.005,0.000,1,0.569,0.779,no clear difference
f1_trial_5,20.250,0.747,0.002,-0.001,0.006,-0.000,-0.000,-0.000,0,0.571,0.778,better
f1_trial_30,69.250,0.746,0.001,-0.005,0.007,0.003,-0.001,0.007,3,0.573,0.774,no clear difference


Chosen: log_loss_trial_54 {'columns': 'selected', 'shrinkage_mode': 'value', 'shrinkage': 0.11423379054425592}


In [39]:
tuned_runs["lda"] = runs[chosen_lda]
default_of["lda"] = lda_runs["lda_selected"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,22.000,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
hist_gb,41.000,0.748,0.001,-0.013,0.021,-0.006,-0.013,0.002,1,0.670,0.790,0.541,0.360,no clear difference
naive_bayes,20.250,0.736,-0.012,-0.027,0.011,0.026,0.014,0.039,4,0.624,0.768,0.573,0.350,worse
lda,20.250,0.743,-0.004,-0.017,0.014,0.017,0.008,0.027,4,0.659,0.780,0.565,0.330,worse


### Validation curve

The validation curve varies one setting while the others stay at their chosen values. The dark line is the training score (mean over the four folds), the red line the validation score on the later years. Where the lines move apart, the model is learning the training years rather than patterns that carry over.


In [40]:
lda_base = lda_params if lda_params.get("shrinkage_mode") == "value" else {
    "columns": lda_params.get("columns", "selected"), "shrinkage_mode": "value"}
lda_curve = plot_validation_curve(get_lda_cache, make_lda, lda_base, "shrinkage", np.logspace(-4, 0, 9), "LDA",
                                  log_x=True)

,train_f1,validation_f1,train_log_loss,validation_log_loss
shrinkage,,,,
0.000,0.743,0.745,0.560,0.571
0.000,0.741,0.747,0.560,0.571
0.001,0.743,0.745,0.560,0.571
0.003,0.742,0.745,0.560,0.571
0.010,0.744,0.744,0.560,0.570
0.032,0.742,0.746,0.560,0.567
0.100,0.743,0.744,0.561,0.565
0.316,0.738,0.741,0.564,0.571
1.000,0.732,0.734,0.873,0.899


Seven candidates pass the rule against the default LDA. Log-loss trial 54 is chosen: selected columns with shrinkage 0.114, log-loss 0.565 against 0.571 (interval [-0.012, -0.000]) and F1 0.743 against 0.745. The F1-search trials with automatic shrinkage gain at most +0.003 in F1. Against tuned logistic regression, LDA stays worse (log-loss 0.565 against 0.547, interval [0.008, 0.027], higher in all four years).

The validation curve shows a model that does not overfit: training and validation log-loss stay within 0.011 of each other up to shrinkage 0.32. Validation log-loss falls from 0.571 with almost no shrinkage to 0.565 at 0.1, then rises to 0.571 at 0.32 and 0.899 at 1, where the covariance becomes diagonal and LDA ignores how the columns move together. Training log-loss rises too (0.873 at 1), so the loss at large shrinkage is underfitting, not overfitting.



In [41]:
# F1 and log-loss across the shrinkage values tried (both searches, fixed-value trials only)
tried = pd.concat([trials_table(study) for study in study_lda.values()])
tried = tried.loc[tried["shrinkage_mode"] == "value"]
assert len(tried), "No search tried a fixed shrinkage value."
tried = tried.sort_values("shrinkage")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Pooled F1", "Mean validation log-loss"])
for i, (columns, part) in enumerate(tried.groupby("columns")):
    for col, metric in enumerate(["f1", "log_loss"], start=1):
        fig.add_trace(go.Scatter(
            x=part["shrinkage"], y=part[metric], mode="markers", name=f"{columns} columns",
            marker=dict(color=SERIES_COLOURS[i], size=8), legendgroup=columns, showlegend=col == 1,
            hovertemplate="shrinkage %{x:.3g}<br>" + metric + " %{y:.3f}<extra>" + columns + "</extra>",
        ), row=1, col=col)
fig.update_xaxes(type="log", title="Shrinkage (log scale)")
fig.update_layout(title="LDA: effect of shrinkage on F1 and log-loss", height=430)
fig.write_html(FIGURES_DIR / "04_lda_shrinkage.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_lda_shrinkage.png", **PNG_OPTIONS)
fig.show()

F1 barely moves with shrinkage: 0.737–0.747 across the fixed amounts tried. Log-loss shows the trade-off more clearly. On the selected columns it falls from 0.571 with almost no shrinkage to 0.565 around 0.1, then rises to 0.619 at the largest amount tried (0.67), where the covariance moves towards a diagonal one and the model ignores how the columns move together. All columns behave the same way (0.576 to 0.570, then 0.622 at 0.50).


## 4.3. Quadratic discriminant analysis

The default QDA cannot invert the class covariances. `reg_param` shrinks each class's covariance towards the identity, in the same spirit as LDA's shrinkage: at 0 it is plain QDA, at 1 every column is treated as independent with unit variance. The search tries values from 0.001 to 1, on all prepared columns or on the elastic-net selection. The range cannot go past 1, its maximum, even if the best trials sit there.


In [42]:
# Two cached input sets, as for LDA
qda_caches = {
    "all": FoldCache(build_linear_pipeline(QuadraticDiscriminantAnalysis(), select=False).named_steps["prep"], linear_inputs),
    "selected": FoldCache(Pipeline(build_linear_pipeline(QuadraticDiscriminantAnalysis()).steps[:-1]), linear_inputs),
}
get_qda_cache = lambda params: qda_caches[params["columns"]]


def suggest_qda(trial):
    return {
        "columns": trial.suggest_categorical("columns", ["all", "selected"]),
        "reg_param": trial.suggest_float("reg_param", QDA_MIN_REG, 1.0, log=True),
    }


def make_qda(params):
    return QuadraticDiscriminantAnalysis(reg_param=params["reg_param"])


with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)  # collinearity warnings at the smallest reg_param values
    study_qda = {objective: tune(get_qda_cache, make_qda, suggest_qda, objective, f"qda_{objective}")
                 for objective in OBJECTIVES}

for objective, study in study_qda.items():
    print(f"\nSteered by {objective}")
    display(trials_table(study).head(5).reindex(
        columns=["columns", "reg_param", "f1", "log_loss", "roc_auc", "threshold"]).style.format(
        {"reg_param": "{:.3g}"}, precision=3))


qda_log_loss: 60 trials
qda_f1: 60 trials

Steered by log_loss


,columns,reg_param,f1,log_loss,roc_auc,threshold
trial,,,,,,
33,selected,0.995,0.736,0.702,0.768,0.140
32,selected,0.989,0.735,0.704,0.768,0.150
43,selected,0.98,0.734,0.707,0.767,0.160
19,selected,0.977,0.735,0.708,0.767,0.160
24,selected,0.972,0.735,0.709,0.767,0.160



Steered by f1


,columns,reg_param,f1,log_loss,roc_auc,threshold
trial,,,,,,
51,all,0.871,0.738,1.236,0.756,0.090
49,all,0.94,0.738,1.223,0.755,0.090
52,all,0.941,0.738,1.223,0.755,0.090
10,all,0.925,0.738,1.225,0.755,0.090
32,all,0.989,0.737,1.199,0.755,0.060


In [43]:
plot_search(study_qda, "QDA")

In [44]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    qda_candidates = shortlist(study_qda, get_qda_cache, make_qda)
runs = {"default": qda_runs["qda_selected"], **{name: run for name, (run, _) in qda_candidates.items()}}
chosen_qda, board_qda_tuned = choose_winner(runs, reference="default")
qda_params = qda_candidates[chosen_qda][1] if chosen_qda != "default" else {}

display(board_qda_tuned[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high",
                         "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_qda} {qda_params}")


,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,,
default,20.250,0.700,0.000,NaN,NaN,0.000,NaN,NaN,0,2.208,0.756,reference
log_loss_trial_33,20.250,0.736,0.036,0.008,0.086,-1.469,-1.755,-1.222,0,0.702,0.768,better
log_loss_trial_32,20.250,0.735,0.035,0.007,0.084,-1.467,-1.754,-1.221,0,0.704,0.768,better
log_loss_trial_43,20.250,0.734,0.034,0.008,0.082,-1.465,-1.749,-1.219,0,0.707,0.767,better
log_loss_trial_19,20.250,0.735,0.035,0.008,0.084,-1.464,-1.748,-1.218,0,0.708,0.767,better
log_loss_trial_24,20.250,0.735,0.035,0.008,0.084,-1.462,-1.746,-1.217,0,0.709,0.767,better
f1_trial_51,69.250,0.738,0.038,0.012,0.084,-0.940,-1.184,-0.739,0,1.236,0.756,better
f1_trial_49,69.250,0.738,0.038,0.012,0.081,-0.955,-1.189,-0.756,0,1.223,0.755,better
f1_trial_52,69.250,0.738,0.038,0.012,0.081,-0.955,-1.189,-0.756,0,1.223,0.755,better


Chosen: log_loss_trial_33 {'columns': 'selected', 'reg_param': 0.9952086340284035}


In [45]:
tuned_runs["qda"] = runs[chosen_qda]
default_of["qda"] = qda_runs["qda_selected"]
plot_tuning_progress()


,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,22.000,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
hist_gb,41.000,0.748,0.001,-0.013,0.021,-0.006,-0.013,0.002,1,0.670,0.790,0.541,0.360,no clear difference
naive_bayes,20.250,0.736,-0.012,-0.027,0.011,0.026,0.014,0.039,4,0.624,0.768,0.573,0.350,worse
lda,20.250,0.743,-0.004,-0.017,0.014,0.017,0.008,0.027,4,0.659,0.780,0.565,0.330,worse
qda,20.250,0.736,-0.011,-0.026,0.008,0.156,0.112,0.203,4,0.635,0.768,0.702,0.140,worse


### Validation curve

The validation curve varies one setting while the others stay at their chosen values. The dark line is the training score (mean over the four folds), the red line the validation score on the later years. Where the lines move apart, the model is learning the training years rather than patterns that carry over.


In [46]:
qda_curve = plot_validation_curve(get_qda_cache, make_qda, qda_params or {"columns": "selected", "reg_param": QDA_MIN_REG},
                                  "reg_param", np.logspace(-3, 0, 8), "QDA", log_x=True)

,train_f1,validation_f1,train_log_loss,validation_log_loss
reg_param,,,,
0.001,0.711,0.700,1.622,2.198
0.003,0.713,0.701,1.588,2.154
0.007,0.717,0.703,1.511,2.050
0.019,0.725,0.711,1.380,1.877
0.052,0.735,0.716,1.201,1.640
0.139,0.738,0.720,0.990,1.345
0.373,0.732,0.721,0.786,1.043
1.000,0.732,0.735,0.681,0.701


Both searches push `reg_param` to the top of its range. The log-loss search prefers the selected columns with values of 0.90–0.995 (log-loss 0.702–0.738); the F1 search prefers all columns with values of 0.72–0.99, at F1 0.736–0.738 but log-loss 1.20–1.31. All ten candidates pass the rule against the default, and log-loss trial 33 is chosen: selected columns, `reg_param` 0.995, F1 0.736 against 0.700 and log-loss 0.702 against 2.208.

The validation curve explains the edge. At small values each class covariance is fitted closely to the training years: training log-loss 1.622 against 2.198 on validation at 0.001. The gap closes steadily as `reg_param` grows, to 0.681 against 0.701 at 1. Validation F1 rises from 0.700 to 0.735 over the same range. At 1 each column is treated as independent with unit variance inside each class, so the curved boundary that QDA was added for is mostly gone. The range cannot be widened past 1. Even at its best, QDA is clearly worse than logistic regression (log-loss 0.702 against 0.547, interval [0.112, 0.203]), and its threshold of 0.14 shows its probabilities are still too low.



## 4.4. K-nearest neighbours

Four settings are searched: the input columns (all or the elastic-net selection), the number of neighbours k from 1 to 501 (an earlier search up to 151 placed its best F1 trials near that limit), uniform or distance-weighted votes, and Manhattan (p = 1) or Euclidean (p = 2) distance. Manhattan distance adds absolute differences, so a single extreme column weighs less than with squared differences.

In [47]:
knn_caches = {
    "all": FoldCache(build_linear_pipeline(KNeighborsClassifier(), select=False).named_steps["prep"], linear_inputs),
    "selected": FoldCache(Pipeline(build_linear_pipeline(KNeighborsClassifier()).steps[:-1]), linear_inputs),
}


def suggest_knn(trial):
    return {
        "columns": trial.suggest_categorical("columns", ["all", "selected"]),
        "n_neighbors": trial.suggest_int("n_neighbors", 1, 501, log=True),
        "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
        "p": trial.suggest_categorical("p", [1, 2]),  # 1 Manhattan, 2 Euclidean
    }


def make_knn(params):
    return KNeighborsClassifier(n_neighbors=params["n_neighbors"], weights=params["weights"], p=params["p"])

In [48]:
get_knn_cache = lambda params: knn_caches[params["columns"]]

study_knn = {objective: tune(get_knn_cache, make_knn, suggest_knn, objective, f"knn_wide_{objective}")
             for objective in OBJECTIVES}

for objective, study in study_knn.items():
    print(f"\nSteered by {objective}")
    display(trials_table(study).head(5)[["columns", "n_neighbors", "weights", "p", "f1", "log_loss", "roc_auc", "threshold"]])

knn_wide_log_loss: 60 trials
knn_wide_f1: 60 trials

Steered by log_loss


,columns,n_neighbors,weights,p,f1,log_loss,roc_auc,threshold
trial,,,,,,,,
14,selected,235,distance,1,0.742,0.558,0.780,0.380
34,selected,252,distance,1,0.744,0.558,0.780,0.390
43,selected,258,distance,1,0.740,0.559,0.780,0.380
37,selected,255,distance,1,0.741,0.559,0.779,0.380
51,selected,256,distance,1,0.745,0.559,0.779,0.390



Steered by f1


,columns,n_neighbors,weights,p,f1,log_loss,roc_auc,threshold
trial,,,,,,,,
55,selected,291,distance,1,0.747,0.560,0.780,0.390
59,selected,192,distance,1,0.746,0.569,0.779,0.370
57,selected,195,distance,1,0.745,0.569,0.779,0.370
28,selected,95,distance,1,0.745,0.566,0.778,0.370
29,selected,122,distance,1,0.745,0.566,0.780,0.360


In [49]:
plot_search(study_knn, "KNN")

Both searches converge on the selected columns, distance-weighted votes and Manhattan distance (40 and 37 of 60 trials use distance weights). The log-loss search settles on 235–258 neighbours (log-loss 0.558–0.559, F1 0.740–0.745). The F1 search spreads from 95 to 291 neighbours (F1 0.745–0.747, log-loss 0.560–0.569). An earlier search capped at 151 placed its best trials near that limit; with the wider range the best values sit well inside it, and log-loss improves to 0.558. ROC-AUC rises from 0.737 at the default to 0.778–0.780.



In [50]:
knn_candidates = shortlist(study_knn, get_knn_cache, make_knn)
runs = {"default": knn_runs["knn_selected"], **{name: run for name, (run, _) in knn_candidates.items()}}
chosen_knn, board_knn_tuned = choose_winner(runs, reference="default")
knn_params = knn_candidates[chosen_knn][1] if chosen_knn != "default" else {}

display(board_knn_tuned[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_knn} {knn_params}")

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,,
default,20.250,0.731,0.000,NaN,NaN,0.000,NaN,NaN,0,2.169,0.737,reference
log_loss_trial_14,20.250,0.742,0.011,-0.002,0.026,-1.557,-1.923,-1.227,0,0.558,0.780,better
log_loss_trial_34,20.250,0.744,0.013,-0.001,0.027,-1.557,-1.922,-1.226,0,0.558,0.780,better
log_loss_trial_43,20.250,0.740,0.009,-0.004,0.024,-1.556,-1.922,-1.226,0,0.559,0.780,better
log_loss_trial_37,20.250,0.741,0.010,-0.003,0.024,-1.556,-1.922,-1.226,0,0.559,0.779,better
log_loss_trial_51,20.250,0.745,0.014,0.000,0.029,-1.556,-1.922,-1.226,0,0.559,0.779,better
f1_trial_55,20.250,0.747,0.016,0.003,0.031,-1.555,-1.922,-1.224,0,0.560,0.780,better
f1_trial_59,20.250,0.746,0.015,0.002,0.029,-1.547,-1.923,-1.216,0,0.569,0.779,better
f1_trial_57,20.250,0.745,0.015,0.002,0.029,-1.547,-1.923,-1.216,0,0.569,0.779,better


Chosen: log_loss_trial_14 {'columns': 'selected', 'n_neighbors': 235, 'weights': 'distance', 'p': 1}


In [51]:
tuned_runs["knn"] = runs[chosen_knn]
default_of["knn"] = knn_runs["knn_selected"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,22.000,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
hist_gb,41.000,0.748,0.001,-0.013,0.021,-0.006,-0.013,0.002,1,0.670,0.790,0.541,0.360,no clear difference
naive_bayes,20.250,0.736,-0.012,-0.027,0.011,0.026,0.014,0.039,4,0.624,0.768,0.573,0.350,worse
lda,20.250,0.743,-0.004,-0.017,0.014,0.017,0.008,0.027,4,0.659,0.780,0.565,0.330,worse
qda,20.250,0.736,-0.011,-0.026,0.008,0.156,0.112,0.203,4,0.635,0.768,0.702,0.140,worse
knn,20.250,0.742,-0.005,-0.020,0.013,0.011,-0.001,0.023,3,0.667,0.780,0.558,0.380,no clear difference


All ten candidates pass the rule against the default: log-loss falls from 2.169 to 0.558–0.569, because many neighbours give smooth probabilities instead of six coarse ones. Log-loss trial 14 is chosen as the lowest log-loss at the selected column count: 235 neighbours, distance-weighted votes and Manhattan distance, F1 0.742 against 0.731. Against logistic regression, tuned KNN shows no clear difference (log-loss 0.558 against 0.547, interval [-0.001, 0.023]), although its log-loss is higher in three of four years.



### Validation curve (elbow)

The validation curve varies one setting while the others stay at their chosen values. The dark line is the training score (mean over the four folds), the red line the validation score on the later years. Where the lines move apart, the model is learning the training years rather than patterns that carry over.


In [52]:
knn_curve = plot_validation_curve(lambda p: knn_caches[p["columns"]], make_knn,
                                  knn_params or {"columns": "selected", "weights": "uniform", "p": 2},
                                  "n_neighbors", [1, 3, 5, 10, 25, 50, 100, 200, 300, 500], "KNN", log_x=True)

,train_f1,validation_f1,train_log_loss,validation_log_loss
n_neighbors,,,,
1,1.000,0.674,0.000,12.012
3,1.000,0.731,0.000,3.820
5,1.000,0.741,0.000,2.106
10,1.000,0.735,0.000,0.877
25,1.000,0.740,0.000,0.588
50,1.000,0.743,0.000,0.566
100,1.000,0.742,0.000,0.566
200,1.000,0.744,0.000,0.568
300,1.000,0.744,0.000,0.560


With distance-weighted votes, the training scores are perfect at every k (F1 1.000, log-loss 0.000): each training row is its own nearest neighbour at distance zero, so it decides its own vote. Only the validation curve is informative here. Validation log-loss falls from 12.0 at k = 1 to 0.588 at k = 25 and 0.566 at k = 50, reaches its lowest value (0.560) at k = 300 and rises to 0.568 at 500. Validation F1 rises from 0.674 at k = 1 to 0.741 at k = 5 and then stays between 0.735 and 0.744. The elbow is around k = 25 for F1 and k = 50 for log-loss; beyond that, more neighbours change the scores by less than 0.01.



## 4.5. Decision tree

The default tree grew until almost every leaf was pure and fell below the floor. Three settings limit its growth: `max_depth` caps the number of questions (1–30; an earlier search starting at 2 placed its best log-loss trials at that limit, so single-split stumps are now allowed), `min_samples_leaf` sets the smallest group that can answer, and `ccp_alpha` removes branches whose impurity reduction is worth less than alpha per leaf (cost-complexity pruning). The split criterion, class weights and the share of columns tried at each split (`max_features`, 20–100%) are searched too. Before the search, the pruning path shows how size and performance move together; after it, a validation curve over depth.

### Pruning path

In [53]:
# Missing values stay in the tree branch; preprocessing is cached once per fold
tree_cache = FoldCache(build_tree_pipeline(DecisionTreeClassifier()).named_steps["prep"], tree_inputs)

# Cost-complexity pruning path of the full tree in the last fold: the alphas at which branches disappear
last = tree_cache.folds[-1]
path = DecisionTreeClassifier(random_state=RANDOM_STATE).cost_complexity_pruning_path(last["X_train"], last["y_train"])
# Log-spaced, so small and large prunings are both visible
alphas = np.geomspace(1e-5, path.ccp_alphas[:-1].max(), 30)

path_rows = []
for alpha in alphas:
    table, oof, threshold = evaluate_cached(tree_cache, DecisionTreeClassifier(ccp_alpha=alpha, random_state=RANDOM_STATE))
    leaves = DecisionTreeClassifier(ccp_alpha=alpha, random_state=RANDOM_STATE).fit(
        last["X_train"], last["y_train"]).get_n_leaves()
    path_rows.append({"ccp_alpha": alpha, "leaves_last_fold": leaves,
                      "validation_f1": f1_score(oof["y"], oof["p"] >= threshold),
                      "log_loss": table["log_loss"].mean()})
pruning = pd.DataFrame(path_rows)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=pruning["ccp_alpha"], y=pruning["leaves_last_fold"], mode="lines+markers",
                         name="leaves (last fold)", line=dict(color=ACCENT["near_black"]),
                         hovertemplate="alpha %{x:.2e}<br>%{y} leaves<extra></extra>"), secondary_y=False)
fig.add_trace(go.Scatter(x=pruning["ccp_alpha"], y=pruning["validation_f1"], mode="lines+markers",
                         name="validation F1 (pooled)", line=dict(color=ACCENT["burgundy"]),
                         hovertemplate="alpha %{x:.2e}<br>F1 %{y:.3f}<extra></extra>"), secondary_y=True)
fig.add_hline(y=f1_floor(dev[target]), line=dict(dash="dash", color="grey"), secondary_y=True,
              annotation_text="always repeat", annotation_position="bottom right")
fig.update_xaxes(type="log", title="ccp_alpha (log scale)")
fig.update_yaxes(title_text="Leaves", type="log", secondary_y=False)
fig.update_yaxes(title_text="F1", secondary_y=True)
fig.update_layout(title="Decision tree: pruning path, tree size and validation F1", height=450)
fig.write_html(FIGURES_DIR / "04_tree_pruning_path.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_tree_pruning_path.png", **PNG_OPTIONS)
fig.show()

best_alpha = pruning.loc[pruning["validation_f1"].idxmax()]
print(f"Path alphas: {path.ccp_alphas.min():.1e} to {path.ccp_alphas[:-1].max():.1e}")
print(f"Best on the path: alpha {best_alpha['ccp_alpha']:.1e}, {best_alpha['leaves_last_fold']:.0f} leaves, "
      f"F1 {best_alpha['validation_f1']:.3f}")

Path alphas: 0.0e+00 to 1.1e-02
Best on the path: alpha 3.3e-03, 7 leaves, F1 0.741


The pruning path of the last fold runs from alpha 0 to about 1.1 × 10⁻². Small alphas leave the hundreds of one-row leaves in place. The best point on the path is alpha 3.3 × 10⁻³, a tree of only 7 leaves with F1 0.741: most of the gain comes from removing the small leaves.


### Search

In [54]:
def suggest_tree(trial):
    return {
        "max_depth": trial.suggest_int("max_depth", 1, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 300, log=True),
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-6, 0.05, log=True),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "class_weight": trial.suggest_categorical("class_weight", ["none", "balanced"]),
        "max_features": trial.suggest_float("max_features", 0.2, 1.0),
    }


def make_tree(params):
    settings = {**params, "class_weight": None if params["class_weight"] == "none" else "balanced"}
    return DecisionTreeClassifier(**settings, random_state=RANDOM_STATE)


study_tree = {objective: tune(tree_cache, make_tree, suggest_tree, objective, f"decision_tree_wide_{objective}")
              for objective in OBJECTIVES}

for objective, study in study_tree.items():
    print(f"\nSteered by {objective}")
    top = trials_table(study).head(5)[["max_depth", "min_samples_leaf", "ccp_alpha", "criterion", "class_weight",
                                       "max_features", "f1", "log_loss", "roc_auc", "threshold"]]
    display(top.style.format({"ccp_alpha": "{:.1e}"}, precision=3))

decision_tree_wide_log_loss: 60 trials
decision_tree_wide_f1: 60 trials

Steered by log_loss


,max_depth,min_samples_leaf,ccp_alpha,criterion,class_weight,max_features,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,,
58,8,147,1.4e-03,gini,none,0.756,0.745,0.554,0.773,0.330
35,12,138,5.9e-04,gini,none,0.791,0.746,0.561,0.768,0.360
37,13,128,4.2e-03,gini,none,0.796,0.736,0.562,0.756,0.430
44,2,123,1.3e-03,gini,none,0.687,0.729,0.563,0.752,0.450
0,12,219,2.8e-03,gini,none,0.893,0.736,0.567,0.763,0.370



Steered by f1


,max_depth,min_samples_leaf,ccp_alpha,criterion,class_weight,max_features,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,,
20,23,78,5.4e-06,gini,none,0.244,0.751,0.565,0.773,0.410
49,24,78,3.6e-05,gini,none,0.253,0.751,0.565,0.773,0.410
24,30,114,2.6e-06,gini,none,0.324,0.746,0.582,0.763,0.350
40,21,195,4.3e-05,gini,none,0.204,0.745,0.563,0.766,0.430
42,17,235,3.0e-05,gini,none,0.299,0.745,0.564,0.764,0.380


In [55]:
plot_search(study_tree, "Decision tree")

The two searches find different trees. The log-loss search keeps Gini splits and no class weights (55 of 60 trials each) with large leaves (65–253 rows) and moderate depth (2–13), reaching log-loss 0.554–0.567. The F1 search grows deep trees (depth 17–30) with large leaves (78–235 rows), almost no pruning and few columns per split (`max_features` 0.20–0.34), reaching F1 0.742–0.751 at log-loss 0.559–0.589. The earlier search, whose best log-loss trials sat at depth 2, now finds a better tree at depth 8 once the leaf size and the column share can be tuned with it.



In [56]:
tree_candidates = shortlist(study_tree, tree_cache, make_tree)
runs = {"default": tree_runs["decision_tree"], **{name: run for name, (run, _) in tree_candidates.items()}}
chosen_tree, board_tree_tuned = choose_winner(runs, reference="default")
tree_params = tree_candidates[chosen_tree][1] if chosen_tree != "default" else {}

display(board_tree_tuned[["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_tree} {tree_params}")

# Size of the chosen tree in the last fold
chosen_fit = make_tree(tree_params).fit(last["X_train"], last["y_train"])
print(f"Chosen tree in the last fold: depth {chosen_fit.get_depth()}, {chosen_fit.get_n_leaves()} leaves")

,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,
default,0.631,0.000,NaN,NaN,0.000,NaN,NaN,0,14.478,0.597,reference
log_loss_trial_58,0.745,0.114,0.080,0.157,-13.230,-14.767,-11.595,0,0.554,0.773,better
log_loss_trial_35,0.746,0.116,0.084,0.157,-13.224,-14.763,-11.589,0,0.561,0.768,better
log_loss_trial_37,0.736,0.105,0.075,0.139,-13.223,-14.765,-11.582,0,0.562,0.756,better
log_loss_trial_44,0.729,0.098,0.069,0.131,-13.222,-14.763,-11.581,0,0.563,0.752,better
log_loss_trial_0,0.736,0.105,0.075,0.144,-13.218,-14.756,-11.575,0,0.567,0.763,better
f1_trial_20,0.751,0.120,0.089,0.160,-13.219,-14.761,-11.587,0,0.565,0.773,better
f1_trial_24,0.746,0.115,0.083,0.156,-13.204,-14.755,-11.551,0,0.582,0.763,better
f1_trial_40,0.745,0.115,0.084,0.152,-13.222,-14.762,-11.585,0,0.563,0.766,better


Chosen: log_loss_trial_58 {'max_depth': 8, 'min_samples_leaf': 147, 'ccp_alpha': 0.0014465438431017271, 'criterion': 'gini', 'class_weight': 'none', 'max_features': 0.7559846838936468}
Chosen tree in the last fold: depth 5, 11 leaves


All ten candidates are clearly better than the default tree: F1 gains of +0.098 to +0.120, with every interval well above zero, and log-loss from 14.5 to 0.554–0.582. Log-loss trial 58 is chosen as the lowest log-loss: depth up to 8, at least 147 rows per leaf, `ccp_alpha` 0.0014, Gini splits, no class weights and 76% of the columns per split. In the last fold it has depth 5 and 11 leaves, against depth 29 and 790 leaves for the default.



In [57]:
tuned_runs["decision_tree"] = runs[chosen_tree]
default_of["decision_tree"] = tree_runs["decision_tree"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,22.000,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
hist_gb,41.000,0.748,0.001,-0.013,0.021,-0.006,-0.013,0.002,1,0.670,0.790,0.541,0.360,no clear difference
naive_bayes,20.250,0.736,-0.012,-0.027,0.011,0.026,0.014,0.039,4,0.624,0.768,0.573,0.350,worse
lda,20.250,0.743,-0.004,-0.017,0.014,0.017,0.008,0.027,4,0.659,0.780,0.565,0.330,worse
qda,20.250,0.736,-0.011,-0.026,0.008,0.156,0.112,0.203,4,0.635,0.768,0.702,0.140,worse
knn,20.250,0.742,-0.005,-0.020,0.013,0.011,-0.001,0.023,3,0.667,0.780,0.558,0.380,no clear difference
decision_tree,41.000,0.745,-0.003,-0.020,0.022,0.009,-0.003,0.021,1,0.630,0.773,0.554,0.330,no clear difference


### Validation curve: depth

The validation curve varies one setting while the others stay at their chosen values. The dark line is the training score (mean over the four folds), the red line the validation score on the later years. Where the lines move apart, the model is learning the training years rather than patterns that carry over.


In [58]:
tree_curve = plot_validation_curve(tree_cache, make_tree, tree_params or {
    "max_depth": 30, "min_samples_leaf": 1, "ccp_alpha": 0.0, "criterion": "gini", "class_weight": "none",
    "max_features": 1.0}, "max_depth", [1, 2, 3, 4, 6, 8, 12, 20, 30], "Decision tree")

,train_f1,validation_f1,train_log_loss,validation_log_loss
max_depth,,,,
1,0.716,0.709,0.607,0.599
2,0.725,0.739,0.577,0.567
3,0.744,0.737,0.564,0.561
4,0.751,0.745,0.556,0.560
6,0.751,0.745,0.552,0.559
8,0.751,0.745,0.552,0.554
12,0.751,0.745,0.552,0.554
20,0.751,0.745,0.552,0.554
30,0.751,0.745,0.552,0.554


The depth curve shows how little depth the tree needs once its leaves are large. Validation F1 rises from 0.709 at depth 1, the always-repeat floor, to 0.745 at depth 4, and stays there; log-loss reaches its lowest value (0.554) at depth 8. Beyond depth 8 nothing changes, because the 147-row leaves and the pruning stop the tree from growing further. Training and validation stay close at every depth (log-loss 0.552 and 0.554 from depth 8 on), so the tuned tree no longer memorises the training years.

Tuning moves the tree from F1 0.631 to 0.745, the largest gain of any family. Against logistic regression it shows no clear difference (log-loss 0.554 against 0.547, interval [-0.003, 0.021]).



## 4.6. Random forest

The default forest grew 100 unlimited trees with √p columns per split, and it followed the training years closely (log-loss gap 0.435). The search varies the number of trees, the share of columns tried at each split, the minimum leaf size (1–200), the depth limit, the class weights, the share of rows drawn for each tree (30–100%) and the split criterion. Averaging more trees smooths the probabilities, while fewer columns per split make the trees more different from one another.


In [59]:
# Same cached tree-branch matrices as the single tree
forest_cache = tree_cache


def suggest_forest(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "max_features": trial.suggest_float("max_features", 0.1, 1.0),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 200, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "class_weight": trial.suggest_categorical("class_weight", ["none", "balanced", "balanced_subsample"]),
        "max_samples": trial.suggest_float("max_samples", 0.3, 1.0),  # share of rows drawn for each tree
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
    }


def make_forest(params):
    settings = {**params, "class_weight": None if params["class_weight"] == "none" else params["class_weight"]}
    return RandomForestClassifier(**settings, random_state=RANDOM_STATE, n_jobs=-1)


study_forest = {objective: tune(forest_cache, make_forest, suggest_forest, objective, f"random_forest_wide_{objective}")
                for objective in OBJECTIVES}

for objective, study in study_forest.items():
    print(f"\nSteered by {objective}")
    display(trials_table(study).head(5)[["n_estimators", "max_features", "min_samples_leaf", "max_depth",
                                         "class_weight", "max_samples", "criterion", "f1", "log_loss", "roc_auc", "threshold"]])

random_forest_wide_log_loss: 60 trials
random_forest_wide_f1: 60 trials

Steered by log_loss


,n_estimators,max_features,min_samples_leaf,max_depth,class_weight,max_samples,criterion,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,,,
53,150,0.669,39,12,balanced_subsample,0.705,gini,0.751,0.543,0.791,0.380
50,150,0.665,44,9,balanced_subsample,0.885,gini,0.751,0.543,0.790,0.380
55,150,0.610,52,11,balanced_subsample,0.907,gini,0.748,0.544,0.789,0.400
59,200,0.649,53,8,balanced_subsample,0.965,gini,0.752,0.544,0.789,0.380
43,200,0.762,55,6,balanced_subsample,0.924,gini,0.749,0.544,0.789,0.380



Steered by f1


,n_estimators,max_features,min_samples_leaf,max_depth,class_weight,max_samples,criterion,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,,,
42,200,0.227,45,16,balanced,0.944,entropy,0.753,0.545,0.789,0.390
55,350,0.290,2,30,none,0.791,gini,0.752,0.568,0.780,0.410
2,350,0.226,3,13,balanced,0.660,gini,0.752,0.552,0.786,0.390
43,350,0.273,55,11,balanced,0.927,entropy,0.751,0.545,0.789,0.370
22,400,0.339,1,28,none,0.779,gini,0.751,0.573,0.775,0.430


The log-loss search prefers `balanced_subsample` class weights (40 of 60 trials), many columns per split (`max_features` 0.59–0.76), large leaves (32–68 rows), moderate depth (6–12), 150–200 trees, 70–97% of the rows per tree and Gini splits, reaching log-loss 0.543–0.545 and ROC-AUC 0.789–0.791. The F1 search prefers few columns per split (0.23–0.34) and reaches F1 0.751–0.753, at a log-loss of 0.545–0.574.



In [60]:
plot_search(study_forest, "Random forest")

In [61]:
forest_candidates = shortlist(study_forest, forest_cache, make_forest)
runs = {"default": forest_runs["random_forest"], **{name: run for name, (run, _) in forest_candidates.items()}}
chosen_forest, board_forest_tuned = choose_winner(runs, reference="default")
forest_params = forest_candidates[chosen_forest][1] if chosen_forest != "default" else {}

display(board_forest_tuned[["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_forest} {forest_params}")

,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,
default,0.745,0.000,NaN,NaN,0.000,NaN,NaN,0,0.588,0.774,reference
log_loss_trial_53,0.751,0.006,-0.004,0.016,-0.042,-0.076,-0.016,0,0.543,0.791,better
log_loss_trial_50,0.751,0.006,-0.004,0.016,-0.042,-0.076,-0.015,0,0.543,0.790,better
log_loss_trial_55,0.748,0.003,-0.009,0.014,-0.041,-0.076,-0.015,0,0.544,0.789,better
log_loss_trial_59,0.752,0.008,-0.003,0.018,-0.041,-0.076,-0.014,0,0.544,0.789,better
log_loss_trial_43,0.749,0.005,-0.006,0.015,-0.041,-0.076,-0.014,0,0.544,0.789,better
f1_trial_42,0.753,0.008,-0.003,0.018,-0.040,-0.075,-0.014,0,0.545,0.789,better
f1_trial_55,0.752,0.007,0.000,0.016,-0.019,-0.049,-0.003,0,0.568,0.780,better
f1_trial_2,0.752,0.007,-0.002,0.016,-0.034,-0.068,-0.009,0,0.552,0.786,better


Chosen: log_loss_trial_53 {'n_estimators': 150, 'max_features': 0.6694018415498361, 'min_samples_leaf': 39, 'max_depth': 12, 'class_weight': 'balanced_subsample', 'max_samples': 0.7045516429832803, 'criterion': 'gini'}


Nine of the ten candidates pass the rule against the default forest, with log-loss falling from 0.588 to 0.543–0.573. Log-loss trial 53 is chosen as the lowest log-loss: 150 trees, 67% of the columns per split, at least 39 rows per leaf, depth up to 12, `balanced_subsample` weights and 70% of the rows per tree. It scores F1 0.751 and log-loss 0.543 against 0.745 and 0.588 for the default (interval [-0.076, -0.016]), and is worse in no year.



In [62]:
# Number of trees: the chosen settings with more or fewer trees
settings = forest_params or {"max_features": "sqrt", "min_samples_leaf": 1, "max_depth": None, "class_weight": "none"}
tree_count_rows = []
for n_trees in [10, 25, 50, 100, 200, 300, 500]:
    table, oof, threshold = evaluate_cached(forest_cache, make_forest({**settings, "n_estimators": n_trees}))
    tree_count_rows.append({"n_estimators": n_trees, "validation_f1": f1_score(oof["y"], oof["p"] >= threshold),
                            "log_loss": table["log_loss"].mean(), "roc_auc": table["roc_auc"].mean()})
tree_counts = pd.DataFrame(tree_count_rows).set_index("n_estimators")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Pooled validation F1", "Mean validation log-loss"])
for col, metric in enumerate(["validation_f1", "log_loss"], start=1):
    fig.add_trace(go.Scatter(x=tree_counts.index, y=tree_counts[metric], mode="lines+markers", showlegend=False,
                             line=dict(color=ACCENT["burgundy"]),
                             hovertemplate="%{x} trees<br>%{y:.3f}<extra></extra>"), row=1, col=col)
fig.update_xaxes(type="log", title="Number of trees (log scale)", tickvals=list(tree_counts.index))
fig.update_layout(title="Random forest: effect of the number of trees (chosen settings)", height=420)
fig.write_html(FIGURES_DIR / "04_forest_tree_count.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_forest_tree_count.png", **PNG_OPTIONS)
fig.show()

display(tree_counts.round(3).T)

n_estimators,10,25,50,100,200,300,500
validation_f1,0.746,0.748,0.750,0.751,0.748,0.749,0.748
log_loss,0.546,0.542,0.542,0.543,0.544,0.544,0.545
roc_auc,0.787,0.790,0.791,0.791,0.790,0.789,0.789


F1 is almost flat in the number of trees: 0.746–0.751 from 10 to 500 trees. Log-loss is 0.546 with 10 trees, 0.542 with 25–50 and 0.543–0.545 from 100 on; ROC-AUC moves between 0.787 and 0.791. More trees mostly smooth the probabilities; beyond 50 they only add fitting time.



In [63]:
tuned_runs["random_forest"] = runs[chosen_forest]
default_of["random_forest"] = forest_runs["random_forest"]
plot_tuning_progress()

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,22.000,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
hist_gb,41.000,0.748,0.001,-0.013,0.021,-0.006,-0.013,0.002,1,0.670,0.790,0.541,0.360,no clear difference
naive_bayes,20.250,0.736,-0.012,-0.027,0.011,0.026,0.014,0.039,4,0.624,0.768,0.573,0.350,worse
lda,20.250,0.743,-0.004,-0.017,0.014,0.017,0.008,0.027,4,0.659,0.780,0.565,0.330,worse
qda,20.250,0.736,-0.011,-0.026,0.008,0.156,0.112,0.203,4,0.635,0.768,0.702,0.140,worse
knn,20.250,0.742,-0.005,-0.020,0.013,0.011,-0.001,0.023,3,0.667,0.780,0.558,0.380,no clear difference
decision_tree,41.000,0.745,-0.003,-0.020,0.022,0.009,-0.003,0.021,1,0.630,0.773,0.554,0.330,no clear difference
random_forest,41.000,0.751,0.003,-0.009,0.023,-0.004,-0.012,0.003,2,0.681,0.791,0.543,0.380,no clear difference


### Validation curve

The validation curve varies one setting while the others stay at their chosen values. The dark line is the training score (mean over the four folds), the red line the validation score on the later years. Where the lines move apart, the model is learning the training years rather than patterns that carry over.


In [64]:
forest_curve = plot_validation_curve(forest_cache, make_forest, forest_params or {
    "n_estimators": 100, "max_features": "sqrt", "max_depth": None, "class_weight": "none", "max_samples": None,
    "criterion": "gini"}, "min_samples_leaf", [1, 2, 5, 10, 20, 40, 80, 150, 300], "Random forest", log_x=True)

,train_f1,validation_f1,train_log_loss,validation_log_loss
min_samples_leaf,,,,
1,0.938,0.747,0.288,0.571
2,0.928,0.750,0.309,0.564
5,0.889,0.750,0.363,0.563
10,0.851,0.747,0.415,0.550
20,0.811,0.753,0.466,0.547
40,0.787,0.749,0.507,0.544
80,0.761,0.750,0.535,0.546
150,0.751,0.747,0.552,0.551
300,0.741,0.746,0.571,0.561


The leaf-size curve shows the forest's main control on overfitting. With one-row leaves, training F1 is 0.938 against 0.747 on validation, and training log-loss 0.288 against 0.571. Larger leaves close the gap: at 150 rows the two F1 values are 0.751 and 0.747. Validation log-loss is lowest at 40 rows (0.544), close to the chosen 39, and rises again to 0.561 at 300 rows, where the trees become too coarse.

Against logistic regression, the tuned forest shows no clear difference (log-loss 0.543 against 0.547, interval [-0.012, 0.003]), and neither does gradient boosting (0.541, interval [-0.013, 0.002]). The tuned logistic regression from notebook 3 is a harder reference than the default one in section 3.



## 4.7. Logistic regression with splines

Logistic regression draws a straight boundary in its inputs: each extra day since the last purchase changes the log-odds by the same amount. A cubic spline basis replaces each numeric column by a few smooth curve pieces, so the model can bend, for example a risk that rises and then levels off. The model stays a logistic regression, now on a richer set of inputs, and the elastic-net selection keeps only the useful pieces.

Five settings are searched: the notebook 3 log transform (on or off), the number of spline knots (2–6; an earlier search from 3 to 5 placed its best F1 trials at 3), the selector's strength and L1 share, and the final model's regularisation. The family's default uses 4 knots and the saved selector settings. The reference is the tuned logistic regression from notebook 3.


In [65]:
# One cache per (log transform, knots); the selector and the model are fitted per trial
spline_caches = {
    (use_log, knots): FoldCache(build_linear_pipeline(LogisticRegression(), select=False, log=use_log,
                                                      splines=knots).named_steps["prep"], linear_inputs)
    for use_log in (False, True) for knots in (2, 3, 4, 5, 6)
}
get_spline_cache = lambda params: spline_caches[(params["use_log"], params["n_knots"])]


def suggest_spline(trial):
    return {
        "use_log": trial.suggest_categorical("use_log", [False, True]),
        "n_knots": trial.suggest_int("n_knots", 2, 6),
        "select_C": trial.suggest_float("select_C", 1e-3, 1.0, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.05, 0.95),
        "model_C": trial.suggest_float("model_C", 1e-3, 10.0, log=True),
    }


def make_spline(params):
    return Pipeline([
        ("select", make_selector(params["select_C"], params["l1_ratio"])),
        ("model", LogisticRegression(C=params["model_C"], max_iter=5000, random_state=RANDOM_STATE)),
    ])


spline_default = score_folds(build_linear_pipeline(LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
                                                   splines=4), linear_inputs)
study_spline = {objective: tune(get_spline_cache, make_spline, suggest_spline, objective, f"logistic_splines_wide_{objective}")
                for objective in OBJECTIVES}

for objective, study in study_spline.items():
    print(f"\nSteered by {objective}")
    display(trials_table(study).head(5).reindex(
        columns=["use_log", "n_knots", "select_C", "l1_ratio", "model_C", "f1", "log_loss", "roc_auc", "threshold"]
    ).style.format({"select_C": "{:.3g}", "model_C": "{:.3g}"}, precision=3))


logistic_splines_wide_log_loss: 60 trials
logistic_splines_wide_f1: 60 trials

Steered by log_loss


,use_log,n_knots,select_C,l1_ratio,model_C,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,
45,True,5,0.396,0.578,0.056,0.748,0.545,0.787,0.400
16,True,5,0.431,0.760,0.239,0.750,0.545,0.786,0.400
42,True,5,0.276,0.521,0.0393,0.748,0.545,0.786,0.400
41,True,5,0.311,0.398,0.0337,0.748,0.546,0.786,0.400
43,True,5,0.382,0.542,0.0399,0.747,0.546,0.786,0.410



Steered by f1


,use_log,n_knots,select_C,l1_ratio,model_C,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,
22,True,2,0.109,0.403,0.0186,0.753,0.551,0.784,0.440
52,True,2,0.131,0.425,0.0341,0.753,0.551,0.784,0.460
53,True,2,0.154,0.455,0.0513,0.752,0.550,0.784,0.460
56,True,3,0.0977,0.500,0.0456,0.752,0.548,0.785,0.410
55,True,2,0.0971,0.440,0.0772,0.752,0.549,0.784,0.440


In [66]:
plot_search(study_spline, "Logistic regression with splines")

In [67]:
spline_candidates = shortlist(study_spline, get_spline_cache, make_spline)
runs = {"default": spline_default, **{name: run for name, (run, _) in spline_candidates.items()}}
chosen_spline, board_spline_tuned = choose_winner(runs, reference="default")
spline_params = spline_candidates[chosen_spline][1] if chosen_spline != "default" else {}

display(board_spline_tuned[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high",
                            "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_spline} {spline_params}")

tuned_runs["logistic_splines"] = runs[chosen_spline]
default_of["logistic_splines"] = spline_default
plot_tuning_progress()


,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,,
default,32.750,0.745,0.000,NaN,NaN,0.000,NaN,NaN,0,0.553,0.782,reference
log_loss_trial_45,209.250,0.748,0.003,-0.006,0.010,-0.008,-0.015,-0.001,0,0.545,0.787,better
log_loss_trial_16,209.250,0.750,0.005,-0.004,0.014,-0.007,-0.015,0.000,0,0.545,0.786,no clear difference
log_loss_trial_42,209.250,0.748,0.003,-0.006,0.011,-0.008,-0.015,-0.001,0,0.545,0.786,better
log_loss_trial_41,209.250,0.748,0.003,-0.006,0.010,-0.007,-0.015,-0.001,0,0.546,0.786,better
log_loss_trial_43,209.250,0.747,0.002,-0.006,0.009,-0.007,-0.014,-0.001,0,0.546,0.786,better
f1_trial_22,125.250,0.753,0.008,-0.002,0.018,-0.002,-0.011,0.006,0,0.551,0.784,no clear difference
f1_trial_52,125.250,0.753,0.008,-0.003,0.017,-0.002,-0.010,0.005,1,0.551,0.784,no clear difference
f1_trial_53,125.250,0.752,0.007,-0.005,0.016,-0.003,-0.010,0.004,1,0.550,0.784,no clear difference


Chosen: log_loss_trial_42 {'use_log': True, 'n_knots': 5, 'select_C': 0.27570920144274874, 'l1_ratio': 0.5207448455946112, 'model_C': 0.03928423945535104}


,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,22.000,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
hist_gb,41.000,0.748,0.001,-0.013,0.021,-0.006,-0.013,0.002,1,0.670,0.790,0.541,0.360,no clear difference
naive_bayes,20.250,0.736,-0.012,-0.027,0.011,0.026,0.014,0.039,4,0.624,0.768,0.573,0.350,worse
lda,20.250,0.743,-0.004,-0.017,0.014,0.017,0.008,0.027,4,0.659,0.780,0.565,0.330,worse
qda,20.250,0.736,-0.011,-0.026,0.008,0.156,0.112,0.203,4,0.635,0.768,0.702,0.140,worse
knn,20.250,0.742,-0.005,-0.020,0.013,0.011,-0.001,0.023,3,0.667,0.780,0.558,0.380,no clear difference
decision_tree,41.000,0.745,-0.003,-0.020,0.022,0.009,-0.003,0.021,1,0.630,0.773,0.554,0.330,no clear difference
random_forest,41.000,0.751,0.003,-0.009,0.023,-0.004,-0.012,0.003,2,0.681,0.791,0.543,0.380,no clear difference
logistic_splines,209.250,0.748,0.000,-0.011,0.016,-0.002,-0.007,0.003,2,0.679,0.786,0.545,0.400,no clear difference


### Validation curve

The validation curve varies one setting while the others stay at their chosen values. The dark line is the training score (mean over the four folds), the red line the validation score on the later years. Where the lines move apart, the model is learning the training years rather than patterns that carry over.


In [68]:
spline_curve = plot_validation_curve(get_spline_cache, make_spline, spline_params or {
    "use_log": False, "n_knots": 4, "select_C": config["linear"]["selector_C"],
    "l1_ratio": config["linear"]["selector_l1_ratio"], "model_C": 1.0},
    "model_C", np.logspace(-3, 1, 8), "Logistic regression with splines", log_x=True)

,train_f1,validation_f1,train_log_loss,validation_log_loss
model_C,,,,
0.001,0.736,0.741,0.597,0.589
0.004,0.742,0.746,0.568,0.560
0.014,0.749,0.747,0.551,0.549
0.052,0.759,0.748,0.541,0.545
0.193,0.765,0.748,0.535,0.545
0.720,0.766,0.747,0.532,0.547
2.683,0.767,0.749,0.531,0.550
10.000,0.766,0.748,0.531,0.552


The log-loss search uses the log transform (55 of 60 trials) and mostly 5 knots, with weak regularisation of the final model (`model_C` 0.03–0.26), reaching log-loss 0.545–0.547. The F1 search prefers 2 knots (36 of 60 trials) and reaches F1 0.750–0.753, at log-loss 0.548–0.554. Two knots is the smallest spline basis possible, so the F1 search again sits at the edge, but the log-loss search, which the decision rule follows, sits well inside the range.

Four log-loss candidates pass the rule against the default splines (4 knots and saved selector settings). Log-loss trial 42 is chosen: log transform, 5 knots, `select_C` 0.276, `l1_ratio` 0.52 and `model_C` 0.039, with log-loss 0.545 against 0.553 (interval [-0.015, -0.001]) and F1 0.748 against 0.745. It uses 209 columns on average, against 22 for logistic regression, and against it shows no clear difference (log-loss 0.545 against 0.547, interval [-0.007, 0.003]).

The validation curve over `model_C` shows the usual regularisation trade-off. Strong regularisation (0.001) underfits: log-loss 0.597 on training and 0.589 on validation. Validation log-loss is lowest (0.545) between 0.05 and 0.2, where the chosen 0.039 sits. Weaker regularisation keeps lowering the training loss (0.531 at 10) while the validation loss rises to 0.552, so the extra spline pieces begin to fit the training years.



## 4.8. Multilayer perceptron

Seven settings are searched: the input columns, one or two hidden layers of 8–128 units, the L2 penalty `alpha`, the step size `learning_rate_init`, the batch size and the number of passes over the data (20–400). Built-in early stopping holds out a random tenth of the training rows, which would mix the years, so it stays off; the number of passes is tuned instead and plays the same role. The default reference is the default MLP on the selected columns.


In [69]:
class QuietMLP(MLPClassifier):
    """MLP whose pass limit is a tuned setting: stopping before full convergence is intended, not an error."""

    def fit(self, X, y):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            return super().fit(X, y)


get_mlp_cache = lambda params: knn_caches[params["columns"]]  # same prepared linear-branch matrices as KNN


def suggest_mlp(trial):
    return {
        "columns": trial.suggest_categorical("columns", ["all", "selected"]),
        "n_layers": trial.suggest_int("n_layers", 1, 2),
        "units": trial.suggest_int("units", 8, 128, log=True),
        "alpha": trial.suggest_float("alpha", 1e-6, 10.0, log=True),
        "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 3e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128, 256]),
        "max_iter": trial.suggest_int("max_iter", 20, 400, log=True),
    }


def make_mlp(params):
    return QuietMLP(hidden_layer_sizes=(params["units"],) * params["n_layers"], alpha=params["alpha"],
                    learning_rate_init=params["learning_rate_init"], batch_size=params["batch_size"],
                    max_iter=params["max_iter"], random_state=RANDOM_STATE)


study_mlp = {objective: tune(get_mlp_cache, make_mlp, suggest_mlp, objective, f"mlp_{objective}")
             for objective in OBJECTIVES}
for objective, study in study_mlp.items():
    print(f"\nSteered by {objective}")
    display(trials_table(study).head(5).reindex(
        columns=["columns", "n_layers", "units", "alpha", "learning_rate_init", "batch_size", "max_iter",
                 "f1", "log_loss", "roc_auc", "threshold"]).style.format(
        {"alpha": "{:.2g}", "learning_rate_init": "{:.2g}"}, precision=3))


mlp_log_loss: 60 trials
mlp_f1: 60 trials

Steered by log_loss


,columns,n_layers,units,alpha,learning_rate_init,batch_size,max_iter,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,,,
6,selected,1,124,0.25,0.00031,64,201,0.746,0.549,0.788,0.390
59,selected,1,71,0.57,0.00023,64,40,0.746,0.549,0.787,0.430
57,selected,1,82,0.43,0.00033,64,68,0.743,0.550,0.787,0.400
43,selected,1,119,0.91,0.00025,64,39,0.746,0.550,0.786,0.380
48,selected,1,92,0.012,0.00029,64,23,0.746,0.550,0.785,0.360



Steered by f1


,columns,n_layers,units,alpha,learning_rate_init,batch_size,max_iter,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,,,
44,selected,1,122,1.8e-06,0.00039,256,90,0.748,0.552,0.785,0.420
16,selected,2,82,1.2e-05,0.00029,256,62,0.748,0.556,0.780,0.310
22,selected,2,53,0.00096,0.00011,256,96,0.748,0.552,0.787,0.400
57,selected,1,124,1.1e-05,0.00032,256,90,0.747,0.553,0.785,0.360
14,selected,1,88,2.8e-05,0.00016,256,83,0.747,0.553,0.785,0.380


In [70]:
plot_search(study_mlp, "MLP")

In [71]:
mlp_candidates = shortlist(study_mlp, get_mlp_cache, make_mlp)
runs = {"default": mlp_runs["mlp_selected"], **{name: run for name, (run, _) in mlp_candidates.items()}}
chosen_mlp, board_mlp_tuned = choose_winner(runs, reference="default")
mlp_params = mlp_candidates[chosen_mlp][1] if chosen_mlp != "default" else {}

display(board_mlp_tuned[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high",
                         "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_mlp} {mlp_params}")

tuned_runs["mlp"] = runs[chosen_mlp]
default_of["mlp"] = mlp_runs["mlp_selected"]
plot_tuning_progress()


,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,,,,
default,20.250,0.738,0.000,NaN,NaN,0.000,NaN,NaN,0,0.595,0.767,reference
log_loss_trial_6,20.250,0.746,0.008,-0.008,0.020,-0.044,-0.059,-0.030,0,0.549,0.788,better
log_loss_trial_59,20.250,0.746,0.008,-0.013,0.024,-0.043,-0.061,-0.027,0,0.549,0.787,better
log_loss_trial_57,20.250,0.743,0.005,-0.012,0.019,-0.043,-0.060,-0.028,0,0.550,0.787,better
log_loss_trial_43,20.250,0.746,0.008,-0.007,0.021,-0.043,-0.061,-0.026,0,0.550,0.786,better
log_loss_trial_48,20.250,0.746,0.008,-0.007,0.021,-0.043,-0.059,-0.028,0,0.550,0.785,better
f1_trial_44,20.250,0.748,0.010,-0.008,0.024,-0.040,-0.054,-0.027,0,0.552,0.785,better
f1_trial_16,20.250,0.748,0.010,0.000,0.020,-0.036,-0.050,-0.023,0,0.556,0.780,better
f1_trial_22,20.250,0.748,0.010,-0.007,0.024,-0.040,-0.057,-0.025,0,0.552,0.787,better


Chosen: log_loss_trial_6 {'columns': 'selected', 'n_layers': 1, 'units': 124, 'alpha': 0.2545150013091294, 'learning_rate_init': 0.00031062960067335327, 'batch_size': 64, 'max_iter': 201}


,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,22.000,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
hist_gb,41.000,0.748,0.001,-0.013,0.021,-0.006,-0.013,0.002,1,0.670,0.790,0.541,0.360,no clear difference
naive_bayes,20.250,0.736,-0.012,-0.027,0.011,0.026,0.014,0.039,4,0.624,0.768,0.573,0.350,worse
lda,20.250,0.743,-0.004,-0.017,0.014,0.017,0.008,0.027,4,0.659,0.780,0.565,0.330,worse
qda,20.250,0.736,-0.011,-0.026,0.008,0.156,0.112,0.203,4,0.635,0.768,0.702,0.140,worse
knn,20.250,0.742,-0.005,-0.020,0.013,0.011,-0.001,0.023,3,0.667,0.780,0.558,0.380,no clear difference
decision_tree,41.000,0.745,-0.003,-0.020,0.022,0.009,-0.003,0.021,1,0.630,0.773,0.554,0.330,no clear difference
random_forest,41.000,0.751,0.003,-0.009,0.023,-0.004,-0.012,0.003,2,0.681,0.791,0.543,0.380,no clear difference
logistic_splines,209.250,0.748,0.000,-0.011,0.016,-0.002,-0.007,0.003,2,0.679,0.786,0.545,0.400,no clear difference


### Validation curve

The validation curve varies one setting while the others stay at their chosen values. The dark line is the training score (mean over the four folds), the red line the validation score on the later years. Where the lines move apart, the model is learning the training years rather than patterns that carry over.


In [72]:
mlp_curve = plot_validation_curve(get_mlp_cache, make_mlp, mlp_params or {
    "columns": "selected", "n_layers": 1, "units": 100, "learning_rate_init": 1e-3, "batch_size": 128, "max_iter": 200},
    "alpha", np.logspace(-6, 1, 8), "MLP", log_x=True)

,train_f1,validation_f1,train_log_loss,validation_log_loss
alpha,,,,
0.000,0.783,0.741,0.488,0.570
0.000,0.784,0.741,0.488,0.569
0.000,0.776,0.739,0.488,0.570
0.001,0.781,0.740,0.489,0.570
0.010,0.780,0.740,0.491,0.567
0.100,0.765,0.742,0.510,0.555
1.000,0.751,0.744,0.549,0.551
10.000,0.736,0.739,0.598,0.588


The log-loss search settles on one hidden layer (56 of 60 trials) on the selected columns, small batches of 64 rows, slow learning rates (0.0002–0.0004) and a moderate to strong penalty (`alpha` 0.01–0.9), reaching log-loss 0.549–0.550. The F1 search prefers batches of 256 rows and almost no penalty, at F1 0.746–0.748 but log-loss 0.552–0.565. All ten candidates pass the rule against the default MLP. Log-loss trial 6 is chosen: one layer of 124 units, `alpha` 0.25, learning rate 0.0003, batches of 64 and 201 passes. It scores F1 0.746 against 0.738 and log-loss 0.549 against 0.595 (interval [-0.059, -0.030]). Against logistic regression it shows no clear difference (log-loss 0.549 against 0.547, interval [-0.005, 0.009]).

The validation curve over `alpha` shows what the penalty does. With almost no penalty, the network fits the training years more closely than it generalises: training log-loss 0.488 against 0.570 on validation, training F1 0.783 against 0.741. Raising `alpha` to 1 closes the gap (0.549 and 0.551) and gives the lowest validation log-loss. At 10 both losses rise (0.598 and 0.588): the weights are held so small that the network underfits.



## 4.9. Gradient boosting with a broader search

Notebook 3 tuned gradient boosting, and its best trials sat at the edges of the ranges it allowed: the slowest learning rates (near 0.01), the most iterations (near 800) and the smallest trees (near 4 leaves). The optimum may lie beyond those limits. This search widens them: learning rate 0.002–0.3, 50–2,000 iterations, 2–64 leaves per tree, 5–500 rows per leaf, L2 penalty 0.0001–100 and 20–100% of the features per split. Notebook 3's settings are the reference; the broader search replaces them only if it passes the rule.


In [73]:
def suggest_hgb_wide(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.002, 0.3, log=True),
        "max_iter": trial.suggest_int("max_iter", 50, 2000, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 2, 64, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 500, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-4, 100, log=True),
        "max_features": trial.suggest_float("max_features", 0.2, 1.0),
    }


def make_hgb_wide(params):
    return HistGradientBoostingClassifier(**params, categorical_features=list(range(len(branch_roles("tree")["low"]))),
                                          early_stopping=False, random_state=RANDOM_STATE)


study_hgb_wide = {objective: tune(tree_cache, make_hgb_wide, suggest_hgb_wide, objective, f"hist_gb_wide_{objective}")
                  for objective in OBJECTIVES}
for objective, study in study_hgb_wide.items():
    print(f"\nSteered by {objective}")
    display(trials_table(study).head(5).reindex(
        columns=["learning_rate", "max_iter", "max_leaf_nodes", "min_samples_leaf", "l2_regularization",
                 "max_features", "f1", "log_loss", "roc_auc", "threshold"]).style.format(
        {"learning_rate": "{:.3g}", "l2_regularization": "{:.2g}"}, precision=3))


hist_gb_wide_log_loss: 60 trials
hist_gb_wide_f1: 60 trials

Steered by log_loss


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,,
57,0.00372,1452,7,111,0.00064,0.331,0.752,0.540,0.792,0.380
51,0.00266,1671,9,60,0.00016,0.252,0.751,0.542,0.791,0.380
38,0.0085,652,9,54,0.00014,0.307,0.750,0.542,0.791,0.370
41,0.00367,1033,14,81,0.00012,0.203,0.753,0.542,0.792,0.390
49,0.00297,1587,12,69,0.00028,0.392,0.751,0.542,0.791,0.380



Steered by f1


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features,f1,log_loss,roc_auc,threshold
trial,,,,,,,,,,
8,0.031,98,58,173,43,0.916,0.753,0.556,0.790,0.420
22,0.111,51,25,152,1.9,0.971,0.753,0.548,0.785,0.340
54,0.0396,71,21,256,3.7,0.948,0.753,0.547,0.789,0.400
14,0.0144,101,45,212,0.074,0.894,0.752,0.564,0.786,0.440
45,0.0232,91,23,300,0.88,0.971,0.751,0.555,0.788,0.430


In [74]:
plot_search(study_hgb_wide, "Gradient boosting (broader search)")

In [75]:
hgb_candidates = shortlist(study_hgb_wide, tree_cache, make_hgb_wide)
runs = {"notebook 3": tuned_runs["hist_gb"], **{name: run for name, (run, _) in hgb_candidates.items()}}
chosen_hgb, board_hgb_wide = choose_winner(runs, reference="notebook 3")
display(board_hgb_wide[["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high",
                        "folds_worse", "overfit_gap", "log_loss", "roc_auc", "verdict"]])

# decision: the broader search replaces notebook 3's settings only if it passes the rule
if chosen_hgb != "notebook 3":
    hgb_params = dict(hgb_candidates[chosen_hgb][1])
    tuned_runs["hist_gb"] = runs[chosen_hgb]
print(f"Gradient boosting settings: {chosen_hgb} {hgb_params}")
plot_tuning_progress()


,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,overfit_gap,log_loss,roc_auc,verdict
model,,,,,,,,,,,,
notebook 3,0.748,0.000,NaN,NaN,0.000,NaN,NaN,0,0.038,0.541,0.790,reference
log_loss_trial_57,0.752,0.004,-0.002,0.010,-0.001,-0.003,0.002,2,0.037,0.540,0.792,no clear difference
log_loss_trial_51,0.751,0.002,-0.005,0.010,0.001,-0.003,0.004,2,0.045,0.542,0.791,no clear difference
log_loss_trial_38,0.750,0.002,-0.005,0.010,0.001,-0.002,0.004,2,0.058,0.542,0.791,no clear difference
log_loss_trial_41,0.753,0.005,-0.003,0.012,0.001,-0.003,0.005,4,0.058,0.542,0.792,no clear difference
log_loss_trial_49,0.751,0.003,-0.005,0.010,0.001,-0.002,0.005,4,0.069,0.542,0.791,no clear difference
f1_trial_8,0.753,0.005,-0.003,0.014,0.016,0.004,0.027,4,0.025,0.556,0.790,worse
f1_trial_22,0.753,0.004,-0.004,0.015,0.008,0.003,0.012,4,0.102,0.548,0.785,worse
f1_trial_54,0.753,0.004,-0.003,0.013,0.007,-0.000,0.013,4,0.025,0.547,0.789,no clear difference


Gradient boosting settings: notebook 3 {'learning_rate': 0.015142507125324717, 'max_iter': 555, 'max_leaf_nodes': 5, 'min_samples_leaf': 185, 'l2_regularization': 0.2208507098160641, 'max_features': 0.7176387213680354}


,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,22.000,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
hist_gb,41.000,0.748,0.001,-0.013,0.021,-0.006,-0.013,0.002,1,0.670,0.790,0.541,0.360,no clear difference
naive_bayes,20.250,0.736,-0.012,-0.027,0.011,0.026,0.014,0.039,4,0.624,0.768,0.573,0.350,worse
lda,20.250,0.743,-0.004,-0.017,0.014,0.017,0.008,0.027,4,0.659,0.780,0.565,0.330,worse
qda,20.250,0.736,-0.011,-0.026,0.008,0.156,0.112,0.203,4,0.635,0.768,0.702,0.140,worse
knn,20.250,0.742,-0.005,-0.020,0.013,0.011,-0.001,0.023,3,0.667,0.780,0.558,0.380,no clear difference
decision_tree,41.000,0.745,-0.003,-0.020,0.022,0.009,-0.003,0.021,1,0.630,0.773,0.554,0.330,no clear difference
random_forest,41.000,0.751,0.003,-0.009,0.023,-0.004,-0.012,0.003,2,0.681,0.791,0.543,0.380,no clear difference
logistic_splines,209.250,0.748,0.000,-0.011,0.016,-0.002,-0.007,0.003,2,0.679,0.786,0.545,0.400,no clear difference


### Validation curve

The validation curve varies one setting while the others stay at their chosen values. The dark line is the training score (mean over the four folds), the red line the validation score on the later years. Where the lines move apart, the model is learning the training years rather than patterns that carry over.


In [76]:
hgb_curve = plot_validation_curve(tree_cache, make_hgb_wide, hgb_params, "max_iter",
                                  [50, 100, 200, 400, 800, 1200, 2000], "Gradient boosting", log_x=True)

,train_f1,validation_f1,train_log_loss,validation_log_loss
max_iter,,,,
50,0.746,0.742,0.597,0.601
100,0.756,0.748,0.564,0.570
200,0.766,0.750,0.537,0.548
400,0.776,0.751,0.514,0.541
800,0.788,0.752,0.490,0.542
1200,0.794,0.753,0.471,0.545
2000,0.808,0.745,0.443,0.555


The broader search does not find better settings. Its best log-loss trial (57: learning rate 0.0037, 1,452 iterations, 7 leaves, 33% of the features) scores log-loss 0.540 against 0.541 for notebook 3's settings, with no clear difference (interval [-0.003, 0.002]). The log-loss search stays in the same region as notebook 3: slow learning (0.002–0.009), many iterations (650–1,700) and small trees (7–16 leaves). The F1 search mostly finds fast, short models (51–138 iterations, 20–58 leaves) with F1 up to 0.753, but four of its five candidates are worse on log-loss. Notebook 3's settings are kept. Its earlier "edges" were therefore the edges of a flat region rather than signs of a better optimum outside the range.

The validation curve over `max_iter` (learning rate 0.015, 5 leaves) shows the trade-off that those settings balance. Validation log-loss falls from 0.601 at 50 iterations to 0.541 at 400 and stays at 0.542 at 800, close to the chosen 555. Training log-loss keeps falling (0.443 at 2,000 iterations), while validation log-loss rises to 0.555 and validation F1 drops from 0.753 at 1,200 to 0.745. Each extra tree beyond about 800 fits the training years more than the later ones.



## 4.10. Gradient boosting with monotonic constraints

A monotonic constraint forces the model's prediction to move in one direction with an input, all else equal. Here the constraint says that a higher value can only raise the chance of a repeat, for six inputs where that direction is the business expectation: the purchase-process probability, purchases in the last 30, 90, 180 and 365 days, and purchases per year. Days since the previous purchase is left free: a customer can be "due" or "lost", so its effect need not move in one direction.

The settings are those chosen in section 4.9; only the constraints are added. A constrained model cannot learn a reversed pattern that exists only in the training years, so it should be at least as stable on later data, and each constrained input can be explained in one sentence. When the rule finds no clear difference, the constrained version is preferred for those two reasons.


In [77]:
# Upward constraints: the business direction is clear for these inputs
MONOTONIC_UP = ["nbd_repeat_prob", "purchase_count_30d", "purchase_count_90d", "purchase_count_180d",
                "purchase_count_365d", "purchases_per_year"]

# Column order of the tree branch: small categories, large categories, numbers, then the fitted probability
r_tree = branch_roles("tree")
tree_columns = r_tree["low"] + r_tree["high"] + r_tree["num"] + (["nbd_repeat_prob"] if NBD_INPUTS else [])
missing = [c for c in MONOTONIC_UP if c not in tree_columns]
assert not missing, f"Constrained inputs not in the tree branch: {missing}"
HGB_MONOTONIC = [1 if c in MONOTONIC_UP else 0 for c in tree_columns]  # 1 = can only increase, 0 = free

hgb_monotonic_run = score_folds(
    build_tree_pipeline(HistGradientBoostingClassifier(**hgb_params, monotonic_cst=HGB_MONOTONIC,
                                                       random_state=RANDOM_STATE)), tree_inputs)
board_mono = judge({"unconstrained": tuned_runs["hist_gb"], "monotonic": hgb_monotonic_run}, reference="unconstrained")
display(board_mono[["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse",
                    "overfit_gap", "roc_auc", "log_loss", "verdict"]])

# decision: keep the constraints unless they are clearly worse (simpler to explain, safer on later years)
use_monotonic = board_mono.loc["monotonic", "verdict"] != "worse"
if use_monotonic:
    tuned_runs["hist_gb"] = hgb_monotonic_run
print(f"Monotonic constraints kept: {use_monotonic}")


,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,overfit_gap,roc_auc,log_loss,verdict
model,,,,,,,,,,,,
unconstrained,0.748,0.000,NaN,NaN,0.000,NaN,NaN,0,0.038,0.790,0.541,reference
monotonic,0.751,0.003,-0.001,0.007,0.000,-0.001,0.001,3,0.037,0.791,0.541,no clear difference


Monotonic constraints kept: True


The constraints change almost nothing: F1 0.751 against 0.748 and the same log-loss (0.541, interval of the difference [-0.001, 0.001]), with the same training–validation gap (0.037 against 0.038). The log-loss is marginally higher in three of the four years, but by too little to count as worse. Under the preference set out above, the constrained version is kept: the six constrained inputs now always push towards a repeat, at no measurable cost.



## 4.11. Seed averaging

The random forest and gradient boosting use randomness: the forest samples rows and columns, and boosting samples columns at each split. Another random seed gives a slightly different model with the same settings. Averaging the probabilities of 10 seeds removes most of that noise and keeps the signal, much like averaging the answers of ten equally good analysts.

Each tuned model is refitted with seeds 0–9 and its probabilities averaged. The averaged version is judged against the single-seed version. Averaging cannot add a pattern that the single model lacks, so when no clear difference is found the averaged version is still preferred for its steadier probabilities.


In [78]:
class SeedAverage(ClassifierMixin, BaseEstimator):
    """Mean probability of the same model fitted with several random seeds."""

    def __init__(self, estimator=None, n_seeds=10):
        self.estimator = estimator
        self.n_seeds = n_seeds

    def fit(self, X, y):
        self.models_ = [clone(self.estimator).set_params(random_state=seed).fit(X, y) for seed in range(self.n_seeds)]
        self.classes_ = self.models_[0].classes_
        return self

    def predict_proba(self, X):
        return np.mean([model.predict_proba(X) for model in self.models_], axis=0)


N_SEEDS = 10


def seed_averaged(name):
    """Tree pipeline of a tuned family with its model replaced by a 10-seed average."""
    pipe = family_tree_pipe(name)
    pipe.steps[-1] = ("model", SeedAverage(pipe.named_steps["model"], n_seeds=N_SEEDS))
    return pipe


def family_tree_pipe(name):
    """Tuned tree pipeline of the forest or of gradient boosting (with constraints if kept)."""
    if name == "random_forest":
        return build_tree_pipeline(make_forest(forest_params) if forest_params
                                   else RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
    extra = {"monotonic_cst": HGB_MONOTONIC} if use_monotonic else {}
    return build_tree_pipeline(HistGradientBoostingClassifier(**hgb_params, **extra, random_state=RANDOM_STATE))


seed_boards, use_seed_average = {}, {}
for name in ["random_forest", "hist_gb"]:
    averaged = score_folds(seed_averaged(name), tree_inputs)
    board = judge({"single seed": tuned_runs[name], f"{N_SEEDS} seeds": averaged}, reference="single seed")
    seed_boards[name] = board
    # decision: keep the average unless it is clearly worse
    use_seed_average[name] = board.loc[f"{N_SEEDS} seeds", "verdict"] != "worse"
    if use_seed_average[name]:
        tuned_runs[name] = averaged

display(pd.concat(seed_boards)[["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high",
                                "folds_worse", "roc_auc", "log_loss", "verdict"]])
print("Seed averaging kept:", use_seed_average)
plot_tuning_progress()


f1  f1_diff  ci_low  ci_high  ll_diff  ll_ci_low  \
              model                                                             
random_forest single seed 0.751    0.000     NaN      NaN    0.000        NaN   
              10 seeds    0.747   -0.003  -0.008    0.001    0.002      0.000   
hist_gb       single seed 0.751    0.000     NaN      NaN    0.000        NaN   
              10 seeds    0.748   -0.003  -0.007    0.000    0.000     -0.000   

                           ll_ci_high  folds_worse  roc_auc  log_loss  \
              model                                                     
random_forest single seed         NaN            0    0.791     0.543   
              10 seeds          0.003            4    0.790     0.545   
hist_gb       single seed         NaN            0    0.791     0.541   
              10 seeds          0.001            2    0.790     0.541   

                                       verdict  
              model                             
random_forest single seed            reference  
              10 seeds                   worse  
hist_gb       single seed            reference  
              10 seeds     no clear difference

Seed averaging kept: {'random_forest': False, 'hist_gb': True}


,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
logistic,22.000,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
hist_gb,41.000,0.748,0.001,-0.013,0.019,-0.006,-0.013,0.002,1,0.678,0.790,0.541,0.380,no clear difference
naive_bayes,20.250,0.736,-0.012,-0.027,0.011,0.026,0.014,0.039,4,0.624,0.768,0.573,0.350,worse
lda,20.250,0.743,-0.004,-0.017,0.014,0.017,0.008,0.027,4,0.659,0.780,0.565,0.330,worse
qda,20.250,0.736,-0.011,-0.026,0.008,0.156,0.112,0.203,4,0.635,0.768,0.702,0.140,worse
knn,20.250,0.742,-0.005,-0.020,0.013,0.011,-0.001,0.023,3,0.667,0.780,0.558,0.380,no clear difference
decision_tree,41.000,0.745,-0.003,-0.020,0.022,0.009,-0.003,0.021,1,0.630,0.773,0.554,0.330,no clear difference
random_forest,41.000,0.751,0.003,-0.009,0.023,-0.004,-0.012,0.003,2,0.681,0.791,0.543,0.380,no clear difference
logistic_splines,209.250,0.748,0.000,-0.011,0.016,-0.002,-0.007,0.003,2,0.679,0.786,0.545,0.400,no clear difference


Seed averaging gives different results for the two models. For gradient boosting it makes no clear difference (log-loss 0.541 in both, F1 0.748 against 0.751), and the 10-seed average is kept for its steadier probabilities. For the random forest it is worse: log-loss is higher by 0.002 (interval [0.000, 0.003]) and higher in all four years, so the single-seed forest is kept. A forest is already an average of 150 trees, so averaging 10 forests adds little and here shifts the probabilities slightly away from the best ones.



## Findings

| Family | Default F1 | Tuned F1 | Tuned log-loss | Tuning kept? | Against logistic regression |
|---|---|---|---|---|---|
| Logistic regression | 0.749 | 0.747 | 0.547 | yes (notebook 3) | reference |
| Gradient boosting | 0.744 | 0.748 | 0.541 | notebook 3 settings, monotonic, 10 seeds | no clear difference |
| Random forest | 0.745 | 0.751 | 0.543 | yes, single seed | no clear difference |
| Logistic regression with splines | 0.745 | 0.748 | 0.545 | yes | no clear difference |
| MLP | 0.738 | 0.746 | 0.549 | yes | no clear difference |
| Decision tree | 0.631 | 0.745 | 0.554 | yes | no clear difference |
| KNN | 0.731 | 0.742 | 0.558 | yes | no clear difference |
| LDA | 0.745 | 0.743 | 0.565 | yes | worse |
| Naive Bayes | 0.687 | 0.736 | 0.573 | yes | worse |
| QDA | 0.700 | 0.736 | 0.702 | yes | worse |

- Tuning helped every family, most where the default overfitted or was overconfident. The tree gained +0.114 F1 by limiting leaves and depth, KNN, Naive Bayes and QDA repaired their probabilities, and the forest and the MLP gained through larger leaves and a weight penalty.
- No single tuned model is clearly better than the tuned logistic regression. Gradient boosting, the random forest and the splines have slightly lower log-loss (0.541–0.545 against 0.547), but every interval includes zero.
- The wider ranges paid off for KNN (log-loss 0.558, its best values now well inside the range) and the decision tree (a depth-8 tree beats the earlier depth-2 one). Naive Bayes and QDA still sit at the top of their ranges, but their scores have flattened there. For gradient boosting the broader search confirmed notebook 3's settings.
- The validation curves show two kinds of fix. Naive Bayes, LDA and QDA mostly needed less bias. The forest, the MLP, the splines and boosting needed limits on how closely they follow the training years.
- The models reach similar F1 in different ways: a linear boundary, boosted small trees, averaged deep trees, a neural network and local neighbourhoods. That makes them candidates for blends.



# 5. Scoreboard and model choice

## 5.1. Final scoreboard

Every family enters with its chosen settings and is judged against the tuned logistic regression. Finalists are logistic regression and the models that are not clearly worse than it. No single model passes the rule against it, so any improvement has to come from combining models (sections 5.2–5.3).



In [79]:
board_final = judge({"dummy": reference_runs["dummy"], **tuned_runs}, reference="logistic").sort_values("f1", ascending=False)
board_final.loc["dummy", "n_features"] = 0
display(board_final[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse",
                     "balanced_accuracy", "roc_auc", "log_loss", "threshold", "verdict"]])

# decision: finalists are the models not clearly worse than logistic regression (including those that pass)
finalists = board_final.index[board_final["verdict"].isin(["reference", "no clear difference", "better"])].tolist()
print("Finalists:", finalists)

,n_features,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,,
random_forest,41.000,0.751,0.003,-0.009,0.023,-0.004,-0.012,0.003,2,0.681,0.791,0.543,0.380,no clear difference
hist_gb,41.000,0.748,0.001,-0.013,0.019,-0.006,-0.013,0.002,1,0.678,0.790,0.541,0.380,no clear difference
logistic_splines,209.250,0.748,0.000,-0.011,0.016,-0.002,-0.007,0.003,2,0.679,0.786,0.545,0.400,no clear difference
logistic,22.000,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
mlp,20.250,0.746,-0.002,-0.014,0.014,0.002,-0.005,0.009,3,0.677,0.788,0.549,0.390,no clear difference
decision_tree,41.000,0.745,-0.003,-0.020,0.022,0.009,-0.003,0.021,1,0.630,0.773,0.554,0.330,no clear difference
lda,20.250,0.743,-0.004,-0.017,0.014,0.017,0.008,0.027,4,0.659,0.780,0.565,0.330,worse
knn,20.250,0.742,-0.005,-0.020,0.013,0.011,-0.001,0.023,3,0.667,0.780,0.558,0.380,no clear difference
qda,20.250,0.736,-0.011,-0.026,0.008,0.156,0.112,0.203,4,0.635,0.768,0.702,0.140,worse


Finalists: ['random_forest', 'hist_gb', 'logistic_splines', 'logistic', 'mlp', 'decision_tree', 'knn']


No tuned model passes the rule against logistic regression. Six are not clearly different and join it as finalists: the random forest (log-loss 0.543, F1 0.751), gradient boosting (0.541, 0.748), the splines (0.545, 0.748), the MLP (0.549, 0.746), the decision tree (0.554, 0.745) and KNN (0.558, 0.742). LDA, Naive Bayes and QDA are worse. Gradient boosting, the random forest, the MLP and the splines rank the purchases better than logistic regression (ROC-AUC 0.786–0.791 against 0.784). Apart from QDA, Naive Bayes and the dummy, all F1 values lie within 0.009 of one another, so the differences show up in the probabilities, not in the labels.



In [80]:
fold_f1_final = pd.DataFrame({name: tuned_runs[name][0]["f1"] for name in finalists})
fold_f1_final["always repeat"] = [f1_floor(part["y"]) for _, part in tuned_runs["logistic"][1].groupby("fold")]

fig = go.Figure()
for name in fold_f1_final:
    floor = name == "always repeat"
    fig.add_trace(go.Scatter(
        x=fold_f1_final.index, y=fold_f1_final[name], mode="lines" if floor else "lines+markers", name=name,
        line=dict(dash="dash", color="grey") if floor else None,
        hovertemplate="%{x}: F1 %{y:.3f}<extra>" + name + "</extra>",
    ))
fig.update_layout(
    title="Finalists: F1 in each validation year (tuned settings)",
    xaxis=dict(title="Validation year", tickmode="array", tickvals=VAL_YEARS),
    yaxis_title="F1 at each model's pooled threshold",
    height=430,
)
fig.write_html(FIGURES_DIR / "04_finalists_f1_by_year.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_finalists_f1_by_year.png", **PNG_OPTIONS)
fig.show()

display(fold_f1_final.round(3))

,random_forest,hist_gb,logistic_splines,logistic,mlp,decision_tree,knn,always repeat
fold,,,,,,,,
2020,0.743,0.747,0.752,0.748,0.755,0.765,0.755,0.745
2021,0.770,0.775,0.767,0.770,0.760,0.744,0.740,0.709
2022,0.723,0.697,0.712,0.712,0.708,0.703,0.705,0.649
2023,0.764,0.768,0.754,0.751,0.755,0.763,0.767,0.728


No finalist leads in every year. The decision tree is the best in 2020 (0.765) but among the weakest in 2021 (0.744). Gradient boosting is the best in 2021 (0.775) but the weakest in 2022 (0.697). The random forest is the best in 2022 (0.723). All finalists fall to their lowest F1 in 2022, the year with the lowest floor (0.649).



## 5.2. Blends

A blend averages the probabilities of several models (soft voting). It can only help where the members disagree, so their agreement is checked first.

In [81]:
# How similar are the finalists' predictions? Blends gain only where models disagree.
probabilities = pd.DataFrame({name: tuned_runs[name][1]["p"].to_numpy() for name in finalists})
labels = pd.DataFrame({name: tuned_runs[name][1]["p"].to_numpy() >= tuned_runs[name][2] for name in finalists})
agreement = pd.DataFrame(
    [[(labels[a] == labels[b]).mean() for b in finalists] for a in finalists], index=finalists, columns=finalists
)
correlation = probabilities.corr(method="spearman")

fig = go.Figure(go.Heatmap(
    z=correlation.to_numpy(), x=finalists, y=finalists, zmin=-1, zmax=1, colorscale=DIVERGING,
    text=correlation.round(2).to_numpy(), texttemplate="%{text}",
    hovertemplate="%{y} vs %{x}<br>rank correlation %{z:.3f}<extra></extra>",
))
fig.update_layout(title="Finalists: rank correlation of out-of-fold probabilities", height=430)
fig.write_html(FIGURES_DIR / "04_finalists_correlation.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_finalists_correlation.png", **PNG_OPTIONS)
fig.show()

print("Share of rows with the same predicted label:")
display(agreement.round(3))

Share of rows with the same predicted label:


,random_forest,hist_gb,logistic_splines,logistic,mlp,decision_tree,knn
random_forest,1.000,0.917,0.902,0.863,0.904,0.836,0.874
hist_gb,0.917,1.000,0.890,0.853,0.888,0.841,0.865
logistic_splines,0.902,0.890,1.000,0.876,0.916,0.822,0.876
logistic,0.863,0.853,0.876,1.000,0.869,0.735,0.837
mlp,0.904,0.888,0.916,0.869,1.000,0.826,0.897
decision_tree,0.836,0.841,0.822,0.735,0.826,1.000,0.844
knn,0.874,0.865,0.876,0.837,0.897,0.844,1.000


The tree ensembles, the splines and the MLP agree on 89–92% of the labels: the random forest and gradient boosting on 91.7%, the splines and the MLP on 91.6%. Logistic regression is more different from all of them (83.7–87.6%, leaving out the tree). The decision tree is the most different model, with 73.5% agreement with logistic regression and 82.2–84.4% with the others. Blending models that disagree more has more room to help.



In [82]:
def blend(names, weights=None):
    """Soft vote: weighted mean of the members' probabilities, with its own pooled threshold."""
    weights = np.ones(len(names)) / len(names) if weights is None else np.asarray(weights) / np.sum(weights)
    base = tuned_runs[names[0]][1]
    for name in names[1:]:
        assert tuned_runs[name][1]["ID"].equals(base["ID"]), "Members must be scored on the same rows."

    oof = base[["fold", "ID", "customer_id", "seen", "y"]].copy()
    oof["p"] = np.column_stack([tuned_runs[name][1]["p"] for name in names]) @ weights
    threshold, _, _ = best_threshold(oof["y"], oof["p"])

    rows = []
    for year, part in oof.groupby("fold"):
        rows.append({
            "fold": year,
            "n_features": sum(tuned_runs[name][0].loc[year, "n_features"] for name in names),  # total member inputs
            "log_loss": log_loss(part["y"], part["p"], labels=[0, 1]),
            "train_log_loss": np.nan,  # members were fitted separately
            "roc_auc": roc_auc_score(part["y"], part["p"]),
            "fit_seconds": sum(tuned_runs[name][0].loc[year, "fit_seconds"] for name in names),
            "f1": f1_score(part["y"], part["p"] >= threshold),
        })
    return pd.DataFrame(rows).set_index("fold"), oof, threshold


# Illustrative equal-weight blends, picked from the agreement table; section 5.3 chooses members by a rule
blend_members = {
    "logistic + forest": ["logistic", "random_forest"],
    "logistic + boosting": ["logistic", "hist_gb"],
    "logistic + knn": ["logistic", "knn"],
    "logistic + forest + boosting": ["logistic", "random_forest", "hist_gb"],
    "all finalists": finalists,
}
blend_runs = {name: blend(members) for name, members in blend_members.items()}

board_blend = judge({"logistic": tuned_runs["logistic"], **blend_runs}, reference="logistic")
display(board_blend[["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse",
                     "balanced_accuracy", "roc_auc", "log_loss", "threshold", "verdict"]])

,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,,
logistic,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.719,0.784,0.547,0.460,reference
logistic + forest,0.753,0.005,-0.004,0.021,-0.006,-0.010,-0.002,0,0.692,0.791,0.541,0.390,better
logistic + boosting,0.750,0.002,-0.008,0.017,-0.007,-0.011,-0.003,0,0.688,0.791,0.540,0.390,better
logistic + knn,0.745,-0.002,-0.014,0.016,-0.000,-0.007,0.006,3,0.662,0.786,0.547,0.360,no clear difference
logistic + forest + boosting,0.751,0.004,-0.008,0.021,-0.008,-0.013,-0.003,0,0.678,0.792,0.539,0.370,better
all finalists,0.753,0.006,-0.006,0.025,-0.007,-0.013,-0.002,1,0.678,0.793,0.540,0.380,better


Four of the five blends pass the rule against logistic regression: log-loss falls from 0.547 to 0.539–0.541, every interval lies below zero, and at most one year is worse. Logistic regression + KNN shows no clear difference (log-loss 0.547). ROC-AUC rises to 0.786–0.793. F1 moves little (0.745–0.753), with intervals that include zero.

The best probabilities come from logistic regression + forest + boosting (log-loss 0.539, ROC-AUC 0.792). The blend of all seven finalists has the highest ROC-AUC (0.793) and F1 (0.753, tied with logistic regression + forest) at log-loss 0.540.



## 5.3. Ensemble selection

The blends above were chosen by hand from the agreement table, so they show what averaging can do but not which members to use. Here the members are chosen by a rule. Every tuned family enters, and two methods learn how much each one counts:

- **Ensemble selection (hill-climbing):** start with no models and repeatedly add, with replacement, the member that lowers log-loss most. A member added three times gets three votes; members that never help get weight zero. Weights are never negative and stay coarse, which limits overfitting.
- **Stacking:** a logistic regression learns how to combine the members' log-odds. It is regularised (C = 0.1) because it learns from a few thousand rows.

Weights learned on the same rows they are scored on would look better than they are. Each validation year is combined with weights learned on the earlier years only: 2021 with weights from 2020, 2022 from 2020–2021, and 2023 from 2020–2022. 2020 has no earlier year, so it uses equal weights. The weights chosen for each year are shown, to check that the selection is stable. The resulting predictions are judged against logistic regression and against the equal-weight average of all members.


In [83]:
# Every tuned family is a candidate member; the methods decide which ones count
ENSEMBLE_MEMBERS = list(tuned_runs)
base_oof = tuned_runs[ENSEMBLE_MEMBERS[0]][1]
for name in ENSEMBLE_MEMBERS[1:]:
    assert tuned_runs[name][1]["ID"].equals(base_oof["ID"]), "Members must be scored on the same rows."
P_oof = np.column_stack([tuned_runs[name][1]["p"] for name in ENSEMBLE_MEMBERS])
y_oof = base_oof["y"].to_numpy()
fold_oof = base_oof["fold"].to_numpy()


def logit(P):
    P = np.clip(P, 1e-6, 1 - 1e-6)
    return np.log(P / (1 - P))


def hill_climb(P, y, rounds=50):
    """Ensemble selection with replacement; returns weights that sum to 1 at the best round."""
    counts, best_counts, best_loss = np.zeros(P.shape[1]), None, np.inf
    total = np.zeros(len(y))
    for _ in range(rounds):
        losses = [log_loss(y, (total + P[:, j]) / (counts.sum() + 1), labels=[0, 1]) for j in range(P.shape[1])]
        j = int(np.argmin(losses))
        counts[j] += 1
        total += P[:, j]
        if losses[j] < best_loss - 1e-6:
            best_loss, best_counts = losses[j], counts.copy()
    return best_counts / best_counts.sum()


def fit_stack(P, y):
    return LogisticRegression(C=0.1, max_iter=5000).fit(logit(P), y)


year_weights = {}
COMBINERS = {}  # fitted combiners for the final model, keyed by ensemble name


def chronological_ensemble(kind):
    """Each year combined with weights learned on earlier years only; 2020 uses equal weights."""
    p = P_oof.mean(axis=1)
    for year in VAL_YEARS[1:]:
        past, now = fold_oof < year, fold_oof == year
        if kind == "hill_climb":
            weights = hill_climb(P_oof[past], y_oof[past])
            year_weights[year] = weights
            p[now] = P_oof[now] @ weights
        else:
            p[now] = fit_stack(P_oof[past], y_oof[past]).predict_proba(logit(P_oof[now]))[:, 1]
    return p


def run_from_oof(p, n_features):
    """Same output as score_folds for a combined prediction."""
    oof = base_oof[["fold", "ID", "customer_id", "seen", "y"]].copy()
    oof["p"] = p
    threshold, _, _ = best_threshold(oof["y"], oof["p"])
    rows = [{"fold": year, "n_features": n_features,
             "log_loss": log_loss(part["y"], part["p"], labels=[0, 1]), "train_log_loss": np.nan,
             "roc_auc": roc_auc_score(part["y"], part["p"]), "fit_seconds": np.nan,
             "f1": f1_score(part["y"], part["p"] >= threshold)} for year, part in oof.groupby("fold")]
    return pd.DataFrame(rows).set_index("fold"), oof, threshold


n_members = sum(tuned_runs[name][0]["n_features"].mean() for name in ENSEMBLE_MEMBERS)
weighted_runs = {
    "equal weights (all)": blend(ENSEMBLE_MEMBERS),
    "hill-climbing (all)": run_from_oof(chronological_ensemble("hill_climb"), n_members),
    "stacking (all)": run_from_oof(chronological_ensemble("stack"), n_members),
}

# Weights for the final model come from all four development years
final_weights = hill_climb(P_oof, y_oof)
final_stack = fit_stack(P_oof, y_oof)
year_weights["all years (final)"] = final_weights
print("Hill-climbing weights: earlier years for each validation year, and all years for the final model")
display(pd.DataFrame(year_weights, index=ENSEMBLE_MEMBERS).round(3))
print("Stacking coefficients on log-odds (all years):", dict(zip(ENSEMBLE_MEMBERS, final_stack.coef_[0].round(3))))

cols = ["f1", "f1_diff", "ci_low", "ci_high", "ll_diff", "ll_ci_low", "ll_ci_high", "folds_worse",
        "roc_auc", "log_loss", "threshold", "verdict"]
print("\nAgainst logistic regression")
display(judge({"logistic": tuned_runs["logistic"], **weighted_runs}, reference="logistic")[cols])
print("Against the equal-weight average of all members")
display(judge(weighted_runs, reference="equal weights (all)")[cols])

# The ensembles join the blends for the model choice; hill-climbing keeps only members with weight above zero
kept = [name for name, w in zip(ENSEMBLE_MEMBERS, final_weights) if w > 0]
kept_weights = final_weights[final_weights > 0]
COMBINERS["hill-climbing (all)"] = lambda P: P @ kept_weights
COMBINERS["stacking (all)"] = lambda P: final_stack.predict_proba(logit(P))[:, 1]
for name, run in weighted_runs.items():
    blend_runs[name] = run
    blend_members[name] = kept if name.startswith("hill") else ENSEMBLE_MEMBERS
board_blend = judge({"logistic": tuned_runs["logistic"], **blend_runs}, reference="logistic")


Hill-climbing weights: earlier years for each validation year, and all years for the final model


,2021,2022,2023,all years (final)
logistic,0.000,0.094,0.069,0.194
hist_gb,0.176,0.312,0.207,0.389
naive_bayes,0.000,0.000,0.000,0.000
lda,0.000,0.000,0.000,0.000
qda,0.000,0.000,0.000,0.000
knn,0.353,0.000,0.000,0.000
decision_tree,0.176,0.094,0.103,0.194
random_forest,0.000,0.062,0.379,0.139
logistic_splines,0.000,0.094,0.103,0.000
mlp,0.294,0.344,0.138,0.083


Stacking coefficients on log-odds (all years): {'logistic': np.float64(0.243), 'hist_gb': np.float64(0.244), 'naive_bayes': np.float64(0.118), 'lda': np.float64(-0.013), 'qda': np.float64(-0.131), 'knn': np.float64(0.066), 'decision_tree': np.float64(0.241), 'random_forest': np.float64(0.249), 'logistic_splines': np.float64(0.095), 'mlp': np.float64(0.132)}

Against logistic regression


,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,
logistic,0.747,0.000,NaN,NaN,0.000,NaN,NaN,0,0.784,0.547,0.460,reference
equal weights (all),0.748,0.001,-0.011,0.017,-0.004,-0.009,0.002,2,0.790,0.543,0.370,no clear difference
hill-climbing (all),0.750,0.002,-0.009,0.019,-0.003,-0.009,0.003,1,0.791,0.543,0.390,no clear difference
stacking (all),0.747,-0.000,-0.012,0.017,-0.001,-0.007,0.006,1,0.790,0.546,0.400,no clear difference


Against the equal-weight average of all members


,f1,f1_diff,ci_low,ci_high,ll_diff,ll_ci_low,ll_ci_high,folds_worse,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,,
equal weights (all),0.748,0.000,NaN,NaN,0.000,NaN,NaN,0,0.790,0.543,0.370,reference
hill-climbing (all),0.750,0.001,-0.006,0.009,0.000,-0.003,0.003,1,0.791,0.543,0.390,no clear difference
stacking (all),0.747,-0.001,-0.010,0.007,0.003,-0.002,0.008,2,0.790,0.546,0.400,no clear difference


Neither weighting method improves on equal weights, and none of the three all-member ensembles passes the rule against logistic regression. Equal weights reach log-loss 0.543, hill-climbing 0.543 and stacking 0.546, all with intervals that include zero. Against the equal-weight average, the learned weights show no clear difference either.

The year-by-year weights explain why. Hill-climbing gives KNN the largest weight in 2021 (0.353) and zero afterwards. The random forest goes from zero in 2021 to 0.379 in 2023. Only gradient boosting (0.176–0.312), the MLP (0.138–0.344) and the decision tree (0.094–0.176) are chosen every year. With four years of history the weights follow each year's noise, so learning them does not pay. The hand-picked three-model blend in section 5.2 has lower log-loss (0.539) than all of them, because it leaves out the members that add noise.

With all four years, the final hill-climbing weights are gradient boosting 0.389, logistic regression and the decision tree 0.194 each, the random forest 0.139 and the MLP 0.083. LDA, QDA, Naive Bayes, KNN and the splines receive no weight. Stacking gives its largest coefficients to the random forest, gradient boosting, logistic regression and the decision tree (0.24–0.25 each).



## 5.4. Time-respecting nested check

The tuned scores above come from searches that saw all four validation years and then were scored on the same years, so they carry some optimism. Here each family is tuned again using only the years before each validation year, and then scored on that year once. The threshold also comes from the earlier years only. The question is whether tuning still beats the default, and still matches logistic regression, when the search cannot see the year it is scored on. 2020 has no earlier year, so the outer years are 2021–2023.

In [84]:
NESTED_TRIALS = 20  # per outer year and family; smaller than the main searches to keep the run time reasonable


class FoldSubset:
    """The same cached matrices, restricted to some validation years."""

    def __init__(self, cache, positions):
        self.folds = [cache.folds[i] for i in positions]


def suggest_hgb(trial):
    # Same search space as notebook 3
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_iter": trial.suggest_int("max_iter", 50, 800, log=True),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 4, 64, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 300, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-3, 30, log=True),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0),
    }


def make_hgb(params):
    return HistGradientBoostingClassifier(
        **params, categorical_features=list(range(len(branch_roles("tree")["low"]))),
        early_stopping=False, random_state=RANDOM_STATE,
    )


# family: (cache, suggest, make, default model)
nested_families = {
    "hist_gb": (tree_cache, suggest_hgb, make_hgb, make_hgb({})),
    "random_forest": (tree_cache, suggest_forest, make_forest,
                      RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    "knn": (lambda params: knn_caches[params["columns"]], suggest_knn, make_knn, KNeighborsClassifier()),
}


def outer_f1(cache, model, inner, outer):
    """Threshold from the inner years' predictions, F1 on the outer year."""
    _, _, threshold = evaluate_cached(FoldSubset(cache, inner), model)
    _, oof, _ = evaluate_cached(FoldSubset(cache, [outer]), model)
    return f1_score(oof["y"], oof["p"] >= threshold)


nested_rows = []
for outer in range(1, len(VAL_YEARS)):  # 2020 has no earlier year to tune on
    inner = list(range(outer))
    logistic_f1 = outer_f1(nb_cache, LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), inner, outer)

    for family, (cache, suggest, make, default_model) in nested_families.items():
        def objective(trial):
            params = suggest(trial)
            table, oof, threshold = evaluate_cached(FoldSubset(pick_cache(cache, params), inner), make(params))
            return f1_score(oof["y"], oof["p"] >= threshold)

        # Tuned on the earlier years only; the outer year is never seen by the search
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(objective, n_trials=NESTED_TRIALS, show_progress_bar=False)
        best = study.best_params

        default_cache = cache if not callable(cache) else knn_caches["selected"]
        nested_rows.append({
            "outer_year": VAL_YEARS[outer],
            "family": family,
            "inner_years": len(inner),
            "nested_tuned": outer_f1(pick_cache(cache, best), make(best), inner, outer),
            "default": outer_f1(default_cache, default_model, inner, outer),
            "logistic": logistic_f1,
        })
        print(f"{VAL_YEARS[outer]} {family}: done")

nested = pd.DataFrame(nested_rows)
nested["gain_vs_default"] = nested["nested_tuned"] - nested["default"]
nested["gain_vs_logistic"] = nested["nested_tuned"] - nested["logistic"]

2021 hist_gb: done
2021 random_forest: done
2021 knn: done
2022 hist_gb: done
2022 random_forest: done
2022 knn: done
2023 hist_gb: done
2023 random_forest: done
2023 knn: done


In [85]:
display(nested.round(3))

,outer_year,family,inner_years,nested_tuned,default,logistic,gain_vs_default,gain_vs_logistic
0,2021,hist_gb,1,0.729,0.748,0.765,-0.019,-0.036
1,2021,random_forest,1,0.758,0.750,0.765,0.008,-0.007
2,2021,knn,1,0.736,0.722,0.765,0.014,-0.029
3,2022,hist_gb,2,0.688,0.705,0.711,-0.017,-0.023
4,2022,random_forest,2,0.705,0.701,0.711,0.004,-0.006
5,2022,knn,2,0.701,0.696,0.711,0.004,-0.010
6,2023,hist_gb,3,0.763,0.764,0.742,-0.002,0.020
7,2023,random_forest,3,0.761,0.753,0.742,0.008,0.019
8,2023,knn,3,0.749,0.748,0.742,0.001,0.007


In [86]:
display(nested.groupby("family")[["nested_tuned", "default", "logistic", "gain_vs_default", "gain_vs_logistic"]]
        .mean().round(3))

,nested_tuned,default,logistic,gain_vs_default,gain_vs_logistic
family,,,,,
hist_gb,0.726,0.739,0.739,-0.012,-0.013
knn,0.729,0.722,0.739,0.006,-0.011
random_forest,0.741,0.735,0.739,0.007,0.002


When tuned only on earlier years, the random forest keeps a small gain: +0.007 F1 over its default on average and +0.002 over logistic regression, ahead of logistic regression only in 2023. KNN gains +0.006 over its default but stays below logistic regression (-0.011). Gradient boosting tuned on one or two years falls below its default (-0.012) and below logistic regression (-0.013), recovering only in 2023 (+0.020 over logistic regression), when three years are available. Its tuning needs more history than the early outer years provide.



## 5.5. Choice

The rule keeps logistic regression unless an alternative passes the improvement test against it. Four alternatives pass, all of them equal-weight blends: logistic regression + forest, logistic regression + boosting, logistic regression + forest + boosting, and all finalists. Among passing options, the one with the lowest log-loss is chosen. The number of inputs only breaks ties, because a blend's count is the sum of its members' counts and is not comparable with a single model's.

The chosen model is the equal-weight blend of logistic regression, the random forest and gradient boosting, with threshold 0.37. It has the lowest log-loss of all options (0.539, interval of the difference [-0.013, -0.003]) and F1 0.751. The runner-up is the equal-weight blend of the seven finalists (log-loss 0.540, F1 0.753).

This ranking replaced an earlier version that ordered passing options by input count, which had selected logistic regression + KNN. The change was made after that run's holdout results had been seen, so the holdout is not fully untouched for this choice.



In [87]:
# decision: logistic regression stays unless a tuned model or blend passes the rule against it
all_runs = {**tuned_runs, **blend_runs}
candidates = pd.concat([board_final.drop(index=["dummy", "logistic"]), board_blend.drop(index="logistic")])
better = candidates.index[candidates["verdict"] == "better"].tolist()
# Among passing options: lowest log-loss, the main criterion (a blend's input count is a sum, so not comparable)
chosen_model = candidates.loc[better].sort_values(["ll_diff", "n_features"]).index[0] if better else "logistic"
threshold = all_runs[chosen_model][2]
# Runner-up: the blend with the lowest log-loss that is not the chosen model
runner_up = board_blend.drop(index=["logistic", chosen_model], errors="ignore").sort_values("ll_diff").index[0]
members_of = lambda name: blend_members.get(name, [name])

print(f"Passing against logistic regression: {better or 'none'}")
print(f"Chosen: {chosen_model} (threshold {threshold:.2f}) | runner-up: {runner_up} {members_of(runner_up)}")

log.record(
    "D41", "default models", "default models against logistic regression",
    f"Verdicts: {board_default['verdict'].to_dict()}.",
    0, 0, carries={"default_f1": board_default["f1"].round(4).to_dict(),
                   "default_log_loss": board_default["log_loss"].round(4).to_dict()},
)
log.record(
    "D42", "tuning", "each family tuned with two searches and kept only if it passed against its default",
    "Chosen settings per family recorded in carries.",
    0, 0, carries={"naive_bayes": nb_params, "lda": lda_params, "knn": knn_params, "decision_tree": tree_params,
                   "random_forest": forest_params, "qda": qda_params,
                   "logistic_splines": spline_params, "mlp": mlp_params, "hist_gb": hgb_params, "tuned_f1": board_final["f1"].round(4).to_dict()},
)
log.record(
    "D43", "final model", f"{chosen_model}",
    f"Passing against logistic regression: {better or 'none'}. Finalists {finalists}. Nested gains over logistic "
    f"regression: {nested.groupby('family')['gain_vs_logistic'].mean().round(3).to_dict()}. "
    f"Runner-up: {runner_up} (F1 {board_blend.loc[runner_up, 'f1']:.3f}).",
    0, 0, carries={"chosen": chosen_model, "threshold": float(threshold), "runner_up": members_of(runner_up)},
)
display(log.to_frame().tail(3))


Passing against logistic regression: ['logistic + forest', 'logistic + boosting', 'logistic + forest + boosting', 'all finalists']
Chosen: logistic + forest + boosting (threshold 0.37) | runner-up: all finalists ['random_forest', 'hist_gb', 'logistic_splines', 'logistic', 'mlp', 'decision_tree', 'knn']


,step,column,action,reason,rows_train,rows_test
41,D41,default models,default models against logistic regression,"Verdicts: {'logistic': 'reference', 'lda_selected': 'worse', 'random_forest': 'no clear differen...",0,0
42,D42,tuning,each family tuned with two searches and kept only if it passed against its default,Chosen settings per family recorded in carries.,0,0
43,D43,final model,logistic + forest + boosting,"Passing against logistic regression: ['logistic + forest', 'logistic + boosting', 'logistic + fo...",0,0


## Findings

| Candidate | F1 | F1 difference [95% interval] | Log-loss difference [95% interval] | ROC-AUC | Verdict |
|---|---|---|---|---|---|
| Logistic regression (tuned) | 0.747 | reference | reference | 0.784 | reference |
| Gradient boosting (tuned) | 0.748 | +0.001 [-0.013, 0.019] | -0.006 [-0.013, 0.002] | 0.790 | no clear difference |
| Random forest (tuned) | 0.751 | +0.003 [-0.009, 0.023] | -0.004 [-0.012, 0.003] | 0.791 | no clear difference |
| Logistic + forest + boosting | 0.751 | +0.004 [-0.008, 0.021] | -0.008 [-0.013, -0.003] | 0.792 | better, chosen |
| All seven finalists | 0.753 | +0.006 [-0.006, 0.025] | -0.007 [-0.013, -0.002] | 0.793 | better, runner-up |
| Hill-climbing (all members) | 0.750 | +0.002 [-0.009, 0.019] | -0.003 [-0.009, 0.003] | 0.791 | no clear difference |

- No single tuned model beats the tuned logistic regression; only blends do. Their probabilities are clearly better, while F1 changes by at most +0.006, with intervals that include zero.
- Combining a linear model with two tree ensembles gives the best probabilities: the members make different errors, and averaging cancels part of them.
- Learned weights did not improve on equal weights: they change from year to year and pick up noise.
- The forest's tuning gain survives the nested check; gradient boosting's depends on having several years to tune on.



# 6. Threshold and calibration

The chosen blend's threshold (0.37) comes from its pooled development predictions, with the same rule as every other model. This section checks whether its predicted probabilities match the observed share of repeats.

## 6.1. Calibration



In [88]:
# Reliability of the chosen model's probabilities, with the runner-up blend for contrast
compare = {f"chosen ({chosen_model})": all_runs[chosen_model][1], f"runner-up ({runner_up})": all_runs[runner_up][1]}

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration",
                         line=dict(dash="dash", color="grey")))
calibration_rows = []
for name, oof in compare.items():
    observed, predicted = calibration_curve(oof["y"], oof["p"], n_bins=10, strategy="quantile")
    fig.add_trace(go.Scatter(x=predicted, y=observed, mode="lines+markers", name=name,
                             hovertemplate="predicted %{x:.2f}<br>observed %{y:.2f}<extra>" + name + "</extra>"))
    calibration_rows.append({
        "model": name,
        "brier": brier_score_loss(oof["y"], oof["p"]),
        "mean_predicted": oof["p"].mean(),
        "observed_share": oof["y"].mean(),
        "largest_decile_gap": np.abs(observed - predicted).max(),
    })
fig.update_layout(
    title="Predicted against observed repeat share (deciles of the predictions, development years)",
    xaxis=dict(title="Mean predicted probability", range=[0, 1]),
    yaxis=dict(title="Observed repeat share", range=[0, 1]),
    height=480,
)
fig.write_html(FIGURES_DIR / "04_calibration_final.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_calibration_final.png", **PNG_OPTIONS)
fig.show()

display(pd.DataFrame(calibration_rows).set_index("model").round(3))

,brier,mean_predicted,observed_share,largest_decile_gap
model,,,,
chosen (logistic + forest + boosting),0.184,0.543,0.549,0.039
runner-up (all finalists),0.183,0.550,0.549,0.049


The chosen blend is well calibrated: its mean prediction is 0.543 against an observed share of 0.549, its Brier score is 0.184, and no decile is off by more than 0.039. The runner-up has a similar Brier score (0.183) with a larger decile gap (0.049).



## 6.2. Calibration by year

In [89]:
# Calibration year by year: does the model over- or under-predict repeats as the years change?
oof = all_runs[chosen_model][1]
by_year = oof.groupby("fold").agg(
    rows=("y", "size"),
    observed_share=("y", "mean"),
    mean_predicted=("p", "mean"),
    predicted_positive_share=("p", lambda p: (p >= threshold).mean()),
)
by_year["prediction_gap"] = by_year["mean_predicted"] - by_year["observed_share"]
display(by_year.round(3))

,rows,observed_share,mean_predicted,predicted_positive_share,prediction_gap
fold,,,,,
2020,670,0.594,0.571,0.725,-0.023
2021,782,0.549,0.533,0.643,-0.015
2022,628,0.481,0.541,0.669,0.060
2023,536,0.573,0.523,0.696,-0.050


The mean prediction moves less between years (0.523–0.571) than the observed share of repeats (0.481–0.594). The blend underpredicts in 2020, 2021 and 2023 by 0.015–0.050 and overpredicts in 2022 by 0.060, the year with the fewest repeats. At the fixed threshold it flags 64–73% of purchases as repeats every year, so its errors follow the yearly level of repeats rather than the individual customers.



## 6.3. Threshold from expected F1

The threshold so far comes from the pooled development predictions and stays fixed. If the share of repeats moves between periods, a fixed threshold can flag too many or too few purchases. An alternative uses the probabilities themselves: if they are calibrated, outcomes can be simulated from them, and the threshold with the highest expected F1 can be found without any labels. Applied to the test set, this threshold adapts to the test predictions.

The fair test on development: each validation year gets its own expected-F1 threshold, computed from that year's predicted probabilities only, and its actual F1 is compared with the F1 at the fixed pooled threshold. The pooled threshold has seen every year's labels, so this comparison favours it.

Decision rule: the expected-F1 threshold replaces the fixed one only if its F1 is higher in at least three of the four years and the customer-bootstrap interval of the F1 difference does not lie below zero.


In [90]:
def expected_f1_threshold(p, grid=np.arange(0.05, 0.951, 0.01), n_sim=2000, seed=RANDOM_STATE):
    """Threshold with the highest expected F1 when outcomes are drawn from the probabilities themselves."""
    rng = np.random.default_rng(seed)
    outcomes = rng.random((n_sim, len(p))) < p  # simulated repeats, one row per simulation
    positives = outcomes.sum(axis=1)
    expected = []
    for t in grid:
        flagged = p >= t
        tp = outcomes[:, flagged].sum(axis=1)
        expected.append(np.mean(2 * tp / (flagged.sum() + positives)))
    return float(grid[int(np.argmax(expected))])


chosen_oof = all_runs[chosen_model][1]
rows = []
adaptive = np.zeros(len(chosen_oof), dtype=bool)
for year, part in chosen_oof.groupby("fold"):
    t_year = expected_f1_threshold(part["p"].to_numpy())
    adaptive[part.index] = part["p"].to_numpy() >= t_year
    rows.append({"fold": year, "observed_share": part["y"].mean(), "fixed_threshold": threshold,
                 "expected_f1_threshold": t_year,
                 "f1_fixed": f1_score(part["y"], part["p"] >= threshold),
                 "f1_expected": f1_score(part["y"], part["p"] >= t_year)})
threshold_check = pd.DataFrame(rows).set_index("fold")
threshold_check["difference"] = threshold_check["f1_expected"] - threshold_check["f1_fixed"]
display(threshold_check.round(3))

# Same rows, two rules: customer bootstrap of the F1 difference (thresholds already applied)
y_all = chosen_oof["y"].to_numpy().astype(bool)
fixed_pred = chosen_oof["p"].to_numpy() >= threshold
groups = list(chosen_oof.groupby("customer_id", sort=False).indices.values())
rng = np.random.default_rng(RANDOM_STATE)
diffs = []
for _ in range(2000):
    idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
    diffs.append(fast_f1(y_all[idx], adaptive[idx]) - fast_f1(y_all[idx], fixed_pred[idx]))
ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
pooled_diff = fast_f1(y_all, adaptive) - fast_f1(y_all, fixed_pred)
print(f"Pooled F1: fixed {fast_f1(y_all, fixed_pred):.3f} | expected-F1 per year {fast_f1(y_all, adaptive):.3f} "
      f"| difference {pooled_diff:+.3f} [{ci_low:.3f}, {ci_high:.3f}]")

# decision: adopt only if better in at least 3 of 4 years and the interval does not lie below zero
use_expected_f1 = bool((threshold_check["difference"] > 0).sum() >= 3 and ci_high >= 0)
print(f"Expected-F1 threshold adopted for the submission: {use_expected_f1}")


,observed_share,fixed_threshold,expected_f1_threshold,f1_fixed,f1_expected,difference
fold,,,,,,
2020,0.594,0.370,0.390,0.751,0.742,-0.009
2021,0.549,0.370,0.370,0.775,0.775,0.000
2022,0.481,0.370,0.370,0.701,0.701,0.000
2023,0.573,0.370,0.370,0.774,0.774,0.000


Pooled F1: fixed 0.751 | expected-F1 per year 0.749 | difference -0.002 [-0.006, 0.000]
Expected-F1 threshold adopted for the submission: False


The expected-F1 threshold does not help. Computed from each year's probabilities alone, it equals the fixed threshold (0.37) in 2021–2023 and moves to 0.39 in 2020, where it lowers F1 from 0.751 to 0.742. Pooled F1 falls from 0.751 to 0.749 (difference -0.002, interval [-0.006, 0.000]), and the threshold is higher in no year, so it is not adopted. The fixed threshold stays for the submission.



## Findings

- The chosen blend's threshold is 0.37, from its pooled development predictions.
- Probabilities are close to calibrated overall (Brier 0.184, largest decile gap 0.039).
- The share of repeats changes from year to year by up to 0.11, and the model does not track that level. This drift sets a limit on F1 that tuning cannot remove.
- A threshold set from expected F1 gives the same or a worse result in every year, so the fixed threshold is kept.



# 7. Holdout assessment

The holdout (purchases from 2024-01-01) has not been used for any choice so far. Every model is refitted on the development years and scored once, with the settings and threshold already fixed. The chosen model's result is its unbiased estimate; the other finalists are recorded for reference only.

In [91]:
def family_pipe(name):
    """Complete pipeline of one family with its chosen settings (defaults when tuning was not kept)."""
    linear = lambda model, select=True: (build_linear_pipeline(model, select=select), linear_inputs)
    tree = lambda model: (build_tree_pipeline(model), tree_inputs)
    if name == "logistic":
        return tuned_logistic(), linear_inputs
    if name == "logistic_splines":
        if not spline_params:
            return (build_linear_pipeline(LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), splines=4),
                    linear_inputs)
        return (build_linear_pipeline(
            LogisticRegression(C=spline_params["model_C"], max_iter=5000, random_state=RANDOM_STATE),
            log=spline_params["use_log"], select_C=spline_params["select_C"], l1_ratio=spline_params["l1_ratio"],
            splines=spline_params["n_knots"]), linear_inputs)
    if name == "qda":
        return linear(make_qda(qda_params) if qda_params else QuadraticDiscriminantAnalysis(reg_param=QDA_MIN_REG),
                      select=qda_params.get("columns", "selected") == "selected")
    if name in ("hist_gb", "random_forest") and use_seed_average.get(name):
        return seed_averaged(name), tree_inputs  # 10-seed average, with constraints if kept (4.10, 4.11)
    if name == "hist_gb":
        return family_tree_pipe(name), tree_inputs
    if name == "naive_bayes":
        return linear(make_nb(nb_params) if nb_params else GaussianNB())
    if name == "lda":
        return linear(make_lda(lda_params) if lda_params else LinearDiscriminantAnalysis(),
                      select=lda_params.get("columns", "selected") == "selected")
    if name == "knn":
        return linear(make_knn(knn_params) if knn_params else KNeighborsClassifier(),
                      select=knn_params.get("columns", "selected") == "selected")
    if name == "decision_tree":
        return tree(make_tree(tree_params) if tree_params else DecisionTreeClassifier(random_state=RANDOM_STATE))
    if name == "random_forest":
        return tree(make_forest(forest_params) if forest_params
                    else RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
    if name == "mlp":
        return linear(make_mlp(mlp_params) if mlp_params else QuietMLP(random_state=RANDOM_STATE),
                      select=mlp_params.get("columns", "selected") == "selected")
    raise ValueError(name)


def fit_members(fit_rows, names):
    return {name: clone(final_pipes[name][0]).fit(fit_rows[final_pipes[name][1]], fit_rows[target]) for name in names}


def predict_members(fitted, rows, ensemble=None):
    """Probability of a single model, an equal-weight blend, or a weighted ensemble (fitted combiner)."""
    P = np.column_stack([model.predict_proba(rows[final_pipes[name][1]])[:, 1] for name, model in fitted.items()])
    return COMBINERS[ensemble](P) if ensemble in COMBINERS else P.mean(axis=1)


# Every finalist, the chosen model and the runner-up; blends are scored from their members
shown = list(dict.fromkeys(finalists + [chosen_model, runner_up]))
families = list(dict.fromkeys(m for name in shown for m in members_of(name)))
final_pipes = {name: family_pipe(name) for name in families}
fitted_dev = fit_members(dev, families)
p_holdout = pd.DataFrame({name: predict_members({m: fitted_dev[m] for m in members_of(name)}, holdout, name)
                          for name in shown})

# Thresholds stay those chosen on the development years
thresholds = {name: all_runs[name][2] for name in shown}
y_holdout = holdout[target].to_numpy()
seen = holdout["customer_id"].isin(dev["customer_id"]).to_numpy()

holdout_rows = []
for name, p in p_holdout.items():
    predicted = p.to_numpy() >= thresholds[name]
    holdout_rows.append({
        "model": name,
        "threshold": thresholds[name],
        "f1": f1_score(y_holdout, predicted),
        "f1_seen": f1_score(y_holdout[seen], predicted[seen]),
        "f1_unseen": f1_score(y_holdout[~seen], predicted[~seen]),
        "balanced_accuracy": balanced_accuracy_score(y_holdout, predicted),
        "roc_auc": roc_auc_score(y_holdout, p),
        "log_loss": log_loss(y_holdout, p),
        "predicted_positive_share": predicted.mean(),
    })
board_holdout = pd.DataFrame(holdout_rows).set_index("model")
board_holdout.insert(2, "floor", f1_floor(y_holdout))

print(f"Holdout: {len(holdout)} purchases, {holdout['customer_id'].nunique()} customers, "
      f"{seen.sum()} rows from customers seen in development, repeat share {y_holdout.mean():.3f}")
display(board_holdout.round(3))

# Record only: the expected-F1 threshold computed from the holdout probabilities themselves
t_holdout = expected_f1_threshold(p_holdout[chosen_model].to_numpy())
print(f"{chosen_model}: fixed threshold {threshold:.2f} gives F1 {board_holdout.loc[chosen_model, 'f1']:.3f}; "
      f"expected-F1 threshold {t_holdout:.2f} gives F1 {f1_score(y_holdout, p_holdout[chosen_model] >= t_holdout):.3f}")


Holdout: 835 purchases, 215 customers, 757 rows from customers seen in development, repeat share 0.605


,threshold,f1,floor,f1_seen,f1_unseen,balanced_accuracy,roc_auc,log_loss,predicted_positive_share
model,,,,,,,,,
random_forest,0.380,0.784,0.754,0.798,0.605,0.690,0.796,0.544,0.687
hist_gb,0.380,0.774,0.754,0.797,0.424,0.664,0.794,0.537,0.716
logistic_splines,0.400,0.791,0.754,0.803,0.620,0.701,0.798,0.537,0.685
logistic,0.460,0.786,0.754,0.800,0.552,0.720,0.796,0.538,0.624
mlp,0.390,0.794,0.754,0.800,0.690,0.714,0.794,0.545,0.662
decision_tree,0.330,0.782,0.754,0.801,0.519,0.629,0.776,0.558,0.817
knn,0.380,0.788,0.754,0.796,0.656,0.690,0.787,0.564,0.699
logistic + forest + boosting,0.370,0.790,0.754,0.805,0.575,0.679,0.800,0.535,0.732
all finalists,0.380,0.792,0.754,0.809,0.537,0.689,0.801,0.537,0.714


logistic + forest + boosting: fixed threshold 0.37 gives F1 0.790; expected-F1 threshold 0.38 gives F1 0.788


In [92]:
log.record(
    "D44", "holdout assessment",
    f"{chosen_model} at fixed threshold {threshold:.2f}; finalists recorded only",
    "Settings and thresholds fixed on the development years; holdout results did not change the choice. "
    f"F1 by model: {board_holdout['f1'].round(3).to_dict()}.",
    0, len(holdout), carries={"holdout": board_holdout.round(4).to_dict(orient="index")},
)

**Findings**

- The holdout has 835 purchases from 215 customers. 757 of those rows come from customers already seen in development, and 60.5% of the purchases were repeated.
- The chosen blend scores F1 0.790 at its fixed threshold of 0.37, against the predict-all-repeat floor of 0.754 and 0.786 for logistic regression alone. Its log-loss (0.535) is the best on the board, and its ROC-AUC (0.800) is second only to the seven-model blend (0.801).
- The MLP scores the highest holdout F1 (0.794), followed by the seven-model blend (0.792) and the splines (0.791). The holdout only records these results; the choice was made on the development years, where the MLP alone was not better than logistic regression.
- New customers remain the weak point. The blend reaches F1 0.575 on unseen customers against 0.805 on seen ones. Only 78 holdout rows come from unseen customers, so their F1 values (0.424–0.690 across models) are unstable.
- The blend predicts a repeat for 73.2% of the holdout, against an actual share of 60.5%. F1 rewards recall, so the threshold leans towards predicting repeats.
- For the record only: an expected-F1 threshold computed from the holdout probabilities would be 0.38, with F1 0.788.



# 8. Final fit and submission

The chosen pipeline is refitted on every labelled purchase (development, gap and holdout), keeping its settings and threshold. Before predicting, a check confirms that no training outcome window reaches the test period.

In [93]:
# Refit on every labelled purchase (development, gap and holdout); settings and threshold stay fixed
final_model = fit_members(train, members_of(chosen_model))

assert (train["purchase_date"] + pd.Timedelta(days=GAP_DAYS)).max() < test["purchase_date"].min(), \
    "A training outcome window reaches the test period."

p_test = predict_members(final_model, test, chosen_model)
# The submission threshold: fixed from development, or expected F1 on the test probabilities (section 6.3)
submission_threshold = expected_f1_threshold(p_test) if use_expected_f1 else threshold
print(f"Submission threshold: {submission_threshold:.2f} ({'expected F1 on test' if use_expected_f1 else 'fixed'})")
submission = sample_submission[["ID"]].merge(
    pd.DataFrame({"ID": test["ID"], target: (p_test >= submission_threshold).astype(int)}),
    on="ID", how="left", validate="one_to_one",
)
assert len(submission) == len(test) and submission[target].isin([0, 1]).all()

print(f"Final model ({chosen_model}) fitted on {len(train):,} purchases; threshold {threshold:.2f}")
print(f"Predicted repeat share: test {submission[target].mean():.3f} | holdout "
      f"{board_holdout.loc[chosen_model, 'predicted_positive_share']:.3f}")
print(f"Mean probability: test {p_test.mean():.3f} | holdout {p_holdout[chosen_model].mean():.3f}")


Submission threshold: 0.37 (fixed)
Final model (logistic + forest + boosting) fitted on 6,469 purchases; threshold 0.37
Predicted repeat share: test 0.775 | holdout 0.732
Mean probability: test 0.585 | holdout 0.557


In [94]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

submission_bytes = submission.to_csv(index=False, lineterminator="\n").encode("utf-8")
run_id = hashlib.sha256(submission_bytes).hexdigest()[:16]
stem = f"submission_project_{run_id}"

metadata = {
    "experiment_version": EXPERIMENT_VERSION,
    "input_fingerprint": INPUT_FINGERPRINT,
    "model": chosen_model,
    "threshold": float(submission_threshold),
    "development_f1": float(f1_score(all_runs[chosen_model][1]["y"], all_runs[chosen_model][1]["p"] >= threshold)),
    "holdout_f1": float(board_holdout.loc[chosen_model, "f1"]),
    "predicted_repeat_share": float(submission[target].mean()),
    "training_rows": int(len(train)),
    "members": members_of(chosen_model),
    "runner_up": members_of(runner_up),
    "random_state": int(RANDOM_STATE),
    "package_versions": {pkg: version(pkg) for pkg in ["numpy", "pandas", "scikit-learn", "optuna", "plotly"]},
}

(SUBMISSION_DIR / f"{stem}.csv").write_bytes(submission_bytes)
(SUBMISSION_DIR / f"{stem}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
# The fitted pipelines (one per member) need src/ on the path to load (custom frequency encoder)
joblib.dump(final_model, MODELS_DIR / f"{stem}.joblib")

# Append to the submission log; earlier rows and manually added public scores are kept
log_path = SUBMISSION_DIR / "submission_log.csv"
submission_log = pd.read_csv(log_path) if log_path.exists() else pd.DataFrame()
if f"{stem}.csv" not in submission_log.get("file", pd.Series(dtype=str)).tolist():
    submission_log = pd.concat([submission_log, pd.DataFrame([{
        "file": f"{stem}.csv", "created": pd.Timestamp.now(tz="UTC").isoformat(), "model": chosen_model,
        "threshold": submission_threshold, "fold_f1": metadata["development_f1"], "holdout_f1": metadata["holdout_f1"],
        "predicted_repeat_share": metadata["predicted_repeat_share"], "public_score": np.nan, "notes": "notebook 04",
        "run_id": run_id, "metadata_file": f"{stem}.json",
    }])], ignore_index=True)
    submission_log.to_csv(log_path, index=False)

log.record(
    "D45", "submission", f"{stem}.csv",
    f"{chosen_model} refitted on {len(train):,} labelled purchases; threshold {submission_threshold:.2f}; "
    f"{len(submission)} test predictions in sample-submission order.",
    len(train), len(test), carries={"run_id": run_id, "model_file": f"{stem}.joblib"},
)
log.to_json(PROCESSED_DIR / "decision_log_modelling.json")

# Agreement with the first submission (notebook 3, before the notebook 2 review)
previous = pd.read_csv(SUBMISSION_DIR / "ML_Group_07_Version01.csv").set_index("ID").loc[submission["ID"], target]
print(f"Saved {stem}.csv, .json and .joblib")
print(f"Same decision as the first submission on {(previous.to_numpy() == submission[target].to_numpy()).mean():.1%} "
      f"of test rows; its repeat share was {previous.mean():.3f}")
display(submission_log.tail(3))

Saved submission_project_58744b7de49637c9.csv, .json and .joblib
Same decision as the first submission on 91.1% of test rows; its repeat share was 0.822


,file,created,model,threshold,fold_f1,holdout_f1,predicted_repeat_share,public_score,notes,run_id,metadata_file
7,submission_project_c4f6be8b2e7d464d.csv,2026-11-19T12:50:07.281948+00:00,logistic + forest + boosting,0.380,0.756,0.793,0.765,NaN,notebook 04,c4f6be8b2e7d464d,submission_project_c4f6be8b2e7d464d.json
8,submission_project_0e1021f1290d5efa.csv,2026-11-19T13:53:13.987860+00:00,logistic + forest + boosting,0.370,0.751,0.790,0.774,NaN,notebook 04,0e1021f1290d5efa,submission_project_0e1021f1290d5efa.json
9,submission_project_58744b7de49637c9.csv,2026-11-19T15:10:39.664050+00:00,logistic + forest + boosting,0.370,0.751,0.790,0.775,NaN,notebook 04,58744b7de49637c9,submission_project_58744b7de49637c9.json


**Findings**

- The final blend is refitted on 6,469 labelled purchases; each of its three members is refitted separately. No training outcome window reaches the test period.
- On the test set the blend predicts a repeat for 77.5% of purchases, against 73.2% on the holdout; the mean probability rises from 0.557 to 0.585.
- The submission agrees with the first submission (logistic regression before the notebook 2 review, 82.2% predicted repeats) on 91.1% of the test rows.
- Public leaderboard (recorded manually): 0.78672 for the first submission, 0.79905 for logistic regression + KNN and 0.80668 for the earlier version of this blend with threshold 0.35.



# 9. Drift and refit check

The final model predicts a repeat for a larger share of test purchases than of holdout purchases. Two explanations are checked: a change in the inputs between the holdout and the test period, and the effect of refitting on every labelled purchase.

In [95]:
# Numeric inputs used by the models; the noise column is already excluded
numeric_cols = [c for c in linear_inputs if pd.api.types.is_numeric_dtype(train[c])]
last_dev_year = dev[dev["purchase_date"].dt.year == VAL_YEARS[-1]]


def psi(expected, actual, bins=10):
    """Population stability index; bins are deciles of the expected sample."""
    expected, actual = expected.dropna().astype(float), actual.dropna().astype(float)
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected) + 1e-4  # small floor avoids log(0)
    a = np.histogram(actual, edges)[0] / len(actual) + 1e-4
    return float(((a - e) * np.log(a / e)).sum())


psi_table = pd.DataFrame({
    f"{VAL_YEARS[-1]} vs holdout": {c: psi(last_dev_year[c], holdout[c]) for c in numeric_cols},
    "holdout vs test": {c: psi(holdout[c], test[c]) for c in numeric_cols},
}).sort_values("holdout vs test", ascending=False)

# Adversarial check: can a model tell holdout rows from test rows?
adv_X = pd.concat([holdout[numeric_cols], test[numeric_cols]]).astype(float)
adv_y = np.r_[np.zeros(len(holdout)), np.ones(len(test))]
adv_model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=RANDOM_STATE)
adv_p = cross_val_predict(adv_model, adv_X, adv_y, method="predict_proba",
                          cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))[:, 1]
adv_importance = pd.Series(
    permutation_importance(adv_model.fit(adv_X, adv_y), adv_X, adv_y, scoring="roc_auc",
                           n_repeats=3, random_state=RANDOM_STATE).importances_mean,
    index=numeric_cols,
).sort_values(ascending=False)

seen_test = test["customer_id"].isin(train["customer_id"])
print(f"Test: {len(test)} purchases, {test['customer_id'].nunique()} customers, "
      f"{seen_test.mean():.1%} of rows from customers in the training data")
print(f"Adversarial ROC-AUC, holdout vs test: {roc_auc_score(adv_y, adv_p):.3f}")
print("Features that separate them most:", adv_importance.head(3).round(3).to_dict())
display(psi_table.head(10).round(3))

top = psi_table.head(10).iloc[::-1]
fig = go.Figure([go.Bar(y=top.index, x=top[col], orientation="h", name=col, marker_color=SERIES_COLOURS[i])
                 for i, col in enumerate(top.columns)])
fig.add_vline(x=0.1, line_dash="dot", line_color=ACCENT["near_black"])
fig.add_vline(x=0.25, line_dash="dash", line_color=ACCENT["near_black"])
fig.update_layout(title="Population stability index of the inputs", xaxis_title="PSI (0.1 watch, 0.25 shift)",
                  barmode="group", height=520)
fig.show()
fig.write_html(FIGURES_DIR / "04_psi.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_psi.png", **PNG_OPTIONS)

Test: 986 purchases, 206 customers, 94.5% of rows from customers in the training data
Adversarial ROC-AUC, holdout vs test: 0.982
Features that separate them most: {'customer_tenure_days': 0.334, 'customer_since_year': 0.218, 'prior_purchase_count': 0.001}


,2023 vs holdout,holdout vs test
customer_tenure_days,0.448,0.461
prior_std_purchase_value,0.023,0.115
prior_max_purchase_value,0.088,0.088
prior_mean_purchase_value,0.055,0.072
spend_per_year,0.056,0.068
spend_365d,0.102,0.065
prior_purchase_count,0.063,0.062
prior_min_purchase_value,0.033,0.055
Total Desc_ Linha,0.049,0.051
purchases_per_year,0.055,0.036


In [96]:
# Same model and threshold, fitted on development rows only versus on every labelled row
p_test_dev_fit = predict_members({m: fitted_dev[m] for m in members_of(chosen_model)}, test, chosen_model)

refit_check = pd.DataFrame({
    "fitted on": ["development (2015–2023)", "all labelled (2015–2025)"],
    "rows": [len(dev), len(train)],
    "training repeat share": [dev[target].mean(), train[target].mean()],
    "mean probability, test": [p_test_dev_fit.mean(), p_test.mean()],
    "predicted repeat share, test": [(p_test_dev_fit >= threshold).mean(), (p_test >= threshold).mean()],
}).set_index("fitted on")
display(refit_check.round(3))
print(f"Same decision on {((p_test_dev_fit >= threshold) == (p_test >= threshold)).mean():.1%} of test rows")
print("Predicted repeat share of the final model by threshold:",
      {t: round(float((p_test >= t).mean()), 3) for t in sorted({round(threshold, 2), 0.40, 0.45, 0.50})})

,rows,training repeat share,"mean probability, test","predicted repeat share, test"
fitted on,,,,
development (2015–2023),5403,0.551,0.564,0.753
all labelled (2015–2025),6469,0.560,0.585,0.775


Same decision on 95.5% of test rows
Predicted repeat share of the final model by threshold: {0.37: 0.775, 0.4: 0.736, 0.45: 0.677, 0.5: 0.606}


**Findings**

- 94.5% of test rows come from customers present in the training data.
- A classifier still separates holdout and test rows almost perfectly (ROC-AUC 0.982), through `customer_tenure_days` and `customer_since_year`. Both grow with calendar time. Tenure is no longer a direct model input: it enters only through the purchase-process probability and the per-year rates, whose PSI stays below 0.07.
- The refit shifts the predictions only a little. Fitted on development rows only, the blend predicts a repeat for 75.3% of test purchases; fitted on every labelled row, 77.5%. The two versions agree on 95.5% of test rows.
- Raising the threshold from 0.37 to 0.40 would lower the predicted share from 77.5% to 73.6%. The threshold is kept as chosen on the development years.



# 10. Summary of decisions

| Step | Decision | Evidence |
|---|---|---|
| Validation | Expanding yearly folds 2020–2023 with a 90-day gap; holdout from 2024 | Same design as notebooks 2 and 3, so results are comparable |
| Inputs | Notebook 2 configuration: time-stable rates and the purchase-process probability fitted in each training part | Section 1.4 |
| Threshold | Per model, from pooled out-of-fold predictions (centre of the near-best range) | One rule for every model |
| Comparison rule | Better only if the log-loss interval is below zero, F1 is not clearly worse and log-loss is higher in at most one fold | Log-loss uses every row and detects smaller real differences than F1 |
| Default models | No default model beats logistic regression (F1 0.749, log-loss 0.558) | Section 3 |
| Tuning | Two Optuna searches per family (log-loss and F1), 60 trials each, top 5 rescored; ranges widened where earlier searches hit a limit; every family kept a tuned setting | Section 4; validation curves show over- and underfitting for one setting per family |
| New families | Logistic regression with splines and an MLP, tuned like the others | Both no clear difference against logistic regression |
| Gradient boosting | Notebook 3 settings kept after a broader search; monotonic constraints and a 10-seed average added | No clear difference in either step |
| Single models | None beats the tuned logistic regression (log-loss 0.547) | Best: gradient boosting 0.541, random forest 0.543 |
| Blends | Four of five equal-weight blends better than logistic regression | Log-loss 0.539–0.541 |
| Weighted ensembles | Hill-climbing and stacking, weights learned on earlier years | Neither beats equal weights; weights unstable between years |
| Nested check | Gains over logistic regression: forest +0.002, KNN -0.011, boosting -0.013 | The forest's gain is robust; boosting needs more years to tune |
| Chosen model | Equal-weight blend of logistic regression, random forest and gradient boosting, threshold 0.37 | Lowest log-loss (0.539) among the options that pass |
| Calibration | Brier score 0.184, largest decile gap 0.039; expected-F1 threshold not adopted | Section 6 |
| Holdout | F1 0.790 against a floor of 0.754 | Logistic regression alone 0.786 |
| Submission | Refitted on all labelled purchases; 77.5% predicted repeats | Section 8 |
| Drift | Refit agreement 95.5% of test rows | Section 9 |

